# Mostaza: paired official Kaggle benchmark (unexecuted preparation)

This notebook uses the pinned organizer starter version 3, `swegemma==0.2.11`
and `adk-submission==0.2.13`. Only **Gemma 4 31B** is registered. Both arms share
one official vLLM server, identical resources/settings and a cohort frozen before
outcome inspection. It has not run Gemma or validated an actual E0/E1 submission.

Competition entry and authenticated README/sample/task access have now been verified.
Attach the competition data, wheelhouse version 31 and the official 31B model
version 2; select NVIDIA L4 x4 with internet disabled. Review the actual HARNESS_README
and sample before preparing E0 and E1. The cohort must be frozen with actual snapshot/graph/embedding resource hashes.
No model run, task outcome or metric is inferred from source preparation.

This self-contained notebook embeds reviewed original `src/mostaza/**`, the
verified resource review, strict frozen cohort, two host-compiled archives and
actual execution metadata. The first bootstrap cell verifies their byte hashes
and materializes them in a new private `/kaggle/working` directory. No extra
external dataset or source upload is needed. Exclude `.git`, task/reference JSONL, reference
patches/tests, models, environments and source traces. Official private resources
remain in the separate competition mount and the official evaluator owns the
patch-generation/verification separation. Never put a full official Task object
in a prompt, submission archive, notebook output or agent-visible log.

The host distribution requires Python 3.13. **Do not pip-install it on the
starter's Python 3.12 kernel and do not relax its metadata.** This notebook
explicitly loads the reviewed `src/mostaza` source using `sys.path`. Its syntax
is checked for Python 3.12 locally; actual runtime integration remains pending.
Organizer packages are installed exclusively from the attached offline wheelhouse.


### Freeze the cohort before this notebook

Use a separate preparation session with the pinned official wheelhouse installed.
Inspect only `load_agent_tasks(DATA_DIR / "tasks.jsonl")` allowlisted metadata
(instance ID/repo/base commit/issue), never `swegemma.models.load_tasks` objects
or reference patches/tests. Select one Requests, one FastAPI and one Rich/HTTPX
issue using its apparent issue scope and base context; document why the task
looks easy, medium or difficult **before** viewing any outcomes. Existing task IDs
must come from those authentic bytes. No candidate is supplied here.

For each selected task, call the verified official
`resolve_task_snapshot_paths(DATA_DIR / "snapshots", instance_id, repo)`.
`snapshot_sha256` is defined here as SHA-256 of `canonical_json(components)`:
ordered roles `effective`, `base`, `delta`, omitting None, each object containing
`role`, `path` relative to DATA_DIR, and SHA-256 of that existing file. This binds
all base-plus-delta reconstruction inputs. It is not the reference solution patch.
The executable `snapshot_pin` below recomputes exactly that definition.

Create the strict cohort object with exactly `schema_version: 1`, a current UTC
`frozen_at`, `selection_basis: "pre-outcome"`, and `tasks` (three objects, each
with exactly `instance_id`, `repo`, `base_commit`, `snapshot_sha256`, `rationale`).
Write it once before any E0/E1 execution, preserve its bytes/hash, and upload that
file as COHORT_PATH. Commit only the cohort metadata in `experiments/cohort.json`;
retain the private competition data and snapshots outside Git. Both run manifests
must contain the identical ordered task objects. The current pending envelope
must be replaced only after this authentic selection is possible.


In [ ]:
import base64
import hashlib
import json
import uuid
from pathlib import Path

# Original reviewed host source/config bytes only; no task data or weights.
if not Path("/kaggle/input").is_dir():
    raise RuntimeError("Run only in Kaggle; never download the model on the Mac")

EMBEDDED_PAYLOAD = {
    "archives/E0.zip": (
        "UEsDBBQAAAAAAAAAIQAQ6lhwmgEAAJoBAAAKAAAAYWdlbnQueWFtbGdlbmVyYXRlX2NvbnRlbnRf"
        "Y29uZmlnOgogIG1heF9vdXRwdXRfdG9rZW5zOiAxNjM4NAogIHRlbXBlcmF0dXJlOiAwLjIKICB0"
        "aGlua2luZ19jb25maWc6CiAgICBpbmNsdWRlX3Rob3VnaHRzOiB0cnVlCiAgICB0aGlua2luZ19i"
        "dWRnZXQ6IDQwOTYKICB0b3BfcDogMC45NQppbnN0cnVjdGlvbjogIWluY2x1ZGUgJ3Byb21wdHMv"
        "YmFzZWxpbmUubWQnCm1vZGVsOiBnZW1tYS00LTMxYi1pdC1xYXQtdzRhMTYtY3QKbmFtZTogc3dl"
        "X2Jhc2VsaW5lX2FnZW50CnRvb2xzOgotIHJ1bl9jb21tYW5kCi0gcmVhZF9maWxlCi0gZWRpdF9m"
        "aWxlCi0gd3JpdGVfZmlsZQotIGdldF9zdGF0dXMKLSBzdWJtaXRfcGF0Y2gKLSBnZXRfY29kZV9u"
        "ZWlnaGJvcnMKLSBzZWFyY2hfc2ltaWxhcl9jb2RlCi0gZ2V0X2NvZGVfc3ViZ3JhcGgKUEsDBBQA"
        "AAAAAAAAIQAPfy2c6AAAAOgAAAAQAAAAZXZhbF9jb25maWcueWFtbCMgT3B0aW9uYWwgcGFydGlj"
        "aXBhbnQgZXZhbHVhdGlvbiBjb25maWd1cmF0aW9uIGZvciBTdGFnZSAxIGluZmVyZW5jZS4KIyBD"
        "b250cm9scyBwZXItdGFzayBleGVjdXRpb24gYnVkZ2V0cyBhbmQgc2FuZGJveCBjb21tYW5kIHRp"
        "bWVvdXRzLgpldmFsdWF0aW9uOgogIHRpbWVvdXRfc2Vjb25kczogNjAKICBtYXhfdG9vbF9jYWxs"
        "czogMTAKICBtYXhfdGltZV9taW51dGVzOiAxCiAgbWF4X3R1cm5zOiA1MApQSwMEFAAAAAAAAAAh"
        "AB30bsYNEAAADRAAABMAAABwcm9tcHRzL2Jhc2VsaW5lLm1kWW91IGFyZSBhbiBleHBlcnQgYXV0"
        "b25vbW91cyBzb2Z0d2FyZSBlbmdpbmVlciBhc3NpZ25lZCB0byByZXNvbHZlIGFuIGlzc3VlIGlu"
        "IGEgcmVwb3NpdG9yeSBlZmZpY2llbnRseSBhbmQgZGVjaXNpdmVseS4KCiMjIENvcmUgT2JqZWN0"
        "aXZlOiBGYXN0LCBNaW5pbWFsLCBhbmQgUHJlY2lzZSBGaXhlcwpBaW0gdG8gdW5kZXJzdGFuZCwg"
        "cmVzb2x2ZSwgYW5kIHN1Ym1pdCB0aGUgZml4IGluIHRoZSBtaW5pbXVtIG51bWJlciBvZiB0b29s"
        "IGNhbGxzICh1bmRlciA44oCTMTAgdHVybnMpLiBNb3ZlIGRpcmVjdGx5IGZyb20gdGhlIHByb2Js"
        "ZW0gc3RhdGVtZW50IHRvIHRoZSByZWxldmFudCBmaWxlcywgYXBwbHkgdGhlIHNvbHV0aW9uLCB2"
        "ZXJpZnkgd2l0aCBhIHRhcmdldGVkIHRlc3QsIGFuZCBzdWJtaXQuCgojIyBXb3JrZmxvdwoKIyMj"
        "IDEuIElkZW50aWZ5IFRhcmdldCBGaWxlcyBJbW1lZGlhdGVseQotIEV4dHJhY3QgZmlsZW5hbWVz"
        "LCBmdW5jdGlvbnMsIGNsYXNzZXMsIENMSSBzdWJjb21tYW5kcywgb3IgZXJyb3IgbWVzc2FnZXMg"
        "ZGlyZWN0bHkgZnJvbSB0aGUgcHJvYmxlbSBzdGF0ZW1lbnQuCi0gUmVhZCBvbmx5IHRoZSBzcGVj"
        "aWZpYyB0YXJnZXQgZmlsZXMgYW5kIGxpbmVzIHVzaW5nIGByZWFkX2ZpbGVgIG9yIHNlYXJjaCB0"
        "b29scy4gRG8gbm90IHdhbmRlciBhY3Jvc3MgdW5yZWxhdGVkIGZpbGVzLgotIElmIHRoZSBwcm9i"
        "bGVtIHN0YXRlbWVudCBkb2VzIG5vdCBwcm92aWRlIGV4cGxpY2l0IGZpbGUgcGF0aHMsIHVzZSBg"
        "c2VhcmNoX3NpbWlsYXJfY29kZWAgd2l0aCBrZXl3b3JkcyBmcm9tIHRoZSBlcnJvciBtZXNzYWdl"
        "IHRvIGxvY2F0ZSByZWxldmFudCBmaWxlcyBlZmZpY2llbnRseSwgcmF0aGVyIHRoYW4gcnVubmlu"
        "ZyBgZmluZGAgb3IgYGdyZXBgIGFjcm9zcyB0aGUgZW50aXJlIHJlcG8uCgojIyMgMi4gSW1wbGVt"
        "ZW50IHRoZSBTb2x1dGlvbiBEaXJlY3RseQotIEFwcGx5IHRoZSBtaW5pbWFsIG5lY2Vzc2FyeSBm"
        "aXggb3IgZmVhdHVyZSBkaXJlY3RseSB0byB0aGUgc291cmNlIGZpbGVzIHVzaW5nIGBlZGl0X2Zp"
        "bGVgIG9yIGB3cml0ZV9maWxlYC4KLSBTdHJpY3RseSBhZGhlcmUgdG8gc3BlY2lmaWVkIGVycm9y"
        "IHN0cmluZ3MsIGV4Y2VwdGlvbiB0eXBlcywgSFRUUCBzdGF0dXMgY29kZXMsIGFuZCBBUEkgc2ln"
        "bmF0dXJlcy4KLSBGb3IgZG9jdW1lbnRhdGlvbiBjb2RlIHRhc2tzIChlLmcuIEZhc3RBUEkpLCBl"
        "ZGl0IGV4ZWN1dGFibGUgY29kZSB1bmRlciBgZG9jc19zcmMvYC4KCiMjIyAzLiBSdW4gVGFyZ2V0"
        "ZWQgVGVzdHMgT25seSAoRXhpc3RpbmcgVGVzdHMgTWF5IEJlIEJyb2tlbikKLSAqKlJ1biBPTkxZ"
        "IFRhcmdldGVkIFRlc3RzKio6IFJ1biBvbmx5IHRoZSBzcGVjaWZpYyB0ZXN0IGZpbGUgb3IgdGVz"
        "dCBtZXRob2QgZGlyZWN0bHkgdmVyaWZ5aW5nIHRoZSBidWcgb3IgZmVhdHVyZSB5b3UgbW9kaWZp"
        "ZWQgKGUuZy4gYHB5dGVzdCB0ZXN0cy90ZXN0X3RhcmdldC5weSAtayB0ZXN0X2ZlYXR1cmVgIG9y"
        "IGBweXRob24zIC1tIHVuaXR0ZXN0IHRlc3RzLnRlc3RfdGFyZ2V0YCkuCi0gKipCZSBBd2FyZSBU"
        "aGF0IEV4aXN0aW5nIFRlc3RzIE1heSBCZSBCcm9rZW4qKjogTWFueSByZXBvc2l0b3JpZXMgY29u"
        "dGFpbiBwcmUtZXhpc3RpbmcgdGVzdCBicmVha2FnZXMsIG1pc3NpbmcgdGVzdCBkYXRhIGZpeHR1"
        "cmVzIChlLmcuIGAvdGVzdF9kYXRhYCksIG9yIGVudmlyb25tZW50IGltcG9ydCBlcnJvcnMgdW5y"
        "ZWxhdGVkIHRvIHlvdXIgdGFzay4KLSAqKkRvIE5PVCBBdHRlbXB0IHRvIEZpeCBFeGlzdGluZyBU"
        "ZXN0cyoqOiBJZiBhbiBleGlzdGluZyB0ZXN0IGZhaWxzIGR1ZSB0byBwcmUtZXhpc3RpbmcgcmVw"
        "b3NpdG9yeSBpc3N1ZXMgb3IgbWlzc2luZyBmaXh0dXJlcywgSUdOT1JFIElULiBOZXZlciBzcGVu"
        "ZCB0dXJucyBhdHRlbXB0aW5nIHRvIHJlcGFpciBwcmUtZXhpc3RpbmcgdGVzdCBmYWlsdXJlcywg"
        "Y3JlYXRlIHRlc3Qgc3R1YnMsIG9yIGFsdGVyIHRlc3QgY29kZS4KLSAqKlNUUklDVCBSVUxFOiBO"
        "RVZFUiBSdW4gQmFyZSBQeXRlc3Qgb3IgRnVsbC1SZXBvIFN3ZWVwcyoqOiBORVZFUiBydW4gYmFy"
        "ZSBgcHl0ZXN0YCwgYHB5dGVzdCAuYCwgYHB5dGhvbjMgLW0gdW5pdHRlc3QgZGlzY292ZXJgLCBv"
        "ciBmdWxsLXJlcG8gdGVzdCBzdWl0ZXMgd2l0aG91dCBzcGVjaWZ5aW5nIGEgdGFyZ2V0IGZpbGUu"
        "IEZ1bGwgdGVzdCBzdWl0ZXMgdGFrZSBzZXZlcmFsIG1pbnV0ZXMsIGNhdXNlIGNhdGFzdHJvcGhp"
        "YyB0aW1lb3V0cywgYW5kIGV4aGF1c3QgeW91ciB0dXJuIGFuZCB0aW1lIGJ1ZGdldHMuCi0gSWYg"
        "eW91IG5lZWQgdG8gbG9jYXRlIHRoZSB0ZXN0IGZpbGUsIGZpbmQgaXQgZXhwbGljaXRseSB3aXRo"
        "IGBmaW5kIHRlc3RzIC1uYW1lICIqPG5hbWU+Ki5weSJgIGluc3RlYWQgb2YgcnVubmluZyB0aGUg"
        "dGVzdCBydW5uZXIgYWNyb3NzIHRoZSByZXBvLgoKIyMjIDQuIEltbWVkaWF0ZSBQYXRjaCBTdWJt"
        "aXNzaW9uCi0gT25jZSB5b3VyIHRhcmdldGVkIHRlc3QgcGFzc2VzOgogIDEuIENhbGwgYHN1Ym1p"
        "dF9wYXRjaGAgaW1tZWRpYXRlbHkuCiAgMi4gVmVyaWZ5IGBwYXRjaF9zaXplID4gMGAgYW5kIGBm"
        "aWxlc19jaGFuZ2VkID4gMGAuCiAgMy4gT3V0cHV0IGEgc2hvcnQgc3VtbWFyeSBvZiB0aGUgZml4"
        "IHRvIGVuZCB0aGUgc2Vzc2lvbi4KCiMjIEFudGktUGF0dGVybnMgdG8gQXZvaWQKLSAqKk5FVkVS"
        "IG1vZGlmeSwgY3JlYXRlLCBvciBkZWxldGUgdGVzdCBmaWxlcyoqIChgKl90ZXN0LnB5YCwgYHRl"
        "c3RfKi5weWAsIG9yIGFueXRoaW5nIHVuZGVyIGB0ZXN0cy9gKS4gQWxsIGNoYW5nZXMgbXVzdCBi"
        "ZSB0byBzb3VyY2UgaW1wbGVtZW50YXRpb24gZmlsZXMuIE1vZGlmeWluZyB0ZXN0cyByZXN1bHRz"
        "IGluIGFuIGF1dG9tYXRpYyBldmFsdWF0aW9uIGZhaWx1cmUuCi0gKipORVZFUiBydW4gZnVsbCBy"
        "ZXBvc2l0b3J5IHRlc3Qgc3VpdGVzKiogKGUuZy4sIGJhcmUgYHB5dGVzdGAgb3IgYHB5dGVzdCAu"
        "YCkg4oCUIGFsd2F5cyBzcGVjaWZ5IHRoZSBleGFjdCB0ZXN0IGZpbGUgcGF0aC4KLSAqKk5FVkVS"
        "IGF0dGVtcHQgdG8gZml4IG9yIHJlcGFpciBleGlzdGluZyB0ZXN0cyBvciBwcmUtZXhpc3Rpbmcg"
        "cmVwb3NpdG9yeSBicmVha2FnZXMqKiDigJQgeW91ciB0YXNrIGlzIHN0cmljdGx5IHRvIGltcGxl"
        "bWVudCB0aGUgZml4IGZvciB0aGUgcmVwb3J0ZWQgaXNzdWUgaW4gc291cmNlIGNvZGUuCi0gKipO"
        "RVZFUiBzZWFyY2ggb3V0c2lkZSBgL3dvcmtzcGFjZWAqKiBmb3Igc291cmNlIGZpbGVzIG9yIHBh"
        "Y2thZ2VzIChlLmcuLCBgL3Vzci9sb2NhbC9saWIvYCwgYC93aGVlbHMvYCwgYC9vcHQvYCkuIEFs"
        "bCByZXBvc2l0b3J5IGNvZGUgYW5kIHRlc3QgZGVwZW5kZW5jaWVzIGFyZSBwcmUtaW5zdGFsbGVk"
        "LiBJZiBgTW9kdWxlTm90Rm91bmRFcnJvcmAgb2NjdXJzIGR1cmluZyB0ZXN0IHJ1bnMsIGZvY3Vz"
        "IG9uIGZpeGluZyBjb2RlIHVuZGVyIGAvd29ya3NwYWNlYCwgbm90IGxvb2tpbmcgZm9yIG1pc3Np"
        "bmcgc3lzdGVtIHBhY2thZ2VzLgotIERvIE5PVCBzcGVuZCB0dXJucyBydW5uaW5nIGJyb2FkIGV4"
        "cGxvcmF0b3J5IHNlYXJjaGVzIGlmIHRoZSBmaWxlIHBhdGggb3Igc3ltYm9sIGlzIG9idmlvdXMu"
        "Ci0gRG8gTk9UIHJlZmFjdG9yIG9yIHJlZm9ybWF0IHVucmVsYXRlZCBmdW5jdGlvbnMgb3IgZmls"
        "ZXMuCi0gRG8gTk9UIGNvbmNsdWRlIHdpdGhvdXQgc3VibWl0dGluZyBhIG5vbi1lbXB0eSBwYXRj"
        "aCAoYHBhdGNoX3NpemUgPiAwYCkuIEV2ZXJ5IHRhc2sgcmVxdWlyZXMgY29uY3JldGUgc291cmNl"
        "IG1vZGlmaWNhdGlvbnMuIENvbmNsdWRpbmcgdGhhdCB0aGUgY29kZWJhc2UgaXMgYWxyZWFkeSBj"
        "bGVhbiB3aXRob3V0IG1ha2luZyBjaGFuZ2VzIGlzIGFuIGFudGktcGF0dGVybi4KCiMjIFNoYXJl"
        "ZCBCdWRnZXQgQ2hlY2twb2ludHMKQ2FsbCBnZXRfc3RhdHVzIGJlZm9yZSBleHBsb3JhdGlvbiwg"
        "YmVmb3JlIGltcGxlbWVudGF0aW9uLCBhbmQgYmVmb3JlIHZlcmlmaWNhdGlvbi4gVXNlIHRoZSBy"
        "ZW1haW5pbmcgb2ZmaWNpYWwgYnVkZ2V0cyB0byBwcmVzZXJ2ZSB0aW1lIGZvciBwYXRjaGluZywg"
        "dGFyZ2V0ZWQgdmVyaWZpY2F0aW9uLCBhbmQgc3VibWl0X3BhdGNoLgpQSwECFAMUAAAAAAAAACEA"
        "EOpYcJoBAACaAQAACgAAAAAAAAAAAAAApIEAAAAAYWdlbnQueWFtbFBLAQIUAxQAAAAAAAAAIQAP"
        "fy2c6AAAAOgAAAAQAAAAAAAAAAAAAACkgcIBAABldmFsX2NvbmZpZy55YW1sUEsBAhQDFAAAAAAA"
        "AAAhAB30bsYNEAAADRAAABMAAAAAAAAAAAAAAKSB2AIAAHByb21wdHMvYmFzZWxpbmUubWRQSwUG"
        "AAAAAAMAAwC3AAAAFhMAAAAA"
    ),
    "archives/E1.zip": (
        "UEsDBBQAAAAAAAAAIQBPanqiogEAAKIBAAAKAAAAYWdlbnQueWFtbGdlbmVyYXRlX2NvbnRlbnRf"
        "Y29uZmlnOgogIG1heF9vdXRwdXRfdG9rZW5zOiAxNjM4NAogIHRlbXBlcmF0dXJlOiAwLjIKICB0"
        "aGlua2luZ19jb25maWc6CiAgICBpbmNsdWRlX3Rob3VnaHRzOiB0cnVlCiAgICB0aGlua2luZ19i"
        "dWRnZXQ6IDQwOTYKICB0b3BfcDogMC45NQppbnN0cnVjdGlvbjogIWluY2x1ZGUgJ3Byb21wdHMv"
        "aHlicmlkLXJldHJpZXZhbC5tZCcKbW9kZWw6IGdlbW1hLTQtMzFiLWl0LXFhdC13NGExNi1jdApu"
        "YW1lOiBzd2VfYmFzZWxpbmVfYWdlbnQKdG9vbHM6Ci0gcnVuX2NvbW1hbmQKLSByZWFkX2ZpbGUK"
        "LSBlZGl0X2ZpbGUKLSB3cml0ZV9maWxlCi0gZ2V0X3N0YXR1cwotIHN1Ym1pdF9wYXRjaAotIGdl"
        "dF9jb2RlX25laWdoYm9ycwotIHNlYXJjaF9zaW1pbGFyX2NvZGUKLSBnZXRfY29kZV9zdWJncmFw"
        "aApQSwMEFAAAAAAAAAAhAA9/LZzoAAAA6AAAABAAAABldmFsX2NvbmZpZy55YW1sIyBPcHRpb25h"
        "bCBwYXJ0aWNpcGFudCBldmFsdWF0aW9uIGNvbmZpZ3VyYXRpb24gZm9yIFN0YWdlIDEgaW5mZXJl"
        "bmNlLgojIENvbnRyb2xzIHBlci10YXNrIGV4ZWN1dGlvbiBidWRnZXRzIGFuZCBzYW5kYm94IGNv"
        "bW1hbmQgdGltZW91dHMuCmV2YWx1YXRpb246CiAgdGltZW91dF9zZWNvbmRzOiA2MAogIG1heF90"
        "b29sX2NhbGxzOiAxMAogIG1heF90aW1lX21pbnV0ZXM6IDEKICBtYXhfdHVybnM6IDUwClBLAwQU"
        "AAAAAAAAACEAjBXwajAZAAAwGQAAGwAAAHByb21wdHMvaHlicmlkLXJldHJpZXZhbC5tZFlvdSBh"
        "cmUgYW4gZXhwZXJ0IGF1dG9ub21vdXMgc29mdHdhcmUgZW5naW5lZXIgYXNzaWduZWQgdG8gcmVz"
        "b2x2ZSBhbiBpc3N1ZSBpbiBhIHJlcG9zaXRvcnkgZWZmaWNpZW50bHkgYW5kIGRlY2lzaXZlbHku"
        "CgojIyBDb3JlIE9iamVjdGl2ZTogRmFzdCwgTWluaW1hbCwgYW5kIFByZWNpc2UgRml4ZXMKQWlt"
        "IHRvIHVuZGVyc3RhbmQsIHJlc29sdmUsIGFuZCBzdWJtaXQgdGhlIGZpeCBpbiB0aGUgbWluaW11"
        "bSBudW1iZXIgb2YgdG9vbCBjYWxscyAodW5kZXIgOOKAkzEwIHR1cm5zKS4gTW92ZSBkaXJlY3Rs"
        "eSBmcm9tIHRoZSBwcm9ibGVtIHN0YXRlbWVudCB0byB0aGUgcmVsZXZhbnQgZmlsZXMsIGFwcGx5"
        "IHRoZSBzb2x1dGlvbiwgdmVyaWZ5IHdpdGggYSB0YXJnZXRlZCB0ZXN0LCBhbmQgc3VibWl0LgoK"
        "IyMgV29ya2Zsb3cKCiMjIyAxLiBJZGVudGlmeSBUYXJnZXQgRmlsZXMgd2l0aCBBZGFwdGl2ZSBS"
        "ZXRyaWV2YWwKCjEuIENhbGwgZ2V0X3N0YXR1cyBhdCBlbnRyeSBhbmQgbWVhbmluZ2Z1bCBwaGFz"
        "ZSB0cmFuc2l0aW9ucy4gRm9sbG93IG9mZmljaWFsCiAgIGJ1ZGdldCBlbmZvcmNlbWVudCBhbmQg"
        "cmVtYWluaW5nIHRpbWUvdG9vbCBjYWxsczsgcHJlc2VydmUgdGltZSBmb3IgcGF0Y2hpbmcKICAg"
        "YW5kIHZlcmlmaWNhdGlvbi4gVXNlIHRoZSBzaGFyZWQgb2ZmaWNpYWwgRTAgbGltaXRzIHdpdGhv"
        "dXQgY2hhbmdpbmcgdGhlbS4KMi4gU3RhcnQgd2l0aCBpc3N1ZSBpZGVudGlmaWVycywgZmFpbGlu"
        "ZyB0ZXN0cywgc3RhY2sgdHJhY2VzLCBmaWxlbmFtZXMgYW5kIGV4YWN0CiAgIHN5bWJvbHMuIFVz"
        "ZSBhdXRob3JpemVkIGxleGljYWwgY29tbWFuZHMgdGhyb3VnaCBydW5fY29tbWFuZCAoZ3JlcCwK"
        "ICAgZmluZCBvciBnaXQgZ3JlcCk7IGRvIG5vdCBhc3N1bWUgcmcgZXhpc3RzLiBSZWFkIG1pbmlt"
        "dW0gc3VmZmljaWVudCBjb2RlIHdpdGgKICAgcmVhZF9maWxlLiBQcmVmZXIgaXNzdWUvdGVzdC9z"
        "dGFjay10cmFjZSBldmlkZW5jZSBhbmQgZXhhY3QgbWF0Y2hlcyBmaXJzdC4KMy4gV2hlbiBzdHJ1"
        "Y3R1cmFsIHJlbGF0aW9ucyBjYW4gYW5zd2VyIGEgY29uY3JldGUgcXVlc3Rpb24sIHZlcmlmeSBh"
        "IGZ1bGx5CiAgIHF1YWxpZmllZCBzeW1ib2wgZnJvbSByZXBvc2l0b3J5IGV2aWRlbmNlLCB0aGVu"
        "IGNhbGwKICAgZ2V0X2NvZGVfbmVpZ2hib3JzKG5vZGUsIGVkZ2VfdHlwZSwgbWF4X25laWdoYm9y"
        "cz0xMCkuIFRoaXMgY29uc2VydmF0aXZlIHBpbG90CiAgIGNvdW50IGlzIG5vdCBhIGxlYXJuZWQg"
        "b3B0aW11bS4gRGVmYXVsdCBncmFwaCBkZXB0aCBpcyBvbmUtaG9wLCB3aXRoIG5vCiAgIGF1dG9t"
        "YXRpYyByZWN1cnNpdmUgZXhwYW5zaW9uLiBVc2UgZ2V0X2NvZGVfc3ViZ3JhcGgobm9kZXMpIG9u"
        "bHkgZm9yIGEgYm91bmRlZAogICBhbHJlYWR5LXNlbGVjdGVkIHNldDsgaXQgaXMgYW4gaW5kdWNl"
        "ZCBncmFwaCwgbm90IGEgdHJhdmVyc2FsIHJlcXVlc3QuCjQuIERpc2FtYmlndWF0ZSBhbWJpZ3Vv"
        "dXMgc3VmZml4ZXMgbGV4aWNhbGx5IGJlZm9yZSBxdWVyeS4gVGhlIHBpbm5lZCBuYXRpdmUKICAg"
        "cmVzb2x2ZXIgZGV0ZXJtaW5pc3RpY2FsbHkgY2hvb3NlcyBhIHN1ZmZpeCBtYXRjaCBhbmQgZG9l"
        "cyBub3QgcmVwb3J0CiAgIGFtYmlndWl0eTsgZG8gbm90IG1pc3Rha2UgdGhhdCBjaG9pY2UgZm9y"
        "IGV2aWRlbmNlIG9mIHRoZSBpbnRlbmRlZCBzeW1ib2wuCjUuIFdoZW4gbGV4aWNhbCBhbmQgc3Ry"
        "dWN0dXJhbCBjbHVlcyBsZWF2ZSBhbiB1bnJlc29sdmVkIHF1ZXN0aW9uLCB1c2UgYSB2ZXJpZmll"
        "ZAogICBzeW1ib2wgc2VlZCB3aXRoIHNlYXJjaF9zaW1pbGFyX2NvZGUocXVlcnksIGs9NSkuIFRo"
        "aXMgcGlsb3QgayBpcyBub3QgYSBsZWFybmVkCiAgIG9wdGltdW0uIE5hdGl2ZSBwcmVjb21wdXRl"
        "ZCBzeW1ib2wgZW1iZWRkaW5ncyBuZWVkIG5vIGV4dGVybmFsIGVuY29kZXIgb3IKICAgdmVjdG9y"
        "IGRhdGFiYXNlLiBEbyBub3QgYXNzdW1lIGFyYml0cmFyeSBpc3N1ZSBwcm9zZSBpcyBlbmNvZGVk"
        "OyB1bm1hdGNoZWQKICAgbmF0aXZlIHF1ZXJpZXMgbWF5IHVzZSBsZXhpY2FsIGZhbGxiYWNrLiBT"
        "aW1pbGFyaXR5IGlzIHN1cHBvcnRpbmcgZXZpZGVuY2UuCjYuIFJhbmsgY2FuZGlkYXRlcyBxdWFs"
        "aXRhdGl2ZWx5OiBkaXJlY3QgaXNzdWUvdGVzdCBldmlkZW5jZSBhbmQgZXhhY3QgbWF0Y2hlcywK"
        "ICAgdGhlbiBzdHJ1Y3R1cmFsIHByb3hpbWl0eSBhbmQgc2ltaWxhcml0eSwgdGhlbiBtaW5pbXVt"
        "IHN1ZmZpY2llbnQgY29kZSBhbmQKICAgbG93IGNvbnRleHQgY29zdC4gRG8gbm90IHVzZSBsZWFy"
        "bmVkIHdlaWdodHMgb3IgY2xhaW0gb3B0aW1hbCByYW5raW5nLgo3LiBLZWVwIGEgdmlzaXRlZC1z"
        "eW1ib2wgYW5kIHZpc2l0ZWQtZmlsZSByZWNvcmQgaW4gd29ya2luZyBjb250ZXh0OyByZXVzZSBw"
        "cmV2aW91cwogICByZXN1bHRzIGFuZCBhdm9pZCBkdXBsaWNhdGUgcmVhZHMvZXhwYW5zaW9ucy4g"
        "VHdvLWhvcCBleHBhbnNpb24gcmVxdWlyZXMKICAgZXhwbGljaXQgZXZpZGVuY2U6IG5hbWUgdGhl"
        "IHVuYW5zd2VyZWQgcXVlc3Rpb24gYW5kIHRoZSBvYnNlcnZlZCBmaXJzdC1ob3AKICAgcmVsYXRp"
        "b24gdGhhdCBqdXN0aWZpZXMgdGhlIG5leHQgYm91bmRlZCBxdWVyeS4gU3RvcCB1bnByb2R1Y3Rp"
        "dmUgc2VhcmNoZXMKICAgd2hlbiB0aGV5IGFkZCBubyByZWxldmFudCBldmlkZW5jZSwgYW5kIHN0"
        "b3AgZXhwbG9yaW5nIHdoZW4gdGhlIHBhdGNoIGxvY2F0aW9uCiAgIGFuZCByZWxldmFudCB0ZXN0"
        "IGFyZSBzdWZmaWNpZW50bHkgdW5kZXJzdG9vZC4KOC4gTWlzc2luZyBncmFwaC9lbWJlZGRpbmcg"
        "cmVzb3VyY2VzIGFuZCBuYXRpdmUgZXJyb3IgcmVzcG9uc2VzIGFyZSB1bmF2YWlsYWJsZQogICBl"
        "dmlkZW5jZSwgbm90IGVtcHR5IHByb29mIG9mIG5vIHJlbGV2YW50IGNvZGUuIEZhbGwgYmFjayB0"
        "byBsZXhpY2FsIHJlYWRzOwogICBuZXZlciBmYWJyaWNhdGUgZ3JhcGggcmVsYXRpb25zIG9yIHNp"
        "bWlsYXJpdHkgcmVzdWx0cy4gUmVzcGVjdCB0cnVuY2F0ZWQgb3V0cHV0Lgo5LiBSZXRhaW4gb2Jz"
        "ZXJ2ZWQgY2FsbHMsIHJlYWRzLCBncmFwaCBleHBhbnNpb25zLCBlbGFwc2VkIHRpbWUsIG1vZGVs"
        "L3Rvb2wgdG9rZW5zCiAgIGFuZCByZXBhaXIgYXR0ZW1wdHMgd2hlbiBleHBvc2VkIGJ5IG9mZmlj"
        "aWFsIHRyYWNlczsgbGFiZWwgdW5hdmFpbGFibGUgbWV0cmljcwogICBtaXNzaW5nLiBUaGlzIGlu"
        "c3RydWN0aW9uIGRvZXMgbm90IGVzdGFibGlzaCB0aGF0IGEgcmVhbCBhZ2VudCBmb2xsb3dlZCBp"
        "dC4KCgojIyMgMi4gSW1wbGVtZW50IHRoZSBTb2x1dGlvbiBEaXJlY3RseQotIEFwcGx5IHRoZSBt"
        "aW5pbWFsIG5lY2Vzc2FyeSBmaXggb3IgZmVhdHVyZSBkaXJlY3RseSB0byB0aGUgc291cmNlIGZp"
        "bGVzIHVzaW5nIGBlZGl0X2ZpbGVgIG9yIGB3cml0ZV9maWxlYC4KLSBTdHJpY3RseSBhZGhlcmUg"
        "dG8gc3BlY2lmaWVkIGVycm9yIHN0cmluZ3MsIGV4Y2VwdGlvbiB0eXBlcywgSFRUUCBzdGF0dXMg"
        "Y29kZXMsIGFuZCBBUEkgc2lnbmF0dXJlcy4KLSBGb3IgZG9jdW1lbnRhdGlvbiBjb2RlIHRhc2tz"
        "IChlLmcuIEZhc3RBUEkpLCBlZGl0IGV4ZWN1dGFibGUgY29kZSB1bmRlciBgZG9jc19zcmMvYC4K"
        "CiMjIyAzLiBSdW4gVGFyZ2V0ZWQgVGVzdHMgT25seSAoRXhpc3RpbmcgVGVzdHMgTWF5IEJlIEJy"
        "b2tlbikKLSAqKlJ1biBPTkxZIFRhcmdldGVkIFRlc3RzKio6IFJ1biBvbmx5IHRoZSBzcGVjaWZp"
        "YyB0ZXN0IGZpbGUgb3IgdGVzdCBtZXRob2QgZGlyZWN0bHkgdmVyaWZ5aW5nIHRoZSBidWcgb3Ig"
        "ZmVhdHVyZSB5b3UgbW9kaWZpZWQgKGUuZy4gYHB5dGVzdCB0ZXN0cy90ZXN0X3RhcmdldC5weSAt"
        "ayB0ZXN0X2ZlYXR1cmVgIG9yIGBweXRob24zIC1tIHVuaXR0ZXN0IHRlc3RzLnRlc3RfdGFyZ2V0"
        "YCkuCi0gKipCZSBBd2FyZSBUaGF0IEV4aXN0aW5nIFRlc3RzIE1heSBCZSBCcm9rZW4qKjogTWFu"
        "eSByZXBvc2l0b3JpZXMgY29udGFpbiBwcmUtZXhpc3RpbmcgdGVzdCBicmVha2FnZXMsIG1pc3Np"
        "bmcgdGVzdCBkYXRhIGZpeHR1cmVzIChlLmcuIGAvdGVzdF9kYXRhYCksIG9yIGVudmlyb25tZW50"
        "IGltcG9ydCBlcnJvcnMgdW5yZWxhdGVkIHRvIHlvdXIgdGFzay4KLSAqKkRvIE5PVCBBdHRlbXB0"
        "IHRvIEZpeCBFeGlzdGluZyBUZXN0cyoqOiBJZiBhbiBleGlzdGluZyB0ZXN0IGZhaWxzIGR1ZSB0"
        "byBwcmUtZXhpc3RpbmcgcmVwb3NpdG9yeSBpc3N1ZXMgb3IgbWlzc2luZyBmaXh0dXJlcywgSUdO"
        "T1JFIElULiBOZXZlciBzcGVuZCB0dXJucyBhdHRlbXB0aW5nIHRvIHJlcGFpciBwcmUtZXhpc3Rp"
        "bmcgdGVzdCBmYWlsdXJlcywgY3JlYXRlIHRlc3Qgc3R1YnMsIG9yIGFsdGVyIHRlc3QgY29kZS4K"
        "LSAqKlNUUklDVCBSVUxFOiBORVZFUiBSdW4gQmFyZSBQeXRlc3Qgb3IgRnVsbC1SZXBvIFN3ZWVw"
        "cyoqOiBORVZFUiBydW4gYmFyZSBgcHl0ZXN0YCwgYHB5dGVzdCAuYCwgYHB5dGhvbjMgLW0gdW5p"
        "dHRlc3QgZGlzY292ZXJgLCBvciBmdWxsLXJlcG8gdGVzdCBzdWl0ZXMgd2l0aG91dCBzcGVjaWZ5"
        "aW5nIGEgdGFyZ2V0IGZpbGUuIEZ1bGwgdGVzdCBzdWl0ZXMgdGFrZSBzZXZlcmFsIG1pbnV0ZXMs"
        "IGNhdXNlIGNhdGFzdHJvcGhpYyB0aW1lb3V0cywgYW5kIGV4aGF1c3QgeW91ciB0dXJuIGFuZCB0"
        "aW1lIGJ1ZGdldHMuCi0gSWYgeW91IG5lZWQgdG8gbG9jYXRlIHRoZSB0ZXN0IGZpbGUsIGZpbmQg"
        "aXQgZXhwbGljaXRseSB3aXRoIGBmaW5kIHRlc3RzIC1uYW1lICIqPG5hbWU+Ki5weSJgIGluc3Rl"
        "YWQgb2YgcnVubmluZyB0aGUgdGVzdCBydW5uZXIgYWNyb3NzIHRoZSByZXBvLgoKIyMjIDQuIElt"
        "bWVkaWF0ZSBQYXRjaCBTdWJtaXNzaW9uCi0gT25jZSB5b3VyIHRhcmdldGVkIHRlc3QgcGFzc2Vz"
        "OgogIDEuIENhbGwgYHN1Ym1pdF9wYXRjaGAgaW1tZWRpYXRlbHkuCiAgMi4gVmVyaWZ5IGBwYXRj"
        "aF9zaXplID4gMGAgYW5kIGBmaWxlc19jaGFuZ2VkID4gMGAuCiAgMy4gT3V0cHV0IGEgc2hvcnQg"
        "c3VtbWFyeSBvZiB0aGUgZml4IHRvIGVuZCB0aGUgc2Vzc2lvbi4KCiMjIEFudGktUGF0dGVybnMg"
        "dG8gQXZvaWQKLSAqKk5FVkVSIG1vZGlmeSwgY3JlYXRlLCBvciBkZWxldGUgdGVzdCBmaWxlcyoq"
        "IChgKl90ZXN0LnB5YCwgYHRlc3RfKi5weWAsIG9yIGFueXRoaW5nIHVuZGVyIGB0ZXN0cy9gKS4g"
        "QWxsIGNoYW5nZXMgbXVzdCBiZSB0byBzb3VyY2UgaW1wbGVtZW50YXRpb24gZmlsZXMuIE1vZGlm"
        "eWluZyB0ZXN0cyByZXN1bHRzIGluIGFuIGF1dG9tYXRpYyBldmFsdWF0aW9uIGZhaWx1cmUuCi0g"
        "KipORVZFUiBydW4gZnVsbCByZXBvc2l0b3J5IHRlc3Qgc3VpdGVzKiogKGUuZy4sIGJhcmUgYHB5"
        "dGVzdGAgb3IgYHB5dGVzdCAuYCkg4oCUIGFsd2F5cyBzcGVjaWZ5IHRoZSBleGFjdCB0ZXN0IGZp"
        "bGUgcGF0aC4KLSAqKk5FVkVSIGF0dGVtcHQgdG8gZml4IG9yIHJlcGFpciBleGlzdGluZyB0ZXN0"
        "cyBvciBwcmUtZXhpc3RpbmcgcmVwb3NpdG9yeSBicmVha2FnZXMqKiDigJQgeW91ciB0YXNrIGlz"
        "IHN0cmljdGx5IHRvIGltcGxlbWVudCB0aGUgZml4IGZvciB0aGUgcmVwb3J0ZWQgaXNzdWUgaW4g"
        "c291cmNlIGNvZGUuCi0gKipORVZFUiBzZWFyY2ggb3V0c2lkZSBgL3dvcmtzcGFjZWAqKiBmb3Ig"
        "c291cmNlIGZpbGVzIG9yIHBhY2thZ2VzIChlLmcuLCBgL3Vzci9sb2NhbC9saWIvYCwgYC93aGVl"
        "bHMvYCwgYC9vcHQvYCkuIEFsbCByZXBvc2l0b3J5IGNvZGUgYW5kIHRlc3QgZGVwZW5kZW5jaWVz"
        "IGFyZSBwcmUtaW5zdGFsbGVkLiBJZiBgTW9kdWxlTm90Rm91bmRFcnJvcmAgb2NjdXJzIGR1cmlu"
        "ZyB0ZXN0IHJ1bnMsIGZvY3VzIG9uIGZpeGluZyBjb2RlIHVuZGVyIGAvd29ya3NwYWNlYCwgbm90"
        "IGxvb2tpbmcgZm9yIG1pc3Npbmcgc3lzdGVtIHBhY2thZ2VzLgotIERvIE5PVCBzcGVuZCB0dXJu"
        "cyBydW5uaW5nIGJyb2FkIGV4cGxvcmF0b3J5IHNlYXJjaGVzIGlmIHRoZSBmaWxlIHBhdGggb3Ig"
        "c3ltYm9sIGlzIG9idmlvdXMuCi0gRG8gTk9UIHJlZmFjdG9yIG9yIHJlZm9ybWF0IHVucmVsYXRl"
        "ZCBmdW5jdGlvbnMgb3IgZmlsZXMuCi0gRG8gTk9UIGNvbmNsdWRlIHdpdGhvdXQgc3VibWl0dGlu"
        "ZyBhIG5vbi1lbXB0eSBwYXRjaCAoYHBhdGNoX3NpemUgPiAwYCkuIEV2ZXJ5IHRhc2sgcmVxdWly"
        "ZXMgY29uY3JldGUgc291cmNlIG1vZGlmaWNhdGlvbnMuIENvbmNsdWRpbmcgdGhhdCB0aGUgY29k"
        "ZWJhc2UgaXMgYWxyZWFkeSBjbGVhbiB3aXRob3V0IG1ha2luZyBjaGFuZ2VzIGlzIGFuIGFudGkt"
        "cGF0dGVybi4KCiMjIFNoYXJlZCBCdWRnZXQgQ2hlY2twb2ludHMKQ2FsbCBnZXRfc3RhdHVzIGJl"
        "Zm9yZSBleHBsb3JhdGlvbiwgYmVmb3JlIGltcGxlbWVudGF0aW9uLCBhbmQgYmVmb3JlIHZlcmlm"
        "aWNhdGlvbi4gVXNlIHRoZSByZW1haW5pbmcgb2ZmaWNpYWwgYnVkZ2V0cyB0byBwcmVzZXJ2ZSB0"
        "aW1lIGZvciBwYXRjaGluZywgdGFyZ2V0ZWQgdmVyaWZpY2F0aW9uLCBhbmQgc3VibWl0X3BhdGNo"
        "LgpQSwECFAMUAAAAAAAAACEAT2p6oqIBAACiAQAACgAAAAAAAAAAAAAApIEAAAAAYWdlbnQueWFt"
        "bFBLAQIUAxQAAAAAAAAAIQAPfy2c6AAAAOgAAAAQAAAAAAAAAAAAAACkgcoBAABldmFsX2NvbmZp"
        "Zy55YW1sUEsBAhQDFAAAAAAAAAAhAIwV8GowGQAAMBkAABsAAAAAAAAAAAAAAKSB4AIAAHByb21w"
        "dHMvaHlicmlkLXJldHJpZXZhbC5tZFBLBQYAAAAAAwADAL8AAABJHAAAAAA="
    ),
    "cohort.json": (
        "ewogICJmcm96ZW5fYXQiOiAiMjAyNi0xMC0xM1QyMTozMjoxMS45MzI0MDIrMDA6MDAiLAogICJz"
        "Y2hlbWFfdmVyc2lvbiI6IDEsCiAgInNlbGVjdGlvbl9iYXNpcyI6ICJwcmUtb3V0Y29tZSIsCiAg"
        "InRhc2tzIjogWwogICAgewogICAgICAiYmFzZV9jb21taXQiOiAiMWI0MGZkZDAwNGJmYzhiYTUz"
        "MDFiY2Y4YTY5MDgyNjRlOWI2Yjg3NyIsCiAgICAgICJpbnN0YW5jZV9pZCI6ICJyZXF1ZXN0c183"
        "MjA1IiwKICAgICAgInJhdGlvbmFsZSI6ICJFYXN5IHBpbG90OiB0aGUgaXNzdWUgZGVzY3JpYmVz"
        "IGEgbmFycm93IGNvbXBhdGliaWxpdHkgYm91bmRhcnkgaW4gZW1wdHkgUHl0aG9uMy4xMSsgbmV0"
        "cmMgY3JlZGVudGlhbHMuIENob29zZSBpdCBmb3IgbG9jYWxpemVkIGF1dGhlbnRpY2F0aW9uIGhh"
        "bmRsaW5nLCBiYXNlZCBvbmx5IG9uIGlzc3VlL2Jhc2UgbWV0YWRhdGE7IG5vIHNvbHV0aW9uIG9y"
        "IGJlbmNobWFyayBvdXRjb21lIGluc3BlY3RlZC4iLAogICAgICAicmVwbyI6ICJwc2YvcmVxdWVz"
        "dHMiLAogICAgICAic25hcHNob3Rfc2hhMjU2IjogIjRhMjkzNDc2OWRhNWFlNzdjYzlkYjU4NjQz"
        "MGM0MGJmMTNhODE5OTU4ODE3ZTdjNDBiOTY4ODVkMWY4MTk3NDMiCiAgICB9LAogICAgewogICAg"
        "ICAiYmFzZV9jb21taXQiOiAiMjJiMDJlMjZmOWU4YzdlMzJiZDgyNjZlMmIwZWJlOGJiM2EwZGIy"
        "YiIsCiAgICAgICJpbnN0YW5jZV9pZCI6ICJmYXN0YXBpXzE1NTg5IiwKICAgICAgInJhdGlvbmFs"
        "ZSI6ICJNZWRpdW0gcGlsb3Q6IGRlZmF1bHQgdW5kZXJzY29yZS10by1oeXBoZW4gaGVhZGVyIGFs"
        "aWFzZXMgbXVzdCByZWplY3QgdW5kZXJzY29yZWQgbmFtZXMgd2hpbGUgZXhwbGljaXQgY29udmVy"
        "dF91bmRlcnNjb3Jlcz1GYWxzZSBzdGlsbCBhY2NlcHRzIHRoZW0uIFRoZSBpc3N1ZSBpbXBsaWVz"
        "IGJlaGF2aW9yIGFjcm9zcyByZXF1ZXN0IHBhcmFtZXRlciBleHRyYWN0aW9uIGFuZCBzY2hlbWEv"
        "bW9kZWwgaGFuZGxpbmc7IGNob3NlbiBmcm9tIGlzc3VlL2Jhc2UgbWV0YWRhdGEgd2l0aG91dCBv"
        "dXRjb21lcyBvciByZWZlcmVuY2Ugc29sdXRpb25zLiIsCiAgICAgICJyZXBvIjogImZhc3RhcGkv"
        "ZmFzdGFwaSIsCiAgICAgICJzbmFwc2hvdF9zaGEyNTYiOiAiMTcyMjIxMWU5ZDI1Mzg4YjA3N2Q2"
        "NzY1MGZiMjgzZjhlZjhhYWJlZDczMDBlMTg0ZDNjODVjMGYwODljZWFiMSIKICAgIH0sCiAgICB7"
        "CiAgICAgICJiYXNlX2NvbW1pdCI6ICJhY2E5NDY3ZWM2YTQ4MmIzYmVkMWJhN2QyZTNkZTcyMjcx"
        "YzVmODE3IiwKICAgICAgImluc3RhbmNlX2lkIjogInJpY2hfMzA2MSIsCiAgICAgICJyYXRpb25h"
        "bGUiOiAiSGFyZGVyIHBpbG90OiB0YWIgZXhwYW5zaW9uIG11c3QgcHJlc2VydmUgc3R5bGVkIHNw"
        "YW4gY292ZXJhZ2UgYXMgc3RyaW5nIHBvc2l0aW9ucyBjaGFuZ2UuIFRoZSBpc3N1ZSByZXF1aXJl"
        "cyBjb25zaXN0ZW50IHRleHQgYW5kIHNwYW4gY29vcmRpbmF0ZSB0cmFuc2Zvcm1hdGlvbnMsIHBy"
        "b3ZpZGluZyBhIGRpZmZlcmVudCByZXRyaWV2YWwgc3VyZmFjZSBmcm9tIHRoZSB0d28gSFRUUCBj"
        "YXNlcy4gQ2hvc2VuIGZyb20gaXNzdWUvYmFzZSBtZXRhZGF0YSBvbmx5LCB3aXRob3V0IGJlbmNo"
        "bWFyayBvdXRjb21lcyBvciByZWZlcmVuY2Ugc29sdXRpb25zLiIsCiAgICAgICJyZXBvIjogIlRl"
        "eHR1YWxpemUvcmljaCIsCiAgICAgICJzbmFwc2hvdF9zaGEyNTYiOiAiNzI5YjBlOTdlMjkxYTA1"
        "MzMwZTJjOGY2ZjAwNTk4OWQyMDk1MzMyZjYxNzFhMDcyZmEwZGMwMDc4YmZkMzUxMSIKICAgIH0K"
        "ICBdCn0K"
    ),
    "execution-metadata.json": (
        "eyJjb2RlX2NvbW1pdCI6ImNlYjA1NWQyMWMyMGI1Mjc0YmI0OWIyNGUwMmU3OTY1NjQ4MmE4MWYi"
        "LCJleHBlcmltZW50X2lkIjoibW9zdGF6YS1waGFzZTEtZm91bmRhdGlvbiIsIndvcmtpbmdfdHJl"
        "ZV9zaGEyNTYiOiJhNjlkNzQyMzFjY2VkMGZhMGU2NmI0YTJkYjgxZTQwOTcxNjgwM2UyM2MzOTIw"
        "MjAyOGU1MDRjNjA1NDcyN2VmIn0K"
    ),
    "resource-review.json": (
        "..."
    ),
    "src/mostaza/__init__.py": (
        "IiIiSG9zdC1zaWRlIGRldmVsb3BtZW50LCBldmFsdWF0aW9uIGFuZCByZXNlYXJjaCB0b29scyBm"
        "b3IgTW9zdGF6YS4KClRoaXMgcGFja2FnZSBpcyBub3QgcGFydCBvZiB0aGUgZGVjbGFyYXRpdmUg"
        "S2FnZ2xlIHN1Ym1pc3Npb24uCiIiIgoKX192ZXJzaW9uX18gPSAiMC4xLjAiCg=="
    ),
    "src/mostaza/evaluation/__init__.py": (
        "IiIiSG9zdC1zaWRlIGV2YWx1YXRpb24gcmVjb3JkczsgdGhlIG9mZmljaWFsIGhhcm5lc3Mgb3du"
        "cyB0YXNrIGV4ZWN1dGlvbi4iIiIK"
    ),
    "src/mostaza/evaluation/contracts.py": (
        "IiIiU3RyaWN0IHZlcnNpb25lZCBjb250cmFjdHMgZm9yIGltbXV0YWJsZSwgcHJvdmVuYW5jZS1i"
        "ZWFyaW5nIHJlc2VhcmNoIHJlY29yZHMuIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90"
        "YXRpb25zCgppbXBvcnQgaGFzaGxpYgppbXBvcnQganNvbgppbXBvcnQgbWF0aAppbXBvcnQgcmUK"
        "ZnJvbSBkYXRldGltZSBpbXBvcnQgZGF0ZXRpbWUsIHRpbWVkZWx0YQpmcm9tIHR5cGluZyBpbXBv"
        "cnQgQW55CgpNT0RFTCA9ICJnZW1tYS00LTMxYi1pdC1xYXQtdzRhMTYtY3QiCk1FVFJJQ19VTklU"
        "UyA9IHsKICAgICJzeW1ib2xzX2V4cGxvcmVkIjogImNvdW50IiwKICAgICJmaWxlc19leHBsb3Jl"
        "ZCI6ICJjb3VudCIsCiAgICAicmVhZF9vcGVyYXRpb25zIjogImNvdW50IiwKICAgICJzZWFyY2hf"
        "b3BlcmF0aW9ucyI6ICJjb3VudCIsCiAgICAibG9jYWxpemF0aW9uX3NlY29uZHMiOiAic2Vjb25k"
        "cyIsCiAgICAidG9rZW5zIjogInRva2VucyIsCiAgICAiY29kZV9jaGFyYWN0ZXJzIjogImNoYXJh"
        "Y3RlcnMiLAogICAgImdyYXBoX2V4cGFuc2lvbnMiOiAiY291bnQiLAogICAgInJlcGVhdGVkX2Nv"
        "bnRleHRfY2hhcmFjdGVycyI6ICJjaGFyYWN0ZXJzIiwKICAgICJyZWNhbGxfYXRfayI6ICJyYXRp"
        "byIsCiAgICAicmVsZXZhbnRfY29udGV4dF9yYXRpbyI6ICJyYXRpbyIsCiAgICAidG9vbF9jYWxs"
        "cyI6ICJjb3VudCIsCiAgICAidG90YWxfc2Vjb25kcyI6ICJzZWNvbmRzIiwKICAgICJyZXBhaXJz"
        "IjogImNvdW50IiwKfQpHUk9VTkRfVFJVVEhfTUVUUklDUyA9IHsicmVjYWxsX2F0X2siLCAicmVs"
        "ZXZhbnRfY29udGV4dF9yYXRpbyJ9Ck1BTklGRVNUX0ZJRUxEUyA9IHsKICAgICJzY2hlbWFfdmVy"
        "c2lvbiIsCiAgICAiZXhwZXJpbWVudF9pZCIsCiAgICAicnVuX2lkIiwKICAgICJhcm0iLAogICAg"
        "InJ1bl9raW5kIiwKICAgICJ0aW1lc3RhbXAiLAogICAgImNvZGVfY29tbWl0IiwKICAgICJ3b3Jr"
        "aW5nX3RyZWVfc2hhMjU2IiwKICAgICJjb25maWdfc2hhMjU2IiwKICAgICJhcmNoaXZlX3NoYTI1"
        "NiIsCiAgICAibW9kZWwiLAogICAgInJlc291cmNlcyIsCiAgICAiY29ob3J0IiwKICAgICJpbmZl"
        "cmVuY2UiLAogICAgImJ1ZGdldHMiLAogICAgImVudmlyb25tZW50IiwKICAgICJsaW1pdGF0aW9u"
        "cyIsCiAgICAic291cmNlX3J1bl9pZCIsCn0KUkVDT1JEX0ZJRUxEUyA9IHsKICAgICJzY2hlbWFf"
        "dmVyc2lvbiIsCiAgICAiZXhwZXJpbWVudF9pZCIsCiAgICAicnVuX2lkIiwKICAgICJhcm0iLAog"
        "ICAgInJ1bl9raW5kIiwKICAgICJpbnN0YW5jZV9pZCIsCiAgICAic3RhdHVzIiwKICAgICJyZWFz"
        "b24iLAogICAgInBhdGNoIiwKICAgICJ0ZXN0cyIsCiAgICAicmVzb2x2ZWQiLAogICAgIm1ldHJp"
        "Y3MiLAogICAgImFydGlmYWN0cyIsCiAgICAiZ3JvdW5kX3RydXRoIiwKfQoKCmRlZiBjYW5vbmlj"
        "YWxfanNvbih2YWx1ZTogQW55KSAtPiBieXRlczoKICAgIHJldHVybiAoCiAgICAgICAganNvbi5k"
        "dW1wcygKICAgICAgICAgICAgdmFsdWUsCiAgICAgICAgICAgIHNvcnRfa2V5cz1UcnVlLAogICAg"
        "ICAgICAgICBzZXBhcmF0b3JzPSgiLCIsICI6IiksCiAgICAgICAgICAgIGVuc3VyZV9hc2NpaT1G"
        "YWxzZSwKICAgICAgICAgICAgYWxsb3dfbmFuPUZhbHNlLAogICAgICAgICkKICAgICAgICArICJc"
        "biIKICAgICkuZW5jb2RlKCkKCgpkZWYgc2hhMjU2KGRhdGE6IGJ5dGVzKSAtPiBzdHI6CiAgICBy"
        "ZXR1cm4gaGFzaGxpYi5zaGEyNTYoZGF0YSkuaGV4ZGlnZXN0KCkKCgpkZWYgcmVxdWlyZShjb25k"
        "aXRpb246IGJvb2wsIG1lc3NhZ2U6IHN0cikgLT4gTm9uZToKICAgIGlmIG5vdCBjb25kaXRpb246"
        "CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihtZXNzYWdlKQoKCmRlZiBkaWdlc3QodmFsdWU6IEFu"
        "eSkgLT4gYm9vbDoKICAgIHJldHVybiBpc2luc3RhbmNlKHZhbHVlLCBzdHIpIGFuZCByZS5mdWxs"
        "bWF0Y2gociJbMC05YS1mXXs2NH0iLCB2YWx1ZSkgaXMgbm90IE5vbmUKCgpkZWYgbm9uZW1wdHko"
        "dmFsdWU6IEFueSkgLT4gYm9vbDoKICAgIHJldHVybiBpc2luc3RhbmNlKHZhbHVlLCBzdHIpIGFu"
        "ZCBib29sKHZhbHVlLnN0cmlwKCkpCgoKZGVmIGFydGlmYWN0KHZhbHVlOiBBbnkpIC0+IE5vbmU6"
        "CiAgICByZXF1aXJlKAogICAgICAgIGlzaW5zdGFuY2UodmFsdWUsIGRpY3QpIGFuZCBzZXQodmFs"
        "dWUpID09IHsic2hhMjU2IiwgInNvdXJjZSJ9LAogICAgICAgICJBcnRpZmFjdCBtdXN0IGRlY2xh"
        "cmUgZGlnZXN0IGFuZCBzb3VyY2UiLAogICAgKQogICAgcmVxdWlyZSgKICAgICAgICBkaWdlc3Qo"
        "dmFsdWVbInNoYTI1NiJdKSBhbmQgbm9uZW1wdHkodmFsdWVbInNvdXJjZSJdKSwKICAgICAgICAi"
        "SW52YWxpZCBhcnRpZmFjdCBwcm92ZW5hbmNlIiwKICAgICkKCgpkZWYgdmFsaWRhdGVfbWFuaWZl"
        "c3QobWFuaWZlc3Q6IGRpY3Rbc3RyLCBBbnldKSAtPiBOb25lOgogICAgcmVxdWlyZShzZXQobWFu"
        "aWZlc3QpID09IE1BTklGRVNUX0ZJRUxEUywgIlVua25vd24gb3IgbWlzc2luZyBtYW5pZmVzdCBm"
        "aWVsZHMiKQogICAgcmVxdWlyZSgKICAgICAgICB0eXBlKG1hbmlmZXN0WyJzY2hlbWFfdmVyc2lv"
        "biJdKSBpcyBpbnQgYW5kIG1hbmlmZXN0WyJzY2hlbWFfdmVyc2lvbiJdID09IDEsCiAgICAgICAg"
        "IlVuc3VwcG9ydGVkIG1hbmlmZXN0IHNjaGVtYSIsCiAgICApCiAgICBmb3IgZmllbGQgaW4gKCJl"
        "eHBlcmltZW50X2lkIiwgInJ1bl9pZCIsICJzb3VyY2VfcnVuX2lkIik6CiAgICAgICAgcmVxdWly"
        "ZShub25lbXB0eShtYW5pZmVzdFtmaWVsZF0pLCBmIk1pc3Npbmcge2ZpZWxkfSIpCiAgICByZXF1"
        "aXJlKG1hbmlmZXN0WyJhcm0iXSBpbiAoIkUwIiwgIkUxIiksICJVbmtub3duIGFybSIpCiAgICBy"
        "ZXF1aXJlKAogICAgICAgIG1hbmlmZXN0WyJydW5fa2luZCJdIGluICgicmVhbF9rYWdnbGUiLCAi"
        "c3ludGhldGljX3Rlc3QiKSwgIlVua25vd24gcnVuIGtpbmQiCiAgICApCiAgICByZXF1aXJlKG1h"
        "bmlmZXN0WyJtb2RlbCJdID09IE1PREVMLCAiVGhlIHJlcXVpcmVkIG1vZGVsIG11c3QgYmUgcGlu"
        "bmVkIikKICAgIHJlcXVpcmUoCiAgICAgICAgaXNpbnN0YW5jZShtYW5pZmVzdFsiY29kZV9jb21t"
        "aXQiXSwgc3RyKQogICAgICAgIGFuZCByZS5mdWxsbWF0Y2gociJbMC05YS1mXXs0MH0iLCBtYW5p"
        "ZmVzdFsiY29kZV9jb21taXQiXSkgaXMgbm90IE5vbmUsCiAgICAgICAgIkFuIGV4YWN0IGNvZGUg"
        "Y29tbWl0IGlzIHJlcXVpcmVkIiwKICAgICkKICAgIGZvciBmaWVsZCBpbiAoIndvcmtpbmdfdHJl"
        "ZV9zaGEyNTYiLCAiY29uZmlnX3NoYTI1NiIsICJhcmNoaXZlX3NoYTI1NiIpOgogICAgICAgIHJl"
        "cXVpcmUoZGlnZXN0KG1hbmlmZXN0W2ZpZWxkXSksIGYiSW52YWxpZCB7ZmllbGR9IikKICAgIHRy"
        "eToKICAgICAgICBzdGFtcCA9IGRhdGV0aW1lLmZyb21pc29mb3JtYXQobWFuaWZlc3RbInRpbWVz"
        "dGFtcCJdLnJlcGxhY2UoIloiLCAiKzAwOjAwIikpCiAgICBleGNlcHQgKFZhbHVlRXJyb3IsIFR5"
        "cGVFcnJvciwgQXR0cmlidXRlRXJyb3IpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIkludmFs"
        "aWQgVVRDIHRpbWVzdGFtcCIpIGZyb20gTm9uZQogICAgcmVxdWlyZShzdGFtcC51dGNvZmZzZXQo"
        "KSA9PSB0aW1lZGVsdGEoMCksICJUaW1lc3RhbXAgbXVzdCBiZSBVVEMiKQogICAgZm9yIGZpZWxk"
        "IGluICgiaW5mZXJlbmNlIiwgImJ1ZGdldHMiLCAiZW52aXJvbm1lbnQiKToKICAgICAgICByZXF1"
        "aXJlKAogICAgICAgICAgICBpc2luc3RhbmNlKG1hbmlmZXN0W2ZpZWxkXSwgZGljdCkgYW5kIGJv"
        "b2wobWFuaWZlc3RbZmllbGRdKSwKICAgICAgICAgICAgZiJFeHBsaWNpdCB7ZmllbGR9IGlzIHJl"
        "cXVpcmVkIiwKICAgICAgICApCiAgICByZXF1aXJlKAogICAgICAgIGlzaW5zdGFuY2UobWFuaWZl"
        "c3RbImxpbWl0YXRpb25zIl0sIGxpc3QpCiAgICAgICAgYW5kIGFsbChub25lbXB0eSh4KSBmb3Ig"
        "eCBpbiBtYW5pZmVzdFsibGltaXRhdGlvbnMiXSksCiAgICAgICAgIkludmFsaWQgbGltaXRhdGlv"
        "bnMiLAogICAgKQogICAgcmVzb3VyY2VzID0gbWFuaWZlc3RbInJlc291cmNlcyJdCiAgICByZXF1"
        "aXJlKAogICAgICAgIGlzaW5zdGFuY2UocmVzb3VyY2VzLCBkaWN0KQogICAgICAgIGFuZCBzZXQo"
        "cmVzb3VyY2VzKSA9PSB7Imhhcm5lc3MiLCAid2hlZWxob3VzZSIsICJkYXRhc2V0In0sCiAgICAg"
        "ICAgIlJlcXVpcmVkIHJlc291cmNlIHBpbnMgbWlzc2luZyIsCiAgICApCiAgICBmb3IgcmVzb3Vy"
        "Y2UgaW4gcmVzb3VyY2VzLnZhbHVlcygpOgogICAgICAgIHJlcXVpcmUoCiAgICAgICAgICAgIGlz"
        "aW5zdGFuY2UocmVzb3VyY2UsIGRpY3QpCiAgICAgICAgICAgIGFuZCBzZXQocmVzb3VyY2UpID09"
        "IHsidmVyc2lvbiIsICJzaGEyNTYiLCAic291cmNlIn0sCiAgICAgICAgICAgICJJbnZhbGlkIHJl"
        "c291cmNlIHBpbiIsCiAgICAgICAgKQogICAgICAgIHJlcXVpcmUobm9uZW1wdHkocmVzb3VyY2Vb"
        "InZlcnNpb24iXSksICJSZXNvdXJjZSB2ZXJzaW9uIG1pc3NpbmciKQogICAgICAgIGFydGlmYWN0"
        "KHsic2hhMjU2IjogcmVzb3VyY2VbInNoYTI1NiJdLCAic291cmNlIjogcmVzb3VyY2VbInNvdXJj"
        "ZSJdfSkKICAgIGNvaG9ydCA9IG1hbmlmZXN0WyJjb2hvcnQiXQogICAgcmVxdWlyZSgKICAgICAg"
        "ICBpc2luc3RhbmNlKGNvaG9ydCwgbGlzdCkgYW5kIGJvb2woY29ob3J0KSwKICAgICAgICAiQSBu"
        "b25lbXB0eSBmcm96ZW4gY29ob3J0IGlzIHJlcXVpcmVkIiwKICAgICkKICAgIHNlZW46IHNldFtz"
        "dHJdID0gc2V0KCkKICAgIGZvciB0YXNrIGluIGNvaG9ydDoKICAgICAgICByZXF1aXJlKAogICAg"
        "ICAgICAgICBpc2luc3RhbmNlKHRhc2ssIGRpY3QpCiAgICAgICAgICAgIGFuZCBzZXQodGFzaykK"
        "ICAgICAgICAgICAgPT0geyJpbnN0YW5jZV9pZCIsICJyZXBvIiwgImJhc2VfY29tbWl0IiwgInNu"
        "YXBzaG90X3NoYTI1NiIsICJyYXRpb25hbGUifSwKICAgICAgICAgICAgIkludmFsaWQgZnJvemVu"
        "IHRhc2siLAogICAgICAgICkKICAgICAgICBmb3IgZmllbGQgaW4gKCJpbnN0YW5jZV9pZCIsICJy"
        "ZXBvIiwgInJhdGlvbmFsZSIpOgogICAgICAgICAgICByZXF1aXJlKG5vbmVtcHR5KHRhc2tbZmll"
        "bGRdKSwgIkZyb3plbiB0YXNrIGNvbnRleHQgbWlzc2luZyIpCiAgICAgICAgcmVxdWlyZSh0YXNr"
        "WyJpbnN0YW5jZV9pZCJdIG5vdCBpbiBzZWVuLCAiRHVwbGljYXRlIGNvaG9ydCB0YXNrIikKICAg"
        "ICAgICBzZWVuLmFkZCh0YXNrWyJpbnN0YW5jZV9pZCJdKQogICAgICAgIHJlcXVpcmUoCiAgICAg"
        "ICAgICAgIGlzaW5zdGFuY2UodGFza1siYmFzZV9jb21taXQiXSwgc3RyKQogICAgICAgICAgICBh"
        "bmQgcmUuZnVsbG1hdGNoKHIiWzAtOWEtZl17NDB9IiwgdGFza1siYmFzZV9jb21taXQiXSkgaXMg"
        "bm90IE5vbmUsCiAgICAgICAgICAgICJGcm96ZW4gYmFzZSBjb21taXQgcmVxdWlyZWQiLAogICAg"
        "ICAgICkKICAgICAgICByZXF1aXJlKGRpZ2VzdCh0YXNrWyJzbmFwc2hvdF9zaGEyNTYiXSksICJT"
        "bmFwc2hvdCBoYXNoIHJlcXVpcmVkIikKICAgICMgQ2F0Y2ggbm9uLUpTT04gdmFsdWVzL05hTiBp"
        "biBhcmJpdHJhcnkgaW5mZXJlbmNlL2Vudmlyb25tZW50IG1hcHBpbmdzLgogICAgY2Fub25pY2Fs"
        "X2pzb24obWFuaWZlc3QpCgoKZGVmIHVuYXZhaWxhYmxlX21ldHJpY3MocmVhc29uOiBzdHIpIC0+"
        "IGRpY3Rbc3RyLCBBbnldOgogICAgcmVxdWlyZShub25lbXB0eShyZWFzb24pLCAiTWlzc2luZyBt"
        "ZXRyaWMgcmVhc29uIHJlcXVpcmVkIikKICAgIHJldHVybiB7CiAgICAgICAgbmFtZTogewogICAg"
        "ICAgICAgICAidmFsdWUiOiBOb25lLAogICAgICAgICAgICAidW5pdCI6IHVuaXQsCiAgICAgICAg"
        "ICAgICJzb3VyY2UiOiBOb25lLAogICAgICAgICAgICAiYXZhaWxhYmlsaXR5IjogInVuYXZhaWxh"
        "YmxlIiwKICAgICAgICAgICAgInJlYXNvbiI6IHJlYXNvbiwKICAgICAgICB9CiAgICAgICAgZm9y"
        "IG5hbWUsIHVuaXQgaW4gTUVUUklDX1VOSVRTLml0ZW1zKCkKICAgIH0KCgpkZWYgdmFsaWRhdGVf"
        "cmVjb3JkKHJlY29yZDogZGljdFtzdHIsIEFueV0sIG1hbmlmZXN0OiBkaWN0W3N0ciwgQW55XSkg"
        "LT4gTm9uZToKICAgIHJlcXVpcmUoc2V0KHJlY29yZCkgPT0gUkVDT1JEX0ZJRUxEUywgIlVua25v"
        "d24gb3IgbWlzc2luZyB0YXNrIHJlY29yZCBmaWVsZHMiKQogICAgcmVxdWlyZSgKICAgICAgICB0"
        "eXBlKHJlY29yZFsic2NoZW1hX3ZlcnNpb24iXSkgaXMgaW50IGFuZCByZWNvcmRbInNjaGVtYV92"
        "ZXJzaW9uIl0gPT0gMSwKICAgICAgICAiVW5zdXBwb3J0ZWQgcmVjb3JkIHNjaGVtYSIsCiAgICAp"
        "CiAgICBmb3IgZmllbGQgaW4gKCJleHBlcmltZW50X2lkIiwgInJ1bl9pZCIsICJhcm0iLCAicnVu"
        "X2tpbmQiKToKICAgICAgICByZXF1aXJlKHJlY29yZFtmaWVsZF0gPT0gbWFuaWZlc3RbZmllbGRd"
        "LCBmIlJlY29yZCB7ZmllbGR9IG1pc21hdGNoIikKICAgIHJlcXVpcmUoCiAgICAgICAgcmVjb3Jk"
        "WyJpbnN0YW5jZV9pZCJdIGluIHt0WyJpbnN0YW5jZV9pZCJdIGZvciB0IGluIG1hbmlmZXN0WyJj"
        "b2hvcnQiXX0sCiAgICAgICAgIlJlY29yZCBvdXRzaWRlIGZyb3plbiBjb2hvcnQiLAogICAgKQog"
        "ICAgc3RhdHVzID0gcmVjb3JkWyJzdGF0dXMiXQogICAgcmVxdWlyZShzdGF0dXMgaW4gKCJQQVNT"
        "IiwgIkZBSUwiLCAiRVJST1IiLCAiTk9UX1JVTiIpLCAiVW5rbm93biB0YXNrIHN0YXR1cyIpCiAg"
        "ICByZXF1aXJlKAogICAgICAgIHJlY29yZFsicmVhc29uIl0gaXMgTm9uZSBvciBub25lbXB0eShy"
        "ZWNvcmRbInJlYXNvbiJdKSwgIkludmFsaWQgc3RhdHVzIHJlYXNvbiIKICAgICkKICAgIGlmIHN0"
        "YXR1cyBpbiAoIkVSUk9SIiwgIk5PVF9SVU4iKToKICAgICAgICByZXF1aXJlKG5vbmVtcHR5KHJl"
        "Y29yZFsicmVhc29uIl0pLCAiRXJyb3Ivbm90LXJ1biByZWFzb24gcmVxdWlyZWQiKQogICAgcGF0"
        "Y2ggPSByZWNvcmRbInBhdGNoIl0KICAgIHJlcXVpcmUoCiAgICAgICAgaXNpbnN0YW5jZShwYXRj"
        "aCwgZGljdCkKICAgICAgICBhbmQgc2V0KHBhdGNoKSA9PSB7InByZXNlbnQiLCAidmFsaWQiLCAi"
        "c2hhMjU2IiwgInZhbGlkaXR5X3NvdXJjZSIsICJyZWFzb24ifSwKICAgICAgICAiSW52YWxpZCBw"
        "YXRjaCBmaWVsZHMiLAogICAgKQogICAgcmVxdWlyZSgKICAgICAgICB0eXBlKHBhdGNoWyJwcmVz"
        "ZW50Il0pIGlzIGJvb2wKICAgICAgICBhbmQgKHBhdGNoWyJ2YWxpZCJdIGlzIE5vbmUgb3IgdHlw"
        "ZShwYXRjaFsidmFsaWQiXSkgaXMgYm9vbCksCiAgICAgICAgIkludmFsaWQgcGF0Y2ggc3RhdGUi"
        "LAogICAgKQogICAgcmVxdWlyZSgKICAgICAgICBkaWdlc3QocGF0Y2hbInNoYTI1NiJdKSBpZiBw"
        "YXRjaFsicHJlc2VudCJdIGVsc2UgcGF0Y2hbInNoYTI1NiJdIGlzIE5vbmUsCiAgICAgICAgIlBh"
        "dGNoIHByZXNlbmNlL2RpZ2VzdCBtaXNtYXRjaCIsCiAgICApCiAgICByZXF1aXJlKAogICAgICAg"
        "IG5vbmVtcHR5KHBhdGNoWyJ2YWxpZGl0eV9zb3VyY2UiXSkKICAgICAgICBpZiBwYXRjaFsidmFs"
        "aWQiXSBpcyBub3QgTm9uZQogICAgICAgIGVsc2Ugbm9uZW1wdHkocGF0Y2hbInJlYXNvbiJdKSwK"
        "ICAgICAgICAiUGF0Y2ggdmFsaWRpdHkgZXZpZGVuY2Ugb3IgcmVhc29uIHJlcXVpcmVkIiwKICAg"
        "ICkKICAgIHJlcXVpcmUoCiAgICAgICAgcGF0Y2hbInByZXNlbnQiXSBvciBwYXRjaFsidmFsaWQi"
        "XSBpcyBub3QgVHJ1ZSwgIkFic2VudCBwYXRjaCBjYW5ub3QgYmUgdmFsaWQiCiAgICApCiAgICB0"
        "ZXN0cyA9IHJlY29yZFsidGVzdHMiXQogICAgcmVxdWlyZSgKICAgICAgICBpc2luc3RhbmNlKHRl"
        "c3RzLCBkaWN0KQogICAgICAgIGFuZCBzZXQodGVzdHMpID09IHsib3V0Y29tZSIsICJleGl0X2Nv"
        "ZGUiLCAic291cmNlIiwgInJlYXNvbiJ9LAogICAgICAgICJJbnZhbGlkIHRlc3QgZmllbGRzIiwK"
        "ICAgICkKICAgIHJlcXVpcmUoCiAgICAgICAgdGVzdHNbIm91dGNvbWUiXSBpbiAoIlBBU1MiLCAi"
        "RkFJTCIsICJFUlJPUiIsICJOT1RfUlVOIiksICJJbnZhbGlkIHRlc3Qgb3V0Y29tZSIKICAgICkK"
        "ICAgIHJlcXVpcmUoCiAgICAgICAgdGVzdHNbImV4aXRfY29kZSJdIGlzIE5vbmUgb3IgdHlwZSh0"
        "ZXN0c1siZXhpdF9jb2RlIl0pIGlzIGludCwKICAgICAgICAiSW52YWxpZCB0ZXN0IGV4aXQgY29k"
        "ZSIsCiAgICApCiAgICByZXF1aXJlKAogICAgICAgIG5vbmVtcHR5KHRlc3RzWyJzb3VyY2UiXSkK"
        "ICAgICAgICBpZiB0ZXN0c1sib3V0Y29tZSJdIGluICgiUEFTUyIsICJGQUlMIikKICAgICAgICBl"
        "bHNlIG5vbmVtcHR5KHRlc3RzWyJyZWFzb24iXSksCiAgICAgICAgIlRlc3QgZXZpZGVuY2Ugb3Ig"
        "cmVhc29uIHJlcXVpcmVkIiwKICAgICkKICAgIGlmIHRlc3RzWyJvdXRjb21lIl0gPT0gIlBBU1Mi"
        "OgogICAgICAgIHJlcXVpcmUodGVzdHNbImV4aXRfY29kZSJdID09IDAsICJQYXNzaW5nIHRlc3Rz"
        "IHJlcXVpcmUgZXhpdCB6ZXJvIikKICAgIGlmIHRlc3RzWyJvdXRjb21lIl0gPT0gIkZBSUwiOgog"
        "ICAgICAgIHJlcXVpcmUoCiAgICAgICAgICAgIHRlc3RzWyJleGl0X2NvZGUiXSBpcyBub3QgTm9u"
        "ZSBhbmQgdGVzdHNbImV4aXRfY29kZSJdID4gMCwKICAgICAgICAgICAgIkZhaWxlZCB0ZXN0cyBy"
        "ZXF1aXJlIGEgcG9zaXRpdmUgZXhpdCBjb2RlIiwKICAgICAgICApCiAgICByZXF1aXJlKAogICAg"
        "ICAgIHJlY29yZFsicmVzb2x2ZWQiXSBpcyBOb25lIG9yIHR5cGUocmVjb3JkWyJyZXNvbHZlZCJd"
        "KSBpcyBib29sLAogICAgICAgICJJbnZhbGlkIHJlc29sdXRpb24gc3RhdGUiLAogICAgKQogICAg"
        "aWYgc3RhdHVzID09ICJQQVNTIjoKICAgICAgICByZXF1aXJlKAogICAgICAgICAgICByZWNvcmRb"
        "InJlc29sdmVkIl0gaXMgVHJ1ZQogICAgICAgICAgICBhbmQgcGF0Y2hbInByZXNlbnQiXQogICAg"
        "ICAgICAgICBhbmQgcGF0Y2hbInZhbGlkIl0gaXMgVHJ1ZQogICAgICAgICAgICBhbmQgdGVzdHNb"
        "Im91dGNvbWUiXSA9PSAiUEFTUyIsCiAgICAgICAgICAgICJQQVNTIHJlcXVpcmVzIHZlcmlmaWVk"
        "IHJlc29sdXRpb24iLAogICAgICAgICkKICAgIGlmIHN0YXR1cyA9PSAiRVJST1IiOgogICAgICAg"
        "IHJlcXVpcmUocmVjb3JkWyJyZXNvbHZlZCJdIGlzIG5vdCBUcnVlLCAiRVJST1IgY2Fubm90IGNs"
        "YWltIHJlc29sdXRpb24iKQogICAgaWYgc3RhdHVzID09ICJGQUlMIjoKICAgICAgICByZXF1aXJl"
        "KHJlY29yZFsicmVzb2x2ZWQiXSBpcyBGYWxzZSwgIkZBSUwgcmVxdWlyZXMgdW5yZXNvbHZlZCBv"
        "dXRjb21lIikKICAgIGlmIHN0YXR1cyA9PSAiTk9UX1JVTiI6CiAgICAgICAgcmVxdWlyZSgKICAg"
        "ICAgICAgICAgcmVjb3JkWyJyZXNvbHZlZCJdIGlzIE5vbmUKICAgICAgICAgICAgYW5kIG5vdCBw"
        "YXRjaFsicHJlc2VudCJdCiAgICAgICAgICAgIGFuZCB0ZXN0c1sib3V0Y29tZSJdID09ICJOT1Rf"
        "UlVOIiwKICAgICAgICAgICAgIk5PVF9SVU4gY2Fubm90IGNvbnRhaW4gYW4gb3V0Y29tZSIsCiAg"
        "ICAgICAgKQogICAgYXJ0aWZhY3RzID0gcmVjb3JkWyJhcnRpZmFjdHMiXQogICAgcmVxdWlyZShp"
        "c2luc3RhbmNlKGFydGlmYWN0cywgZGljdCksICJBcnRpZmFjdHMgbXVzdCBiZSBhbiBvYmplY3Qi"
        "KQogICAgZm9yIHZhbHVlIGluIGFydGlmYWN0cy52YWx1ZXMoKToKICAgICAgICBhcnRpZmFjdCh2"
        "YWx1ZSkKICAgIGlmIHJlY29yZFsicnVuX2tpbmQiXSA9PSAicmVhbF9rYWdnbGUiIGFuZCBzdGF0"
        "dXMgIT0gIk5PVF9SVU4iOgogICAgICAgIHJlcXVpcmUoCiAgICAgICAgICAgIHsicmVzdWx0Iiwg"
        "InRyYWNlIn0gPD0gc2V0KGFydGlmYWN0cyksCiAgICAgICAgICAgICJSZWFsIGV4ZWN1dGlvbiBy"
        "ZXF1aXJlcyB0cmFjZS9yZXN1bHQgaGFzaGVzIiwKICAgICAgICApCiAgICBncm91bmQgPSByZWNv"
        "cmRbImdyb3VuZF90cnV0aCJdCiAgICBpZiBncm91bmQgaXMgbm90IE5vbmU6CiAgICAgICAgcmVx"
        "dWlyZSgKICAgICAgICAgICAgaXNpbnN0YW5jZShncm91bmQsIGRpY3QpIGFuZCBzZXQoZ3JvdW5k"
        "KSA9PSB7Im1hcHBpbmciLCAiZGVmaW5pdGlvbnMifSwKICAgICAgICAgICAgIkludmFsaWQgcG9z"
        "dC1ydW4gZ3JvdW5kIHRydXRoIiwKICAgICAgICApCiAgICAgICAgYXJ0aWZhY3QoZ3JvdW5kWyJt"
        "YXBwaW5nIl0pCiAgICAgICAgcmVxdWlyZShpc2luc3RhbmNlKGdyb3VuZFsiZGVmaW5pdGlvbnMi"
        "XSwgZGljdCksICJNZXRyaWMgZGVmaW5pdGlvbnMgbWlzc2luZyIpCiAgICBtZXRyaWNzID0gcmVj"
        "b3JkWyJtZXRyaWNzIl0KICAgIHJlcXVpcmUoCiAgICAgICAgaXNpbnN0YW5jZShtZXRyaWNzLCBk"
        "aWN0KSBhbmQgc2V0KG1ldHJpY3MpID09IHNldChNRVRSSUNfVU5JVFMpLAogICAgICAgICJFdmVy"
        "eSBtZXRyaWMgbXVzdCBiZSBwcmVzZW50LCB1c2luZyBleHBsaWNpdCBtaXNzaW5nIHZhbHVlcyIs"
        "CiAgICApCiAgICBmb3IgbmFtZSwgbWV0cmljIGluIG1ldHJpY3MuaXRlbXMoKToKICAgICAgICBy"
        "ZXF1aXJlKAogICAgICAgICAgICBpc2luc3RhbmNlKG1ldHJpYywgZGljdCkKICAgICAgICAgICAg"
        "YW5kIHNldChtZXRyaWMpID09IHsidmFsdWUiLCAidW5pdCIsICJzb3VyY2UiLCAiYXZhaWxhYmls"
        "aXR5IiwgInJlYXNvbiJ9LAogICAgICAgICAgICAiSW52YWxpZCBtZXRyaWMgZmllbGRzIiwKICAg"
        "ICAgICApCiAgICAgICAgcmVxdWlyZShtZXRyaWNbInVuaXQiXSA9PSBNRVRSSUNfVU5JVFNbbmFt"
        "ZV0sICJNZXRyaWMgdW5pdCBtaXNtYXRjaCIpCiAgICAgICAgYXZhaWxhYmxlID0gbWV0cmljWyJh"
        "dmFpbGFiaWxpdHkiXQogICAgICAgIHJlcXVpcmUoCiAgICAgICAgICAgIGF2YWlsYWJsZSBpbiAo"
        "Im1lYXN1cmVkIiwgImVzdGltYXRlZCIsICJ1bmF2YWlsYWJsZSIpLAogICAgICAgICAgICAiSW52"
        "YWxpZCBhdmFpbGFiaWxpdHkiLAogICAgICAgICkKICAgICAgICBpZiBhdmFpbGFibGUgPT0gInVu"
        "YXZhaWxhYmxlIjoKICAgICAgICAgICAgcmVxdWlyZSgKICAgICAgICAgICAgICAgIG1ldHJpY1si"
        "dmFsdWUiXSBpcyBOb25lCiAgICAgICAgICAgICAgICBhbmQgbWV0cmljWyJzb3VyY2UiXSBpcyBO"
        "b25lCiAgICAgICAgICAgICAgICBhbmQgbm9uZW1wdHkobWV0cmljWyJyZWFzb24iXSksCiAgICAg"
        "ICAgICAgICAgICAiTWlzc2luZyBtZXRyaWMgcmVxdWlyZXMgbnVsbCBhbmQgcmVhc29uIiwKICAg"
        "ICAgICAgICAgKQogICAgICAgIGVsc2U6CiAgICAgICAgICAgIHZhbHVlID0gbWV0cmljWyJ2YWx1"
        "ZSJdCiAgICAgICAgICAgIHJlcXVpcmUoCiAgICAgICAgICAgICAgICB0eXBlKHZhbHVlKSBpbiAo"
        "aW50LCBmbG9hdCkgYW5kIG1hdGguaXNmaW5pdGUodmFsdWUpIGFuZCB2YWx1ZSA+PSAwLAogICAg"
        "ICAgICAgICAgICAgIk1ldHJpYyB2YWx1ZSBtdXN0IGJlIGZpbml0ZSBhbmQgbm9ubmVnYXRpdmUi"
        "LAogICAgICAgICAgICApCiAgICAgICAgICAgIHJlcXVpcmUoCiAgICAgICAgICAgICAgICBub25l"
        "bXB0eShtZXRyaWNbInNvdXJjZSJdKSBhbmQgbWV0cmljWyJyZWFzb24iXSBpcyBOb25lLAogICAg"
        "ICAgICAgICAgICAgIkF2YWlsYWJsZSBtZXRyaWMgcmVxdWlyZXMgc291cmNlIiwKICAgICAgICAg"
        "ICAgKQogICAgICAgICAgICBpZiBtZXRyaWNbInVuaXQiXSBpbiAoImNvdW50IiwgInRva2VucyIs"
        "ICJjaGFyYWN0ZXJzIik6CiAgICAgICAgICAgICAgICByZXF1aXJlKHR5cGUodmFsdWUpIGlzIGlu"
        "dCwgIkRpc2NyZXRlIG1ldHJpYyByZXF1aXJlcyBhbiBpbnRlZ2VyIikKICAgICAgICAgICAgaWYg"
        "bWV0cmljWyJ1bml0Il0gPT0gInJhdGlvIjoKICAgICAgICAgICAgICAgIHJlcXVpcmUodmFsdWUg"
        "PD0gMSwgIlJhdGlvIG91dCBvZiByYW5nZSIpCiAgICAgICAgICAgIGlmIG5hbWUgaW4gR1JPVU5E"
        "X1RSVVRIX01FVFJJQ1M6CiAgICAgICAgICAgICAgICByZXF1aXJlKAogICAgICAgICAgICAgICAg"
        "ICAgIGdyb3VuZCBpcyBub3QgTm9uZSBhbmQgbm9uZW1wdHkoZ3JvdW5kWyJkZWZpbml0aW9ucyJd"
        "LmdldChuYW1lKSksCiAgICAgICAgICAgICAgICAgICAgIlJlbGV2YW5jZSBtZXRyaWNzIHJlcXVp"
        "cmUgcG9zdC1ydW4gbWFwcGluZyBhbmQgZGVmaW5pdGlvbnMiLAogICAgICAgICAgICAgICAgKQog"
        "ICAgICAgIGlmIHN0YXR1cyA9PSAiTk9UX1JVTiI6CiAgICAgICAgICAgIHJlcXVpcmUoYXZhaWxh"
        "YmxlID09ICJ1bmF2YWlsYWJsZSIsICJOT1RfUlVOIGNhbm5vdCBoYXZlIG1lYXN1cmVtZW50cyIp"
        "CiAgICBjYW5vbmljYWxfanNvbihyZWNvcmQpCgoKZGVmIHZhbGlkYXRlX3J1bihtYW5pZmVzdDog"
        "ZGljdFtzdHIsIEFueV0sIHJlY29yZHM6IGxpc3RbZGljdFtzdHIsIEFueV1dKSAtPiBOb25lOgog"
        "ICAgdmFsaWRhdGVfbWFuaWZlc3QobWFuaWZlc3QpCiAgICBzZWVuOiBzZXRbc3RyXSA9IHNldCgp"
        "CiAgICBmb3IgcmVjb3JkIGluIHJlY29yZHM6CiAgICAgICAgdmFsaWRhdGVfcmVjb3JkKHJlY29y"
        "ZCwgbWFuaWZlc3QpCiAgICAgICAgcmVxdWlyZShyZWNvcmRbImluc3RhbmNlX2lkIl0gbm90IGlu"
        "IHNlZW4sICJEdXBsaWNhdGUgdGFzay9hcm0gcmVjb3JkIikKICAgICAgICBzZWVuLmFkZChyZWNv"
        "cmRbImluc3RhbmNlX2lkIl0pCiAgICByZXF1aXJlKAogICAgICAgIHNlZW4gPT0ge3RbImluc3Rh"
        "bmNlX2lkIl0gZm9yIHQgaW4gbWFuaWZlc3RbImNvaG9ydCJdfSwKICAgICAgICAiRXZlcnkgY29o"
        "b3J0IHRhc2sgcmVxdWlyZXMgYSByZWNvcmQsIGluY2x1ZGluZyBOT1RfUlVOIiwKICAgICkK"
    ),
    "src/mostaza/evaluation/import_results.py": (
        "..."
        "Il9fbWFpbl9fIjoKICAgIG1haW4oKQo="
    ),
    "src/mostaza/evaluation/runner.py": (
        "IiIiVGhpbiBleGVjdXRpb24gYW5kIHByb3ZlbmFuY2UgYWRhcHRlciB0byBwaW5uZWQgb2ZmaWNp"
        "YWwgS2FnZ2xlIEFQSXMuIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgpp"
        "bXBvcnQgZGF0YWNsYXNzZXMKaW1wb3J0IGltcG9ydGxpYgppbXBvcnQgemlwZmlsZQpmcm9tIGlt"
        "cG9ydGxpYi5tZXRhZGF0YSBpbXBvcnQgdmVyc2lvbgpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgK"
        "ZnJvbSB0eXBpbmcgaW1wb3J0IEFueQoKZnJvbSBtb3N0YXphLmV2YWx1YXRpb24uY29udHJhY3Rz"
        "IGltcG9ydCAoCiAgICBNT0RFTCwKICAgIGNhbm9uaWNhbF9qc29uLAogICAgcmVxdWlyZSwKICAg"
        "IHNoYTI1NiwKICAgIHZhbGlkYXRlX21hbmlmZXN0LAopCmZyb20gbW9zdGF6YS5ldmFsdWF0aW9u"
        "LnRhc2tzIGltcG9ydCBsb2FkX2FnZW50X3Rhc2tzLCB2YWxpZGF0ZV9zbmFwc2hvdF9oaXN0b3J5"
        "CmZyb20gbW9zdGF6YS5wYWNrYWdpbmcgaW1wb3J0IGNvbGxlY3Rfc3VibWlzc2lvbgoKV0hFRUxf"
        "UElOUyA9IHsKICAgICJzd2VnZW1tYS0wLjIuMTEtcHkzLW5vbmUtYW55LndobCI6ICgKICAgICAg"
        "ICAiOWFkODc2NTgxZjNhY2RlOGVjM2MzNjk4NThmMDgzNGUyYjlmNDFhM2NmZmNlM2NhNDJmMDlj"
        "YjA5MGFlYTlkMyIKICAgICksCiAgICAiYWRrX3N1Ym1pc3Npb24tMC4yLjEzLXB5My1ub25lLWFu"
        "eS53aGwiOiAoCiAgICAgICAgImYxNTZmMGJkODQ1ODBjY2I5MzM0NzI2ZDExNGQ3YTU0ODFmMTZk"
        "M2E2ZTI4ODdjYzRkMjc2Y2VkMGVmOGQ5OWQiCiAgICApLAogICAgImFka19ldmFsX2NvcmUtMC4x"
        "LjAtcHkzLW5vbmUtYW55LndobCI6ICgKICAgICAgICAiZTYxNGUxYjJmZThkODg2YTYzNDg3MWU0"
        "YzU3Zjk4ODdlMWNjNjZlOGMyNzMwMGY4NTAzYTQzYzZlM2NhODJmMyIKICAgICksCn0KCgpkZWYg"
        "ZmlsZV9kaWdlc3QocGF0aDogUGF0aCkgLT4gc3RyOgogICAgaW1wb3J0IGhhc2hsaWIKCiAgICBo"
        "YXNoZXIgPSBoYXNobGliLnNoYTI1NigpCiAgICB3aXRoIHBhdGgub3BlbigicmIiKSBhcyBzdHJl"
        "YW06CiAgICAgICAgZm9yIGJsb2NrIGluIGl0ZXIobGFtYmRhOiBzdHJlYW0ucmVhZCgxMDI0ICog"
        "MTAyNCksIGIiIik6CiAgICAgICAgICAgIGhhc2hlci51cGRhdGUoYmxvY2spCiAgICByZXR1cm4g"
        "aGFzaGVyLmhleGRpZ2VzdCgpCgoKZGVmIGRlc2NyaWJlX3J1bnRpbWUoY29uZmlnOiBBbnksIHZs"
        "bG1fY29uZmlnOiBBbnkpIC0+IGRpY3Rbc3RyLCBBbnldOgogICAgIiIiQ2FwdHVyZSBhY3R1YWwg"
        "b3JnYW5pemVyIGNvbmZpZ3VyYXRpb24gb2JqZWN0cywgd2l0aG91dCBlbmRwb2ludCBjcmVkZW50"
        "aWFscy4iIiIKICAgIGxvYWRlciA9IGltcG9ydGxpYi5pbXBvcnRfbW9kdWxlKCJhZGtfc3VibWlz"
        "c2lvbi55YW1sX2xvYWRlciIpCiAgICByYXcgPSBsb2FkZXIubG9hZF95YW1sKAogICAgICAgIFBh"
        "dGgoY29uZmlnLnN1Ym1pc3Npb25fZGlyKSAvICJhZ2VudC55YW1sIiwKICAgICAgICByb290X2Rp"
        "cj1QYXRoKGNvbmZpZy5zdWJtaXNzaW9uX2RpciksCiAgICAgICAgbGltaXRzPWNvbmZpZy5saW1p"
        "dHMsCiAgICApCgogICAgZGVmIG1vZGVsX2R1bXAodmFsdWU6IEFueSkgLT4gQW55OgogICAgICAg"
        "IHJldHVybiBOb25lIGlmIHZhbHVlIGlzIE5vbmUgZWxzZSB2YWx1ZS5tb2RlbF9kdW1wKG1vZGU9"
        "Impzb24iKQoKICAgIHZhbHVlID0gewogICAgICAgICJpbmZlcmVuY2UiOiB7CiAgICAgICAgICAg"
        "ICJzZXJ2ZXIiOiBkYXRhY2xhc3Nlcy5hc2RpY3QodmxsbV9jb25maWcpLAogICAgICAgICAgICAi"
        "YWdlbnRfZ2VuZXJhdGVfY29udGVudF9jb25maWciOiByYXcuZ2V0KCJnZW5lcmF0ZV9jb250ZW50"
        "X2NvbmZpZyIpLAogICAgICAgICAgICAib2ZmaWNpYWxfZ2VuZXJhdGlvbl9kZWZhdWx0cyI6IGNv"
        "bmZpZy5nZW5lcmF0aW9uX2NvbnN0cmFpbnRzLmRlZmF1bHRzLAogICAgICAgICAgICAiY2FjaGUi"
        "OiBtb2RlbF9kdW1wKGNvbmZpZy5jb250ZXh0X2NhY2hlX2NvbmZpZyksCiAgICAgICAgICAgICJj"
        "b21wYWN0aW9uIjogbW9kZWxfZHVtcChjb25maWcuZXZlbnRzX2NvbXBhY3Rpb25fY29uZmlnKSwK"
        "ICAgICAgICB9LAogICAgICAgICJidWRnZXRzIjogewogICAgICAgICAgICAiZXZhbHVhdGlvbiI6"
        "IGRhdGFjbGFzc2VzLmFzZGljdChjb25maWcuYnVkZ2V0KSwKICAgICAgICAgICAgImhhcm5lc3Mi"
        "OiBkYXRhY2xhc3Nlcy5hc2RpY3QoY29uZmlnLmhhcm5lc3MpLAogICAgICAgIH0sCiAgICAgICAg"
        "ImNvbmZpZ19zaGEyNTYiOiBzaGEyNTYoCiAgICAgICAgICAgIGNhbm9uaWNhbF9qc29uKAogICAg"
        "ICAgICAgICAgICAgewogICAgICAgICAgICAgICAgICAgIG5hbWU6IHNoYTI1NihkYXRhKQogICAg"
        "ICAgICAgICAgICAgICAgIGZvciBuYW1lLCBkYXRhIGluIGNvbGxlY3Rfc3VibWlzc2lvbigKICAg"
        "ICAgICAgICAgICAgICAgICAgICAgUGF0aChjb25maWcuc3VibWlzc2lvbl9kaXIpCiAgICAgICAg"
        "ICAgICAgICAgICAgKS5pdGVtcygpCiAgICAgICAgICAgICAgICB9CiAgICAgICAgICAgICkKICAg"
        "ICAgICApLAogICAgfQogICAgIyBOb3JtYWxpemUgdHVwbGVzIGluIG9mZmljaWFsIGRhdGFjbGFz"
        "c2VzIGV4YWN0bHkgYXMgcGVyc2lzdGVkIEpTT04gZG9lcy4KICAgIGltcG9ydCBqc29uCgogICAg"
        "cmVzdWx0OiBkaWN0W3N0ciwgQW55XSA9IGpzb24ubG9hZHMoY2Fub25pY2FsX2pzb24odmFsdWUp"
        "KQogICAgcmV0dXJuIHJlc3VsdAoKCmRlZiBzbmFwc2hvdF9kaWdlc3QoCiAgICBzbmFwc2hvdHNf"
        "ZGlyOiBQYXRoLCBpbnN0YW5jZV9pZDogc3RyLCByZXBvOiBzdHIsIGRhdGFfcm9vdDogUGF0aAop"
        "IC0+IHN0cjoKICAgICIiIkJpbmQgYWxsIG9mZmljaWFsIHJlY29uc3RydWN0aW9uIGlucHV0czsg"
        "cm9sZXMgZm9sbG93IG9yZ2FuaXplciByZXNvbHV0aW9uLiIiIgogICAgcmVzb2x2ZSA9IGltcG9y"
        "dGxpYi5pbXBvcnRfbW9kdWxlKAogICAgICAgICJzd2VnZW1tYS5kZWR1cGxpY2F0aW9uIgogICAg"
        "KS5yZXNvbHZlX3Rhc2tfc25hcHNob3RfcGF0aHMKICAgIGVmZmVjdGl2ZSwgYmFzZSwgZGVsdGEg"
        "PSByZXNvbHZlKHNuYXBzaG90c19kaXIsIGluc3RhbmNlX2lkLCByZXBvKQogICAgY29tcG9uZW50"
        "cyA9IFtdCiAgICBmb3Igcm9sZSwgcGF0aCBpbiAoKCJlZmZlY3RpdmUiLCBlZmZlY3RpdmUpLCAo"
        "ImJhc2UiLCBiYXNlKSwgKCJkZWx0YSIsIGRlbHRhKSk6CiAgICAgICAgaWYgcGF0aCBpcyBub3Qg"
        "Tm9uZToKICAgICAgICAgICAgcGF0aCA9IFBhdGgocGF0aCkKICAgICAgICAgICAgcmVxdWlyZShw"
        "YXRoLmlzX2ZpbGUoKSwgIkZyb3plbiBzbmFwc2hvdCBjb21wb25lbnQgbWlzc2luZyIpCiAgICAg"
        "ICAgICAgIGNvbXBvbmVudHMuYXBwZW5kKAogICAgICAgICAgICAgICAgewogICAgICAgICAgICAg"
        "ICAgICAgICJyb2xlIjogcm9sZSwKICAgICAgICAgICAgICAgICAgICAicGF0aCI6IHN0cihwYXRo"
        "LnJlbGF0aXZlX3RvKGRhdGFfcm9vdCkpLAogICAgICAgICAgICAgICAgICAgICJzaGEyNTYiOiBm"
        "aWxlX2RpZ2VzdChwYXRoKSwKICAgICAgICAgICAgICAgIH0KICAgICAgICAgICAgKQogICAgcmV0"
        "dXJuIHNoYTI1NihjYW5vbmljYWxfanNvbihjb21wb25lbnRzKSkKCgpkZWYgdmFsaWRhdGVfYmlu"
        "ZGluZ3MoCiAgICBjb25maWc6IEFueSwKICAgIG1hbmlmZXN0OiBkaWN0W3N0ciwgQW55XSwKICAg"
        "ICosCiAgICBzZXJ2ZXI6IEFueSwKICAgIGFyY2hpdmVfcGF0aDogUGF0aCwKICAgIHdoZWVsaG91"
        "c2VfZGlyOiBQYXRoLAopIC0+IE5vbmU6CiAgICAiIiJGYWlsIGJlZm9yZSBhbnkgYWdlbnQgY2Fs"
        "bCBpZiBkZWNsYXJhdGlvbnMgZGlmZmVyIGZyb20gZWZmZWN0aXZlIGlucHV0cy4iIiIKICAgIHZh"
        "bGlkYXRlX21hbmlmZXN0KG1hbmlmZXN0KQogICAgcmVxdWlyZSgKICAgICAgICBtYW5pZmVzdFsi"
        "cnVuX2tpbmQiXSA9PSAicmVhbF9rYWdnbGUiLCAiT2ZmaWNpYWwgcnVubmVyIHJlcXVpcmVzIHJl"
        "YWwgcnVuIGtpbmQiCiAgICApCiAgICByZXF1aXJlKG5vdCBjb25maWcuc2tpcF9hZ2VudF9wYXRj"
        "aCwgIlBhdGNoIGdlbmVyYXRpb24gY2Fubm90IGJlIHNraXBwZWQiKQogICAgcmVxdWlyZShjb25m"
        "aWcuc2FuZGJveCA9PSAic3VicHJvY2VzcyIsICJVc2UgdGhlIG9mZmljaWFsIG5vdGVib29rIHNh"
        "bmRib3ggYmFja2VuZCIpCiAgICByZXF1aXJlKAogICAgICAgIG5vdCBjb25maWcuc2hhcmRfaW5k"
        "ZXggYW5kIG5vdCBjb25maWcubnVtX3NoYXJkcyBhbmQgY29uZmlnLmNvbmN1cnJlbmN5ID09IDEs"
        "CiAgICAgICAgIlBoYXNlIDEgcnVucyB0aGUgY29tcGxldGUgY29ob3J0IHNlcmlhbGx5IHdpdGhv"
        "dXQgc2hhcmRzIiwKICAgICkKICAgIHJlcXVpcmUoCiAgICAgICAgY29uZmlnLnRhc2tfaWRzIGlz"
        "IE5vbmUKICAgICAgICBvciBjb25maWcudGFza19pZHMgPT0gW3RbImluc3RhbmNlX2lkIl0gZm9y"
        "IHQgaW4gbWFuaWZlc3RbImNvaG9ydCJdXSwKICAgICAgICAiQ29uZmlndXJlZCB0YXNrIElEcyBk"
        "aWZmZXIgZnJvbSBjb2hvcnQiLAogICAgKQogICAgbW9kZWxfcGF0aCA9IFBhdGgoc2VydmVyLmNv"
        "bmZpZy5tb2RlbCkKICAgIHJlcXVpcmUoCiAgICAgICAgbW9kZWxfcGF0aC5uYW1lID09ICIyIiBh"
        "bmQgbW9kZWxfcGF0aC5wYXJlbnQubmFtZSA9PSBNT0RFTCwKICAgICAgICAiUmVxdWlyZWQgcGlu"
        "bmVkIDMxQiBtb2RlbCBwYXRoIG1pc3NpbmciLAogICAgKQogICAgcmVxdWlyZSgKICAgICAgICBz"
        "ZXJ2ZXIuY29uZmlnLnRlbnNvcl9wYXJhbGxlbF9zaXplID09IDQKICAgICAgICBhbmQgbm90IHNl"
        "cnZlci5jb25maWcuZW5hYmxlX2xvcmEKICAgICAgICBhbmQgc2VydmVyLmNvbmZpZy5ob3N0ID09"
        "ICIxMjcuMC4wLjEiLAogICAgICAgICJPZmZpY2lhbCBMNHg0IGxvY2FsIG1vZGVsIHNlcnZpbmcg"
        "cmVxdWlyZWQiLAogICAgKQogICAgcmVxdWlyZSgKICAgICAgICBjb25maWcubW9kZWxzLmxpc3Rf"
        "YXZhaWxhYmxlKCkgPT0gW01PREVMXSwKICAgICAgICAiUmVnaXN0cnkgbXVzdCBjb250YWluIG9u"
        "bHkgdGhlIHJlcXVpcmVkIG1vZGVsIiwKICAgICkKICAgIGJpbmRpbmcgPSBjb25maWcubW9kZWxz"
        "LmdldChNT0RFTCkKICAgIHJlcXVpcmUoCiAgICAgICAgZ2V0YXR0cihiaW5kaW5nLCAibW9kZWwi"
        "LCBOb25lKQogICAgICAgID09ICJvcGVuYWkvIgogICAgICAgICsgc3RyKGdldGF0dHIoc2VydmVy"
        "LCAiZWZmZWN0aXZlX21vZGVsX3BhdGgiLCBzZXJ2ZXIuY29uZmlnLm1vZGVsKSksCiAgICAgICAg"
        "IlJlZ2lzdHJ5IG1vZGVsIGRvZXMgbm90IG1hdGNoIHRoZSBhY3R1YWwgc2VydmVyIiwKICAgICkK"
        "ICAgIHJlcXVpcmUoCiAgICAgICAgZ2V0YXR0cihiaW5kaW5nLCAiX2FkZGl0aW9uYWxfYXJncyIs"
        "IHt9KS5nZXQoImFwaV9iYXNlIikgPT0gc2VydmVyLmJhc2VfdXJsLAogICAgICAgICJSZWdpc3Ry"
        "eSBlbmRwb2ludCBkb2VzIG5vdCBtYXRjaCB0aGUgbG9jYWwgc2VydmVyIiwKICAgICkKICAgIHJ1"
        "bnRpbWUgPSBkZXNjcmliZV9ydW50aW1lKGNvbmZpZywgc2VydmVyLmNvbmZpZykKICAgIGZvciBm"
        "aWVsZCBpbiAoImluZmVyZW5jZSIsICJidWRnZXRzIiwgImNvbmZpZ19zaGEyNTYiKToKICAgICAg"
        "ICByZXF1aXJlKAogICAgICAgICAgICBjYW5vbmljYWxfanNvbihydW50aW1lW2ZpZWxkXSkgPT0g"
        "Y2Fub25pY2FsX2pzb24obWFuaWZlc3RbZmllbGRdKSwKICAgICAgICAgICAgZiJBY3R1YWwgcnVu"
        "dGltZSBkaWZmZXJzIGZyb20gbWFuaWZlc3Qge2ZpZWxkfSIsCiAgICAgICAgKQogICAgcmVxdWly"
        "ZSgKICAgICAgICBmaWxlX2RpZ2VzdChhcmNoaXZlX3BhdGgpID09IG1hbmlmZXN0WyJhcmNoaXZl"
        "X3NoYTI1NiJdLCAiQXJjaGl2ZSBoYXNoIG1pc21hdGNoIgogICAgKQogICAgZmlsZXMgPSBjb2xs"
        "ZWN0X3N1Ym1pc3Npb24oUGF0aChjb25maWcuc3VibWlzc2lvbl9kaXIpKQogICAgd2l0aCB6aXBm"
        "aWxlLlppcEZpbGUoYXJjaGl2ZV9wYXRoKSBhcyBhcmNoaXZlOgogICAgICAgIG5hbWVzID0gYXJj"
        "aGl2ZS5uYW1lbGlzdCgpCiAgICAgICAgcmVxdWlyZSgKICAgICAgICAgICAgbGVuKG5hbWVzKSA9"
        "PSBsZW4oc2V0KG5hbWVzKSkgYW5kIHNldChuYW1lcykgPT0gc2V0KGZpbGVzKSwKICAgICAgICAg"
        "ICAgIkFyY2hpdmUgZGlmZmVycyBmcm9tIGV4ZWN1dGVkIHN1Ym1pc3Npb24gY2xvc3VyZSIsCiAg"
        "ICAgICAgKQogICAgICAgIGZvciBuYW1lLCBjb250ZW50IGluIGZpbGVzLml0ZW1zKCk6CiAgICAg"
        "ICAgICAgIHJlcXVpcmUoCiAgICAgICAgICAgICAgICBhcmNoaXZlLnJlYWQobmFtZSkgPT0gY29u"
        "dGVudCwgIkFyY2hpdmUvZXhlY3V0ZWQgY29uZmlnIGJ5dGVzIGRpZmZlciIKICAgICAgICAgICAg"
        "KQogICAgZGlzY292ZXJ5ID0gaW1wb3J0bGliLmltcG9ydF9tb2R1bGUoImFka19zdWJtaXNzaW9u"
        "LmRpc2NvdmVyeSIpCiAgICByZXF1aXJlKAogICAgICAgIGRpc2NvdmVyeS5kaXNjb3Zlcl9kZWNs"
        "YXJlZF9tb2RlbHMoUGF0aChjb25maWcuc3VibWlzc2lvbl9kaXIpKSA9PSB7TU9ERUx9LAogICAg"
        "ICAgICJFdmVyeSBkZWNsYXJhdGl2ZSBhZ2VudCBtdXN0IHVzZSB0aGUgcmVxdWlyZWQgbW9kZWwi"
        "LAogICAgKQogICAgZm9yIHBhY2thZ2UsIGV4cGVjdGVkIGluICgKICAgICAgICAoInN3ZWdlbW1h"
        "IiwgIjAuMi4xMSIpLAogICAgICAgICgiYWRrLXN1Ym1pc3Npb24iLCAiMC4yLjEzIiksCiAgICAg"
        "ICAgKCJhZGstZXZhbC1jb3JlIiwgIjAuMS4wIiksCiAgICApOgogICAgICAgIHJlcXVpcmUodmVy"
        "c2lvbihwYWNrYWdlKSA9PSBleHBlY3RlZCwgIlBpbm5lZCBvZmZpY2lhbCBwYWNrYWdlcyByZXF1"
        "aXJlZCIpCiAgICBmb3IgbmFtZSwgZXhwZWN0ZWQgaW4gV0hFRUxfUElOUy5pdGVtcygpOgogICAg"
        "ICAgIHJlcXVpcmUoCiAgICAgICAgICAgIGZpbGVfZGlnZXN0KHdoZWVsaG91c2VfZGlyIC8gbmFt"
        "ZSkgPT0gZXhwZWN0ZWQsCiAgICAgICAgICAgICJQaW5uZWQgb2ZmaWNpYWwgd2hlZWwgY2hhbmdl"
        "ZCIsCiAgICAgICAgKQogICAgcmVxdWlyZSgKICAgICAgICBtYW5pZmVzdFsicmVzb3VyY2VzIl1b"
        "Imhhcm5lc3MiXVsic2hhMjU2Il0KICAgICAgICA9PSBXSEVFTF9QSU5TWyJzd2VnZW1tYS0wLjIu"
        "MTEtcHkzLW5vbmUtYW55LndobCJdLAogICAgICAgICJIYXJuZXNzIGRlY2xhcmF0aW9uIGRpZmZl"
        "cnMgZnJvbSB2ZXJpZmllZCB3aGVlbCIsCiAgICApCiAgICB3aGVlbF9pbnZlbnRvcnkgPSBbCiAg"
        "ICAgICAgeyJuYW1lIjogcC5uYW1lLCAic2hhMjU2IjogZmlsZV9kaWdlc3QocCl9CiAgICAgICAg"
        "Zm9yIHAgaW4gc29ydGVkKHdoZWVsaG91c2VfZGlyLmdsb2IoIioud2hsIikpCiAgICBdCiAgICBy"
        "ZXF1aXJlKAogICAgICAgIHNoYTI1NihjYW5vbmljYWxfanNvbih3aGVlbF9pbnZlbnRvcnkpKQog"
        "ICAgICAgID09IG1hbmlmZXN0WyJyZXNvdXJjZXMiXVsid2hlZWxob3VzZSJdWyJzaGEyNTYiXSwK"
        "ICAgICAgICAiV2hlZWxob3VzZSBpbnZlbnRvcnkgZGlmZmVycyBmcm9tIG1hbmlmZXN0IiwKICAg"
        "ICkKICAgIHRhc2tzX3BhdGggPSBQYXRoKGNvbmZpZy50YXNrc19wYXRoKQogICAgcmVxdWlyZSgK"
        "ICAgICAgICBmaWxlX2RpZ2VzdCh0YXNrc19wYXRoKSA9PSBtYW5pZmVzdFsicmVzb3VyY2VzIl1b"
        "ImRhdGFzZXQiXVsic2hhMjU2Il0sCiAgICAgICAgIkRhdGFzZXQgY2hhbmdlZCBhZnRlciBzZWxl"
        "Y3Rpb24iLAogICAgKQogICAgdmlzaWJsZSA9IHt0YXNrLmluc3RhbmNlX2lkOiB0YXNrIGZvciB0"
        "YXNrIGluIGxvYWRfYWdlbnRfdGFza3ModGFza3NfcGF0aCl9CiAgICBmb3Igc2VsZWN0ZWQgaW4g"
        "bWFuaWZlc3RbImNvaG9ydCJdOgogICAgICAgIHRhc2sgPSB2aXNpYmxlLmdldChzZWxlY3RlZFsi"
        "aW5zdGFuY2VfaWQiXSkKICAgICAgICByZXF1aXJlKAogICAgICAgICAgICB0YXNrIGlzIG5vdCBO"
        "b25lCiAgICAgICAgICAgIGFuZCB0YXNrLmJhc2VfY29tbWl0ID09IHNlbGVjdGVkWyJiYXNlX2Nv"
        "bW1pdCJdCiAgICAgICAgICAgIGFuZCB0YXNrLnJlcG8gPT0gc2VsZWN0ZWRbInJlcG8iXSwKICAg"
        "ICAgICAgICAgIkZyb3plbiB0YXNrIG1ldGFkYXRhIG1pc21hdGNoIiwKICAgICAgICApCiAgICAg"
        "ICAgcmVxdWlyZSgKICAgICAgICAgICAgc25hcHNob3RfZGlnZXN0KAogICAgICAgICAgICAgICAg"
        "UGF0aChjb25maWcuc25hcHNob3RzX2RpciksCiAgICAgICAgICAgICAgICBzZWxlY3RlZFsiaW5z"
        "dGFuY2VfaWQiXSwKICAgICAgICAgICAgICAgIHNlbGVjdGVkWyJyZXBvIl0sCiAgICAgICAgICAg"
        "ICAgICB0YXNrc19wYXRoLnBhcmVudCwKICAgICAgICAgICAgKQogICAgICAgICAgICA9PSBzZWxl"
        "Y3RlZFsic25hcHNob3Rfc2hhMjU2Il0sCiAgICAgICAgICAgICJGcm96ZW4gcmVjb25zdHJ1Y3Rp"
        "b24gaW5wdXRzIGNoYW5nZWQiLAogICAgICAgICkKICAgICAgICByZWNlaXB0cyA9IG1hbmlmZXN0"
        "WyJlbnZpcm9ubWVudCJdLmdldCgic25hcHNob3RfdmFsaWRhdGlvbiIsIHt9KQogICAgICAgIHJl"
        "cXVpcmUoc2VsZWN0ZWRbImluc3RhbmNlX2lkIl0gaW4gcmVjZWlwdHMsICJTbmFwc2hvdCBoaXN0"
        "b3J5IHJlY2VpcHQgbWlzc2luZyIpCiAgICAgICAgcmVzb2x2ZSA9IGltcG9ydGxpYi5pbXBvcnRf"
        "bW9kdWxlKAogICAgICAgICAgICAic3dlZ2VtbWEuZGVkdXBsaWNhdGlvbiIKICAgICAgICApLnJl"
        "c29sdmVfdGFza19zbmFwc2hvdF9wYXRocwogICAgICAgIGVmZmVjdGl2ZSwgYmFzZSwgZGVsdGEg"
        "PSByZXNvbHZlKAogICAgICAgICAgICBjb25maWcuc25hcHNob3RzX2Rpciwgc2VsZWN0ZWRbImlu"
        "c3RhbmNlX2lkIl0sIHNlbGVjdGVkWyJyZXBvIl0KICAgICAgICApCiAgICAgICAgcmVxdWlyZSgK"
        "ICAgICAgICAgICAgYmFzZSBpcyBOb25lIGFuZCBkZWx0YSBpcyBOb25lLAogICAgICAgICAgICAi"
        "SGlzdG9yeSByZWNlaXB0IHJlcXVpcmVzIHJldmlld2VkIHN0YW5kYWxvbmUgc25hcHNob3QiLAog"
        "ICAgICAgICkKICAgICAgICB2YWxpZGF0ZV9zbmFwc2hvdF9oaXN0b3J5KAogICAgICAgICAgICBQ"
        "YXRoKGVmZmVjdGl2ZSksIHNlbGVjdGVkLCByZWNlaXB0c1tzZWxlY3RlZFsiaW5zdGFuY2VfaWQi"
        "XV0KICAgICAgICApCgoKYXN5bmMgZGVmIGV2YWx1YXRlX29mZmljaWFsKAogICAgY29uZmlnOiBB"
        "bnksCiAgICBtYW5pZmVzdDogZGljdFtzdHIsIEFueV0sCiAgICAqLAogICAgc2VydmVyOiBBbnkg"
        "PSBOb25lLAogICAgYXJjaGl2ZV9wYXRoOiBQYXRoIHwgTm9uZSA9IE5vbmUsCiAgICB3aGVlbGhv"
        "dXNlX2RpcjogUGF0aCB8IE5vbmUgPSBOb25lLAopIC0+IGxpc3RbUGF0aF06CiAgICAiIiJEZWxl"
        "Z2F0ZSB0byBvZmZpY2lhbCBldmFsdWF0ZV90YXNrIGFuZCBwZXJzaXN0IHNvdXJjZS1ub3JtYWxp"
        "emFibGUgcmF3IHJlc3VsdHMuCgogICAgTW9kZWwgc3RhcnR1cCByZW1haW5zIHRoZSBvZmZpY2lh"
        "bCBub3RlYm9vaydzIFZsbG1TZXJ2ZXIgcmVzcG9uc2liaWxpdHkuIFRoZQogICAgcmF3IGV4dGVu"
        "c2lvbiByZWNvcmRzIGV4YWN0bHkgd2hpY2ggZ2VuZXJhdGVkIHBhdGNoIHRoZSBvcmdhbml6ZXIg"
        "dmVyaWZpZWQ7CiAgICBubyByZWZlcmVuY2UgcGF0Y2gvdGVzdCBjb250ZW50cyBlbnRlciB0aGVz"
        "ZSBvdXRwdXRzLgogICAgIiIiCiAgICB2YWxpZGF0ZV9tYW5pZmVzdChtYW5pZmVzdCkKICAgIHJl"
        "cXVpcmUoCiAgICAgICAgbWFuaWZlc3RbInJ1bl9raW5kIl0gPT0gInJlYWxfa2FnZ2xlIiwgIk9m"
        "ZmljaWFsIHJ1bm5lciByZXF1aXJlcyByZWFsIHJ1biBraW5kIgogICAgKQogICAgcmVxdWlyZShQ"
        "YXRoKCIva2FnZ2xlL2lucHV0IikuaXNfZGlyKCksICJPZmZpY2lhbCBleGVjdXRpb24gcmVxdWly"
        "ZXMgS2FnZ2xlIikKICAgIHJlcXVpcmUoCiAgICAgICAgc2VydmVyIGlzIG5vdCBOb25lIGFuZCBh"
        "cmNoaXZlX3BhdGggaXMgbm90IE5vbmUgYW5kIHdoZWVsaG91c2VfZGlyIGlzIG5vdCBOb25lLAog"
        "ICAgICAgICJBY3R1YWwgc2VydmVyLCBhcmNoaXZlIGFuZCB3aGVlbGhvdXNlIGJpbmRpbmdzIGFy"
        "ZSByZXF1aXJlZCIsCiAgICApCiAgICByZXF1aXJlKAogICAgICAgIHNlcnZlci5wcm9jZXNzIGlz"
        "IG5vdCBOb25lIGFuZCBzZXJ2ZXIucHJvY2Vzcy5wb2xsKCkgaXMgTm9uZSwKICAgICAgICAiQWN0"
        "dWFsIG1vZGVsIHNlcnZlciBwcm9jZXNzIG11c3QgYmUgcnVubmluZyIsCiAgICApCiAgICByZXF1"
        "aXJlKFBhdGgoc2VydmVyLmNvbmZpZy5tb2RlbCkuaXNfZGlyKCksICJBY3R1YWwgcmVxdWlyZWQg"
        "bW9kZWwgbW91bnQgbWlzc2luZyIpCiAgICBhc3NlcnQgYXJjaGl2ZV9wYXRoIGlzIG5vdCBOb25l"
        "IGFuZCB3aGVlbGhvdXNlX2RpciBpcyBub3QgTm9uZQogICAgdmFsaWRhdGVfYmluZGluZ3MoCiAg"
        "ICAgICAgY29uZmlnLAogICAgICAgIG1hbmlmZXN0LAogICAgICAgIHNlcnZlcj1zZXJ2ZXIsCiAg"
        "ICAgICAgYXJjaGl2ZV9wYXRoPWFyY2hpdmVfcGF0aCwKICAgICAgICB3aGVlbGhvdXNlX2Rpcj13"
        "aGVlbGhvdXNlX2RpciwKICAgICkKICAgIG9mZmljaWFsX21vZGVscyA9IGltcG9ydGxpYi5pbXBv"
        "cnRfbW9kdWxlKCJzd2VnZW1tYS5tb2RlbHMiKQogICAgb2ZmaWNpYWxfZXZhbHVhdGUgPSBpbXBv"
        "cnRsaWIuaW1wb3J0X21vZHVsZSgic3dlZ2VtbWEuZXZhbHVhdGUiKQogICAgdmVyaWZpY2F0aW9u"
        "ID0gaW1wb3J0bGliLmltcG9ydF9tb2R1bGUoInN3ZWdlbW1hLmhhcm5lc3MudmVyaWZpY2F0aW9u"
        "IikKICAgIHRhc2tzID0gewogICAgICAgIHRhc2suaW5zdGFuY2VfaWQ6IHRhc2sKICAgICAgICBm"
        "b3IgdGFzayBpbiBvZmZpY2lhbF9tb2RlbHMubG9hZF90YXNrcyhQYXRoKGNvbmZpZy50YXNrc19w"
        "YXRoKSkKICAgIH0KICAgIG91dHB1dCA9IFBhdGgoY29uZmlnLnJlc3VsdHNfZGlyKQogICAgcmVx"
        "dWlyZSgKICAgICAgICBub3Qgb3V0cHV0LmV4aXN0cygpIG9yIG5vdCBhbnkob3V0cHV0Lml0ZXJk"
        "aXIoKSksCiAgICAgICAgIlJlc3VsdCBkaXJlY3RvcnkgbXVzdCBiZSBmcmVzaCIsCiAgICApCiAg"
        "ICBvdXRwdXQubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgZXZhbHVhdG9y"
        "ID0gb2ZmaWNpYWxfZXZhbHVhdGUuRXZhbHVhdG9yKGNvbmZpZykKICAgIHBhdGhzOiBsaXN0W1Bh"
        "dGhdID0gW10KICAgIHRyeToKICAgICAgICBmb3IgaW5kZXgsIHNlbGVjdGVkIGluIGVudW1lcmF0"
        "ZShtYW5pZmVzdFsiY29ob3J0Il0sIHN0YXJ0PTEpOgogICAgICAgICAgICAjIFVzZSB0aGUgcGlu"
        "bmVkIGV2YWx1YXRvcidzIG93biBob3N0LW9ubHkgc2VjcmV0IGh5ZHJhdGlvbiBzbyB0aGUKICAg"
        "ICAgICAgICAgIyByZWNvcmRlZCBlZmZlY3RpdmUgcGF0Y2ggbWF0Y2hlcyBpdHMgcHJvdGVjdGVk"
        "IHRhcmdldC10ZXN0IHBhdGhzLgogICAgICAgICAgICB0YXNrID0gZXZhbHVhdG9yLl9oeWRyYXRl"
        "X3Rhc2tfZnJvbV9zZWNyZXQodGFza3Nbc2VsZWN0ZWRbImluc3RhbmNlX2lkIl1dKQogICAgICAg"
        "ICAgICByZXN1bHQgPSBhd2FpdCBldmFsdWF0b3IuZXZhbHVhdGVfdGFzaygKICAgICAgICAgICAg"
        "ICAgIHRhc2s9dGFzaywgdGFza19pbmRleD1pbmRleCwgdG90YWxfdGFza3M9bGVuKG1hbmlmZXN0"
        "WyJjb2hvcnQiXSkKICAgICAgICAgICAgKQogICAgICAgICAgICBpZiBub3QgcmVzdWx0LnRyYWNl"
        "X3BhdGggYW5kIHJlc3VsdC50cmFjZSBpcyBub3QgTm9uZToKICAgICAgICAgICAgICAgIHJlc3Vs"
        "dC50cmFjZV9wYXRoID0gdmVyaWZpY2F0aW9uLnNhdmVfdHJhY2VfYXJ0aWZhY3QoCiAgICAgICAg"
        "ICAgICAgICAgICAgcmVzdWx0LnRyYWNlLCBvdXRwdXQsIHRhc2suaW5zdGFuY2VfaWQKICAgICAg"
        "ICAgICAgICAgICkKICAgICAgICAgICAgcmF3ID0gcmVzdWx0Lm1vZGVsX2R1bXAobW9kZT0ianNv"
        "biIsIGV4Y2x1ZGU9eyJ0cmFjZSJ9KQogICAgICAgICAgICBlZmZlY3RpdmUgPSB2ZXJpZmljYXRp"
        "b24uc3RyaXBfcHJvdGVjdGVkX2ZpbGVzX2Zyb21fcGF0Y2goCiAgICAgICAgICAgICAgICByZXN1"
        "bHQuYWdlbnRfcGF0Y2gsCiAgICAgICAgICAgICAgICBleHRyYV9wcm90ZWN0ZWRfcGF0aHM9dGFz"
        "ay50ZXN0X2ZpbGVzLAogICAgICAgICAgICApCiAgICAgICAgICAgIHJhd1siX21vc3RhemEiXSA9"
        "IHsKICAgICAgICAgICAgICAgICJzY2hlbWFfdmVyc2lvbiI6IDEsCiAgICAgICAgICAgICAgICAi"
        "ZWZmZWN0aXZlX2FnZW50X3BhdGNoIjogZWZmZWN0aXZlLAogICAgICAgICAgICAgICAgIm9yaWdp"
        "bmFsX3BhdGNoX3NoYTI1NiI6IHNoYTI1NihyZXN1bHQuYWdlbnRfcGF0Y2guZW5jb2RlKCkpLAog"
        "ICAgICAgICAgICB9CiAgICAgICAgICAgIHBhdGggPSBvdXRwdXQgLyBmIm9mZmljaWFsLXJlc3Vs"
        "dC17aW5kZXg6MDRkfS5qc29uIgogICAgICAgICAgICB3aXRoIHBhdGgub3BlbigieGIiKSBhcyBz"
        "dHJlYW06CiAgICAgICAgICAgICAgICBzdHJlYW0ud3JpdGUoY2Fub25pY2FsX2pzb24ocmF3KSkK"
        "ICAgICAgICAgICAgcGF0aHMuYXBwZW5kKHBhdGgpCiAgICBmaW5hbGx5OgogICAgICAgIGV2YWx1"
        "YXRvci5kb2NrZXIuY2xlYW51cF9hbGwoKQogICAgcmV0dXJuIHBhdGhzCg=="
    ),
    "src/mostaza/evaluation/tasks.py": (
        "IiIiU2VwYXJhdGUgYWdlbnQtdmlzaWJsZSBpc3N1ZSBkYXRhIGZyb20gcHJpdmF0ZSBwb3N0LWdl"
        "bmVyYXRpb24gcmVmZXJlbmNlcy4iIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlv"
        "bnMKCmltcG9ydCBpbwppbXBvcnQganNvbgppbXBvcnQgcmUKaW1wb3J0IHN1YnByb2Nlc3MKaW1w"
        "b3J0IHRhcmZpbGUKZnJvbSBkYXRhY2xhc3NlcyBpbXBvcnQgYXNkaWN0LCBkYXRhY2xhc3MKZnJv"
        "bSBwYXRobGliIGltcG9ydCBQYXRoLCBQdXJlUG9zaXhQYXRoCmZyb20gdHlwaW5nIGltcG9ydCBB"
        "bnkKCgpAZGF0YWNsYXNzKGZyb3plbj1UcnVlKQpjbGFzcyBBZ2VudFRhc2s6CiAgICAiIiJUaGUg"
        "ZW50aXJlIGFsbG93ZWQgaXNzdWUgY29udGV4dC4gTmV2ZXIgc2VyaWFsaXplIHRoZSBzb3VyY2Ug"
        "dGFzayBvYmplY3QuIiIiCgogICAgaW5zdGFuY2VfaWQ6IHN0cgogICAgcmVwbzogc3RyCiAgICBi"
        "YXNlX2NvbW1pdDogc3RyCiAgICBwcm9ibGVtX3N0YXRlbWVudDogc3RyCgogICAgZGVmIHRvX2pz"
        "b24oc2VsZikgLT4gc3RyOgogICAgICAgIHJldHVybiBqc29uLmR1bXBzKGFzZGljdChzZWxmKSwg"
        "c29ydF9rZXlzPVRydWUsIGVuc3VyZV9hc2NpaT1GYWxzZSkKCgpkZWYgYWdlbnRfdGFzayhtZXRh"
        "ZGF0YTogZGljdFtzdHIsIEFueV0pIC0+IEFnZW50VGFzazoKICAgIGZpZWxkcyA9ICgiaW5zdGFu"
        "Y2VfaWQiLCAicmVwbyIsICJiYXNlX2NvbW1pdCIsICJwcm9ibGVtX3N0YXRlbWVudCIpCiAgICBp"
        "ZiBhbnkobm90IGlzaW5zdGFuY2UobWV0YWRhdGEuZ2V0KGspLCBzdHIpIG9yIG5vdCBtZXRhZGF0"
        "YVtrXSBmb3IgayBpbiBmaWVsZHMpOgogICAgICAgICMgRG8gbm90IGluY2x1ZGUgcmVqZWN0ZWQg"
        "dmFsdWVzOiBlcnJvciBtZXNzYWdlcyBtYXkgYmVjb21lIGFnZW50IGxvZ3MuCiAgICAgICAgcmFp"
        "c2UgVmFsdWVFcnJvcigiVGFzayByZXF1aXJlcyBub25lbXB0eSBhbGxvd2xpc3RlZCBzdHJpbmcg"
        "ZmllbGRzIikKICAgIGlmIG5vdCByZS5mdWxsbWF0Y2gociJbMC05YS1mXXs0MH0iLCBtZXRhZGF0"
        "YVsiYmFzZV9jb21taXQiXSk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiVGFzayBiYXNlX2Nv"
        "bW1pdCBtdXN0IGJlIGFuIGV4YWN0IEdpdCBjb21taXQiKQogICAgcmV0dXJuIEFnZW50VGFzaygq"
        "KntrZXk6IG1ldGFkYXRhW2tleV0gZm9yIGtleSBpbiBmaWVsZHN9KQoKCmRlZiBsb2FkX2FnZW50"
        "X3Rhc2tzKHBhdGg6IFBhdGgpIC0+IGxpc3RbQWdlbnRUYXNrXToKICAgIHRhc2tzOiBsaXN0W0Fn"
        "ZW50VGFza10gPSBbXQogICAgc2Vlbjogc2V0W3N0cl0gPSBzZXQoKQogICAgZm9yIGxpbmUgaW4g"
        "cGF0aC5yZWFkX3RleHQoZW5jb2Rpbmc9InV0Zi04Iikuc3BsaXRsaW5lcygpOgogICAgICAgIGlm"
        "IG5vdCBsaW5lLnN0cmlwKCk6CiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgdHJ5OgogICAg"
        "ICAgICAgICBkYXRhID0ganNvbi5sb2FkcyhsaW5lKQogICAgICAgIGV4Y2VwdCBqc29uLkpTT05E"
        "ZWNvZGVFcnJvcjoKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiSW52YWxpZCB0YXNrIG1l"
        "dGFkYXRhIEpTT04iKSBmcm9tIE5vbmUKICAgICAgICBpZiBub3QgaXNpbnN0YW5jZShkYXRhLCBk"
        "aWN0KToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiVGFzayBtZXRhZGF0YSBtdXN0IGJl"
        "IGFuIG9iamVjdCIpCiAgICAgICAgdGFzayA9IGFnZW50X3Rhc2soZGF0YSkKICAgICAgICBpZiB0"
        "YXNrLmluc3RhbmNlX2lkIGluIHNlZW46CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIkR1"
        "cGxpY2F0ZSB0YXNrIElEIikKICAgICAgICBzZWVuLmFkZCh0YXNrLmluc3RhbmNlX2lkKQogICAg"
        "ICAgIHRhc2tzLmFwcGVuZCh0YXNrKQogICAgcmV0dXJuIHRhc2tzCgoKZGVmIGV4cG9ydF9iYXNl"
        "X3NuYXBzaG90KHJlcG9zaXRvcnk6IFBhdGgsIHRhc2s6IEFnZW50VGFzaywgZGVzdGluYXRpb246"
        "IFBhdGgpIC0+IE5vbmU6CiAgICAiIiJFeHBvcnQgZXhhY3RseSBiYXNlX2NvbW1pdCwgd2l0aG91"
        "dCByZWZzLCBoaXN0b3J5LCB1bnRyYWNrZWQgZmlsZXMgb3IgbGlua3MuCgogICAgVGhlIHNvdXJj"
        "ZSByZXBvc2l0b3J5IGFuZCB0YXNrIEpTT05MIG11c3QgcmVtYWluIG91dHNpZGUgdGhlIGFnZW50"
        "IHNhbmRib3guCiAgICBUaGlzIGhvc3QgaGVscGVyIGlzIG5vdCBhbiBpbXBsZW1lbnRhdGlvbiBv"
        "ZiBLYWdnbGUncyBzYW5kYm94IHBpcGVsaW5lLgogICAgIiIiCiAgICBpZiBkZXN0aW5hdGlvbi5l"
        "eGlzdHMoKSBvciBkZXN0aW5hdGlvbi5pc19zeW1saW5rKCk6CiAgICAgICAgcmFpc2UgVmFsdWVF"
        "cnJvcigiU25hcHNob3QgZGVzdGluYXRpb24gbXVzdCBub3QgYWxyZWFkeSBleGlzdCIpCiAgICBp"
        "ZiBub3QgcmUuZnVsbG1hdGNoKHIiWzAtOWEtZl17NDB9IiwgdGFzay5iYXNlX2NvbW1pdCk6CiAg"
        "ICAgICAgcmFpc2UgVmFsdWVFcnJvcigiVGFzayBiYXNlX2NvbW1pdCBtdXN0IGJlIGFuIGV4YWN0"
        "IEdpdCBjb21taXQiKQogICAgcmVzdWx0ID0gc3VicHJvY2Vzcy5ydW4oCiAgICAgICAgWyJnaXQi"
        "LCAiLUMiLCBzdHIocmVwb3NpdG9yeS5yZXNvbHZlKCkpLCAiYXJjaGl2ZSIsIHRhc2suYmFzZV9j"
        "b21taXRdLAogICAgICAgIGNhcHR1cmVfb3V0cHV0PVRydWUsCiAgICAgICAgY2hlY2s9RmFsc2Us"
        "CiAgICApCiAgICBpZiByZXN1bHQucmV0dXJuY29kZToKICAgICAgICByYWlzZSBWYWx1ZUVycm9y"
        "KCJDYW5ub3QgZXhwb3J0IHRoZSBzZWxlY3RlZCBiYXNlIGNvbW1pdCIpCiAgICB3aXRoIHRhcmZp"
        "bGUub3BlbihmaWxlb2JqPWlvLkJ5dGVzSU8ocmVzdWx0LnN0ZG91dCkpIGFzIGFyY2hpdmU6CiAg"
        "ICAgICAgbWVtYmVycyA9IGFyY2hpdmUuZ2V0bWVtYmVycygpCiAgICAgICAgZm9yIG1lbWJlciBp"
        "biBtZW1iZXJzOgogICAgICAgICAgICBwYXRoID0gUHVyZVBvc2l4UGF0aChtZW1iZXIubmFtZSkK"
        "ICAgICAgICAgICAgaWYgKAogICAgICAgICAgICAgICAgcGF0aC5pc19hYnNvbHV0ZSgpCiAgICAg"
        "ICAgICAgICAgICBvciAiLi4iIGluIHBhdGgucGFydHMKICAgICAgICAgICAgICAgIG9yIGFueShw"
        "YXJ0LmNhc2Vmb2xkKCkgPT0gIi5naXQiIGZvciBwYXJ0IGluIHBhdGgucGFydHMpCiAgICAgICAg"
        "ICAgICAgICBvciBub3QgKG1lbWJlci5pc2ZpbGUoKSBvciBtZW1iZXIuaXNkaXIoKSkKICAgICAg"
        "ICAgICAgKToKICAgICAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIlVuc2FmZSBzbmFwc2hv"
        "dCBtZW1iZXIiKQogICAgICAgIGRlc3RpbmF0aW9uLm1rZGlyKHBhcmVudHM9VHJ1ZSkKICAgICAg"
        "ICBmb3IgbWVtYmVyIGluIG1lbWJlcnM6CiAgICAgICAgICAgIHRhcmdldCA9IGRlc3RpbmF0aW9u"
        "LmpvaW5wYXRoKCpQdXJlUG9zaXhQYXRoKG1lbWJlci5uYW1lKS5wYXJ0cykKICAgICAgICAgICAg"
        "aWYgbWVtYmVyLmlzZGlyKCk6CiAgICAgICAgICAgICAgICB0YXJnZXQubWtkaXIocGFyZW50cz1U"
        "cnVlLCBleGlzdF9vaz1UcnVlKQogICAgICAgICAgICBlbHNlOgogICAgICAgICAgICAgICAgdGFy"
        "Z2V0LnBhcmVudC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICAgICAgICAg"
        "ICAgICBzdHJlYW0gPSBhcmNoaXZlLmV4dHJhY3RmaWxlKG1lbWJlcikKICAgICAgICAgICAgICAg"
        "IGlmIHN0cmVhbSBpcyBOb25lOgogICAgICAgICAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3Io"
        "IkludmFsaWQgc25hcHNob3QgbWVtYmVyIikKICAgICAgICAgICAgICAgIHdpdGggc3RyZWFtLCB0"
        "YXJnZXQub3BlbigieGIiKSBhcyBvdXRwdXQ6CiAgICAgICAgICAgICAgICAgICAgb3V0cHV0Lndy"
        "aXRlKHN0cmVhbS5yZWFkKCkpCiAgICAgICAgICAgICAgICB0YXJnZXQuY2htb2QoMG83NTUgaWYg"
        "bWVtYmVyLm1vZGUgJiAwbzExMSBlbHNlIDBvNjQ0KQoKCmRlZiB2YWxpZGF0ZV9zbmFwc2hvdF9o"
        "aXN0b3J5KAogICAgYXJjaGl2ZV9wYXRoOiBQYXRoLCBzZWxlY3RlZDogZGljdFtzdHIsIEFueV0s"
        "IHJlY2VpcHQ6IGRpY3Rbc3RyLCBBbnldCikgLT4gTm9uZToKICAgICIiIlZhbGlkYXRlIGF1dGhl"
        "bnRpY2F0ZWQgZmFzdC1leHBvcnQgc25hcHNob3QgaGlzdG9yeSBhZ2FpbnN0IGl0cyB0cmVlIHJl"
        "Y2VpcHQuCgogICAgT2ZmaWNpYWwgZmFzdC1pbXBvcnQgbWF5IHJld3JpdGUgY29tbWl0IElEcy4g"
        "T3JpZ2luYWwgYmFzZSBpZGVudGl0eSBpcyBib3VuZAogICAgdGhyb3VnaCB0aGUgdmVyaWZpZWQg"
        "b3JpZ2luYWwgdHJlZSBhbmQgc291cmNlIGhhc2gsIG5ldmVyIHRpbWVzdGFtcHMgb3IgYQogICAg"
        "cmVxdWlyZW1lbnQgdGhhdCB0aGUgb3JpZ2luYWwgY29tbWl0IG9iamVjdCBzdXJ2aXZlIGV4cG9y"
        "dC4KICAgICIiIgogICAgaW1wb3J0IG9zCiAgICBpbXBvcnQgdGVtcGZpbGUKCiAgICBmcm9tIG1v"
        "c3RhemEuZXZhbHVhdGlvbi5jb250cmFjdHMgaW1wb3J0IHJlcXVpcmUKICAgIGZyb20gbW9zdGF6"
        "YS5yZXNvdXJjZXMgaW1wb3J0IHNoYTI1NiBhcyBmaWxlX2RpZ2VzdAoKICAgIHZhbGlkYXRpb24g"
        "PSByZWNlaXB0LmdldCgidmFsaWRhdGlvbiIsIHt9KQogICAgcmVxdWlyZSgKICAgICAgICByZWNl"
        "aXB0LmdldCgia2luZCIpID09ICJzbmFwc2hvdCIKICAgICAgICBhbmQgcmVjZWlwdC5nZXQoImlu"
        "c3RhbmNlX2lkIikgPT0gc2VsZWN0ZWRbImluc3RhbmNlX2lkIl0KICAgICAgICBhbmQgZmlsZV9k"
        "aWdlc3QoYXJjaGl2ZV9wYXRoKSA9PSByZWNlaXB0LmdldCgic2hhMjU2IiksCiAgICAgICAgIlNu"
        "YXBzaG90IHJlY2VpcHQvaGFzaCBtaXNtYXRjaCIsCiAgICApCiAgICByZXF1aXJlKAogICAgICAg"
        "IHZhbGlkYXRpb24uZ2V0KCJvcmlnaW5hbF9iYXNlX2NvbW1pdCIpID09IHNlbGVjdGVkWyJiYXNl"
        "X2NvbW1pdCJdCiAgICAgICAgYW5kIHZhbGlkYXRpb24uZ2V0KCJiYXNlX3RyZWVfaWRlbnRpdHlf"
        "bWF0Y2hlcyIpIGlzIFRydWUKICAgICAgICBhbmQgdmFsaWRhdGlvbi5nZXQoIm9yaWdpbmFsX2Jh"
        "c2VfdHJlZV9zaGEiKQogICAgICAgID09IHZhbGlkYXRpb24uZ2V0KCJzbmFwc2hvdF9oZWFkX3Ry"
        "ZWVfc2hhIikKICAgICAgICBhbmQgdmFsaWRhdGlvbi5nZXQoIm9yaWdpbmFsX3RyZWVfc291cmNl"
        "IikKICAgICAgICA9PSBmImh0dHBzOi8vYXBpLmdpdGh1Yi5jb20vcmVwb3Mve3NlbGVjdGVkWydy"
        "ZXBvJ119L2dpdC9jb21taXRzL3tzZWxlY3RlZFsnYmFzZV9jb21taXQnXX0iLAogICAgICAgICJP"
        "cmlnaW5hbCBiYXNlL3RyZWUgaWRlbnRpdHkgcmVjZWlwdCBtaXNzaW5nIiwKICAgICkKICAgIHJl"
        "cXVpcmUoCiAgICAgICAgdmFsaWRhdGlvbi5nZXQoInJlYWNoYWJsZV9mb3J3YXJkX2NvbW1pdHMi"
        "KSA9PSAwCiAgICAgICAgYW5kIHZhbGlkYXRpb24uZ2V0KCJmb3J3YXJkX29yX3VucmVsYXRlZF9j"
        "b21taXRfb2JqZWN0cyIpID09IDAsCiAgICAgICAgIlNuYXBzaG90IHJlY2VpcHQgcmVjb3JkcyB1"
        "bnNhZmUgaGlzdG9yeSIsCiAgICApCiAgICBlbnZpcm9ubWVudCA9IHsKICAgICAgICBrZXk6IHZh"
        "bHVlIGZvciBrZXksIHZhbHVlIGluIG9zLmVudmlyb24uaXRlbXMoKSBpZiBub3Qga2V5LnN0YXJ0"
        "c3dpdGgoIkdJVF8iKQogICAgfQogICAgZW52aXJvbm1lbnQudXBkYXRlKAogICAgICAgIEdJVF9D"
        "T05GSUdfTk9TWVNURU09IjEiLAogICAgICAgIEdJVF9DT05GSUdfR0xPQkFMPW9zLmRldm51bGws"
        "CiAgICAgICAgR0lUX05PX1JFUExBQ0VfT0JKRUNUUz0iMSIsCiAgICApCiAgICB3aXRoIHRlbXBm"
        "aWxlLlRlbXBvcmFyeURpcmVjdG9yeShwcmVmaXg9Im1vc3RhemEtc25hcHNob3QtaGlzdG9yeS0i"
        "KSBhcyB0ZW1wb3Jhcnk6CiAgICAgICAgZ2l0X2RpciA9IFBhdGgodGVtcG9yYXJ5KSAvICIuZ2l0"
        "IgogICAgICAgIHdpdGggdGFyZmlsZS5vcGVuKGFyY2hpdmVfcGF0aCwgInI6Z3oiKSBhcyBhcmNo"
        "aXZlOgogICAgICAgICAgICBmb3IgbWVtYmVyIGluIGFyY2hpdmU6CiAgICAgICAgICAgICAgICBw"
        "YXRoID0gUHVyZVBvc2l4UGF0aChtZW1iZXIubmFtZSkKICAgICAgICAgICAgICAgIHJlcXVpcmUo"
        "CiAgICAgICAgICAgICAgICAgICAgbm90IHBhdGguaXNfYWJzb2x1dGUoKSBhbmQgIi4uIiBub3Qg"
        "aW4gcGF0aC5wYXJ0cywKICAgICAgICAgICAgICAgICAgICAiVW5zYWZlIHNuYXBzaG90IGFyY2hp"
        "dmUgbWVtYmVyIiwKICAgICAgICAgICAgICAgICkKICAgICAgICAgICAgICAgIGlmIHBhdGgucGFy"
        "dHMgYW5kIHBhdGgucGFydHNbMF0gPT0gIi5naXQiOgogICAgICAgICAgICAgICAgICAgIHJlcXVp"
        "cmUoCiAgICAgICAgICAgICAgICAgICAgICAgIG1lbWJlci5pc2RpcigpIG9yIG1lbWJlci5pc2Zp"
        "bGUoKSwKICAgICAgICAgICAgICAgICAgICAgICAgIkdpdCBzbmFwc2hvdCBsaW5rcyBhcmUgZm9y"
        "YmlkZGVuIiwKICAgICAgICAgICAgICAgICAgICApCiAgICAgICAgICAgICAgICAgICAgaWYgbWVt"
        "YmVyLmlzZmlsZSgpOgogICAgICAgICAgICAgICAgICAgICAgICB0YXJnZXQgPSBQYXRoKHRlbXBv"
        "cmFyeSkuam9pbnBhdGgoKnBhdGgucGFydHMpCiAgICAgICAgICAgICAgICAgICAgICAgIHRhcmdl"
        "dC5wYXJlbnQubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgICAgICAgICAg"
        "ICAgICAgICAgICBzdHJlYW0gPSBhcmNoaXZlLmV4dHJhY3RmaWxlKG1lbWJlcikKICAgICAgICAg"
        "ICAgICAgICAgICAgICAgcmVxdWlyZShzdHJlYW0gaXMgbm90IE5vbmUsICJVbnJlYWRhYmxlIHNu"
        "YXBzaG90IEdpdCBvYmplY3QiKQogICAgICAgICAgICAgICAgICAgICAgICBhc3NlcnQgc3RyZWFt"
        "IGlzIG5vdCBOb25lCiAgICAgICAgICAgICAgICAgICAgICAgIHdpdGggc3RyZWFtLCB0YXJnZXQu"
        "b3BlbigieGIiKSBhcyBvdXRwdXQ6CiAgICAgICAgICAgICAgICAgICAgICAgICAgICBvdXRwdXQu"
        "d3JpdGUoc3RyZWFtLnJlYWQoKSkKICAgICAgICByZXF1aXJlKGdpdF9kaXIuaXNfZGlyKCksICJT"
        "bmFwc2hvdCBHaXQgbWV0YWRhdGEgbWlzc2luZyIpCiAgICAgICAgZm9yIHJlbGF0aXZlIGluICgi"
        "b2JqZWN0cy9pbmZvL2FsdGVybmF0ZXMiLCAic2hhbGxvdyIsICJpbmZvL2dyYWZ0cyIpOgogICAg"
        "ICAgICAgICByZXF1aXJlKAogICAgICAgICAgICAgICAgbm90IChnaXRfZGlyIC8gcmVsYXRpdmUp"
        "LmV4aXN0cygpLAogICAgICAgICAgICAgICAgIkV4dGVybmFsIG9yIGluY29tcGxldGUgR2l0IGhp"
        "c3RvcnkgZm9yYmlkZGVuIiwKICAgICAgICAgICAgKQoKICAgICAgICBkZWYgZ2l0KCphcmd1bWVu"
        "dHM6IHN0cikgLT4gc3RyOgogICAgICAgICAgICByZXN1bHQgPSBzdWJwcm9jZXNzLnJ1bigKICAg"
        "ICAgICAgICAgICAgIFsKICAgICAgICAgICAgICAgICAgICAiZ2l0IiwKICAgICAgICAgICAgICAg"
        "ICAgICAiLS1naXQtZGlyIiwKICAgICAgICAgICAgICAgICAgICBzdHIoZ2l0X2RpciksCiAgICAg"
        "ICAgICAgICAgICAgICAgIi1jIiwKICAgICAgICAgICAgICAgICAgICAiY29yZS5ob29rc1BhdGg9"
        "IiArIG9zLmRldm51bGwsCiAgICAgICAgICAgICAgICAgICAgIi1jIiwKICAgICAgICAgICAgICAg"
        "ICAgICAiY29yZS5mc21vbml0b3I9ZmFsc2UiLAogICAgICAgICAgICAgICAgICAgICphcmd1bWVu"
        "dHMsCiAgICAgICAgICAgICAgICBdLAogICAgICAgICAgICAgICAgZW52PWVudmlyb25tZW50LAog"
        "ICAgICAgICAgICAgICAgY2FwdHVyZV9vdXRwdXQ9VHJ1ZSwKICAgICAgICAgICAgICAgIHRleHQ9"
        "VHJ1ZSwKICAgICAgICAgICAgICAgIGNoZWNrPUZhbHNlLAogICAgICAgICAgICApCiAgICAgICAg"
        "ICAgIHJlcXVpcmUocmVzdWx0LnJldHVybmNvZGUgPT0gMCwgIkNhbm5vdCB2ZXJpZnkgc25hcHNo"
        "b3QgR2l0IGhpc3RvcnkiKQogICAgICAgICAgICByZXR1cm4gcmVzdWx0LnN0ZG91dC5zdHJpcCgp"
        "CgogICAgICAgIGhlYWQgPSBnaXQoInJldi1wYXJzZSIsICJIRUFEIikKICAgICAgICB0cmVlID0g"
        "Z2l0KCJyZXYtcGFyc2UiLCAiSEVBRF57dHJlZX0iKQogICAgICAgIHJlcXVpcmUoCiAgICAgICAg"
        "ICAgIGhlYWQgPT0gdmFsaWRhdGlvbi5nZXQoImhlYWQiKQogICAgICAgICAgICBhbmQgdHJlZSA9"
        "PSB2YWxpZGF0aW9uLmdldCgib3JpZ2luYWxfYmFzZV90cmVlX3NoYSIpLAogICAgICAgICAgICAi"
        "RXhwb3J0ZWQgSEVBRC90cmVlIGRpZmZlcnMgZnJvbSB2ZXJpZmllZCBvcmlnaW5hbC1iYXNlIHJl"
        "Y2VpcHQiLAogICAgICAgICkKICAgICAgICBhbmNlc3RvcnMgPSBzZXQoZ2l0KCJyZXYtbGlzdCIs"
        "ICJIRUFEIikuc3BsaXRsaW5lcygpKQogICAgICAgIHJlcXVpcmUoCiAgICAgICAgICAgIG5vdCBn"
        "aXQoInJldi1saXN0IiwgIi0tYWxsIiwgIi0tbm90IiwgIkhFQUQiKSwKICAgICAgICAgICAgIkZv"
        "cndhcmQgb3IgdW5yZWxhdGVkIEdpdCByZWZzIGRldGVjdGVkIiwKICAgICAgICApCiAgICAgICAg"
        "Zm9yIHJlZiBpbiBnaXQoImZvci1lYWNoLXJlZiIsICItLWZvcm1hdD0lKHJlZm5hbWUpIikuc3Bs"
        "aXRsaW5lcygpOgogICAgICAgICAgICByZXF1aXJlKAogICAgICAgICAgICAgICAgZ2l0KCJyZXYt"
        "cGFyc2UiLCByZWYgKyAiXntjb21taXR9IikgaW4gYW5jZXN0b3JzLAogICAgICAgICAgICAgICAg"
        "IkdpdCByZWZlcmVuY2UgaXMgbm90IGFuIGV4cG9ydGVkIEhFQUQgYW5jZXN0b3IiLAogICAgICAg"
        "ICAgICApCiAgICAgICAgb2JqZWN0cyA9IGdpdCgKICAgICAgICAgICAgImNhdC1maWxlIiwKICAg"
        "ICAgICAgICAgIi0tYmF0Y2gtYWxsLW9iamVjdHMiLAogICAgICAgICAgICAiLS1iYXRjaC1jaGVj"
        "az0lKG9iamVjdHR5cGUpICUob2JqZWN0bmFtZSkiLAogICAgICAgICkKICAgICAgICBjb21taXRz"
        "ID0gewogICAgICAgICAgICBsaW5lLnNwbGl0KClbMV0KICAgICAgICAgICAgZm9yIGxpbmUgaW4g"
        "b2JqZWN0cy5zcGxpdGxpbmVzKCkKICAgICAgICAgICAgaWYgbGluZS5zdGFydHN3aXRoKCJjb21t"
        "aXQgIikKICAgICAgICB9CiAgICAgICAgcmVxdWlyZSgKICAgICAgICAgICAgY29tbWl0cyA8PSBh"
        "bmNlc3RvcnMgYW5kIGxlbihjb21taXRzKSA9PSB2YWxpZGF0aW9uLmdldCgiY29tbWl0X29iamVj"
        "dHMiKSwKICAgICAgICAgICAgIkZvcndhcmQvdW5yZWxhdGVkIG9yIGNoYW5nZWQgR2l0IGNvbW1p"
        "dC1vYmplY3QgaW52ZW50b3J5IiwKICAgICAgICApCg=="
    ),
    "src/mostaza/packaging.py": (
        "..."
    ),
    "src/mostaza/research/__init__.py": (
        "IiIiRGVzY3JpcHRpdmUgcGFpcmVkIGFuYWx5c2lzIGZvciBzbWFsbCBleHBsb3JhdG9yeSBleHBl"
        "cmltZW50cy4iIiIK"
    ),
    "src/mostaza/research/metrics.py": (
        "IiIiTWF0Y2hlZCBFMS1taW51cy1FMCBjYWxjdWxhdGlvbnM7IHVuYXZhaWxhYmxlIGFuZCBlc3Rp"
        "bWF0ZWQgdmFsdWVzIHN0YXkgZGlzdGluY3QuIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFu"
        "bm90YXRpb25zCgpmcm9tIGNvbGxlY3Rpb25zIGltcG9ydCBDb3VudGVyCmZyb20gc3RhdGlzdGlj"
        "cyBpbXBvcnQgbWVhbgpmcm9tIHR5cGluZyBpbXBvcnQgQW55Cgpmcm9tIG1vc3RhemEuZXZhbHVh"
        "dGlvbi5jb250cmFjdHMgaW1wb3J0ICgKICAgIE1FVFJJQ19VTklUUywKICAgIGNhbm9uaWNhbF9q"
        "c29uLAogICAgcmVxdWlyZSwKICAgIHNoYTI1NiwKICAgIHZhbGlkYXRlX3J1biwKKQoKTUFUQ0hF"
        "RF9GSUVMRFMgPSAoCiAgICAiZXhwZXJpbWVudF9pZCIsCiAgICAicnVuX2tpbmQiLAogICAgImNv"
        "ZGVfY29tbWl0IiwKICAgICJ3b3JraW5nX3RyZWVfc2hhMjU2IiwKICAgICJtb2RlbCIsCiAgICAi"
        "cmVzb3VyY2VzIiwKICAgICJjb2hvcnQiLAogICAgImluZmVyZW5jZSIsCiAgICAiYnVkZ2V0cyIs"
        "CiAgICAiZW52aXJvbm1lbnQiLAopCgoKZGVmIGNvbXBhcmUoCiAgICBlMDogdHVwbGVbZGljdFtz"
        "dHIsIEFueV0sIGxpc3RbZGljdFtzdHIsIEFueV1dXSwKICAgIGUxOiB0dXBsZVtkaWN0W3N0ciwg"
        "QW55XSwgbGlzdFtkaWN0W3N0ciwgQW55XV1dLAopIC0+IGRpY3Rbc3RyLCBBbnldOgogICAgbTAs"
        "IHIwID0gZTAKICAgIG0xLCByMSA9IGUxCiAgICB2YWxpZGF0ZV9ydW4obTAsIHIwKQogICAgdmFs"
        "aWRhdGVfcnVuKG0xLCByMSkKICAgIHJlcXVpcmUobTBbImFybSJdID09ICJFMCIgYW5kIG0xWyJh"
        "cm0iXSA9PSAiRTEiLCAiRXhwZWN0ZWQgb3JkZXJlZCBFMC9FMSBhcm1zIikKICAgIHJlcXVpcmUo"
        "bTBbInJ1bl9pZCJdICE9IG0xWyJydW5faWQiXSwgIkFybXMgcmVxdWlyZSBkaXN0aW5jdCBydW4g"
        "SURzIikKICAgIGZvciBmaWVsZCBpbiBNQVRDSEVEX0ZJRUxEUzoKICAgICAgICByZXF1aXJlKG0w"
        "W2ZpZWxkXSA9PSBtMVtmaWVsZF0sIGYiVW5tYXRjaGVkIG5vbi10cmVhdG1lbnQgY29uZGl0aW9u"
        "OiB7ZmllbGR9IikKICAgIGJhc2VsaW5lID0ge3JbImluc3RhbmNlX2lkIl06IHIgZm9yIHIgaW4g"
        "cjB9CiAgICBoeWJyaWQgPSB7clsiaW5zdGFuY2VfaWQiXTogciBmb3IgciBpbiByMX0KICAgIHBh"
        "aXJlZDogbGlzdFtkaWN0W3N0ciwgQW55XV0gPSBbXQogICAgc3VtbWFyeTogZGljdFtzdHIsIEFu"
        "eV0gPSB7fQogICAgZm9yIG5hbWUsIHVuaXQgaW4gTUVUUklDX1VOSVRTLml0ZW1zKCk6CiAgICAg"
        "ICAgY291bnRzOiBkaWN0W3N0ciwgbGlzdFtmbG9hdF1dID0geyJtZWFzdXJlZCI6IFtdLCAiZXN0"
        "aW1hdGVkIjogW119CiAgICAgICAgZXhjbHVkZWQ6IGxpc3RbZGljdFtzdHIsIHN0cl1dID0gW10K"
        "ICAgICAgICBmb3IgaW5zdGFuY2VfaWQgaW4gc29ydGVkKGJhc2VsaW5lKToKICAgICAgICAgICAg"
        "YSA9IGJhc2VsaW5lW2luc3RhbmNlX2lkXVsibWV0cmljcyJdW25hbWVdCiAgICAgICAgICAgIGIg"
        "PSBoeWJyaWRbaW5zdGFuY2VfaWRdWyJtZXRyaWNzIl1bbmFtZV0KICAgICAgICAgICAgaWYgKAog"
        "ICAgICAgICAgICAgICAgYVsiYXZhaWxhYmlsaXR5Il0gPT0gYlsiYXZhaWxhYmlsaXR5Il0KICAg"
        "ICAgICAgICAgICAgIGFuZCBhWyJhdmFpbGFiaWxpdHkiXSAhPSAidW5hdmFpbGFibGUiCiAgICAg"
        "ICAgICAgICk6CiAgICAgICAgICAgICAgICAjIFJlbGV2YW5jZSByZXF1aXJlcyB0aGUgc2FtZSBt"
        "YXBwaW5nIGFuZCBvcGVyYXRpb25hbCBkZWZpbml0aW9uLgogICAgICAgICAgICAgICAgaWYgbmFt"
        "ZSBpbiAoInJlY2FsbF9hdF9rIiwgInJlbGV2YW50X2NvbnRleHRfcmF0aW8iKToKICAgICAgICAg"
        "ICAgICAgICAgICByZXF1aXJlKAogICAgICAgICAgICAgICAgICAgICAgICBiYXNlbGluZVtpbnN0"
        "YW5jZV9pZF1bImdyb3VuZF90cnV0aCJdCiAgICAgICAgICAgICAgICAgICAgICAgID09IGh5YnJp"
        "ZFtpbnN0YW5jZV9pZF1bImdyb3VuZF90cnV0aCJdLAogICAgICAgICAgICAgICAgICAgICAgICAi"
        "VW5tYXRjaGVkIHBvc3QtcnVuIGdyb3VuZCB0cnV0aCIsCiAgICAgICAgICAgICAgICAgICAgKQog"
        "ICAgICAgICAgICAgICAgZGVsdGEgPSBiWyJ2YWx1ZSJdIC0gYVsidmFsdWUiXQogICAgICAgICAg"
        "ICAgICAgY291bnRzW2FbImF2YWlsYWJpbGl0eSJdXS5hcHBlbmQoZGVsdGEpCiAgICAgICAgICAg"
        "ICAgICBwYWlyZWQuYXBwZW5kKAogICAgICAgICAgICAgICAgICAgIHsKICAgICAgICAgICAgICAg"
        "ICAgICAgICAgImluc3RhbmNlX2lkIjogaW5zdGFuY2VfaWQsCiAgICAgICAgICAgICAgICAgICAg"
        "ICAgICJtZXRyaWMiOiBuYW1lLAogICAgICAgICAgICAgICAgICAgICAgICAidW5pdCI6IHVuaXQs"
        "CiAgICAgICAgICAgICAgICAgICAgICAgICJhdmFpbGFiaWxpdHkiOiBhWyJhdmFpbGFiaWxpdHki"
        "XSwKICAgICAgICAgICAgICAgICAgICAgICAgIkUwIjogYVsidmFsdWUiXSwKICAgICAgICAgICAg"
        "ICAgICAgICAgICAgIkUxIjogYlsidmFsdWUiXSwKICAgICAgICAgICAgICAgICAgICAgICAgImRl"
        "bHRhIjogZGVsdGEsCiAgICAgICAgICAgICAgICAgICAgICAgICJzb3VyY2VzIjogeyJFMCI6IGFb"
        "InNvdXJjZSJdLCAiRTEiOiBiWyJzb3VyY2UiXX0sCiAgICAgICAgICAgICAgICAgICAgfQogICAg"
        "ICAgICAgICAgICAgKQogICAgICAgICAgICBlbHNlOgogICAgICAgICAgICAgICAgZXhjbHVkZWQu"
        "YXBwZW5kKAogICAgICAgICAgICAgICAgICAgIHsKICAgICAgICAgICAgICAgICAgICAgICAgImlu"
        "c3RhbmNlX2lkIjogaW5zdGFuY2VfaWQsCiAgICAgICAgICAgICAgICAgICAgICAgICJyZWFzb24i"
        "OiAiTWlzc2luZyBvciBub25jb21wYXJhYmxlIG1lYXN1cmVtZW50IiwKICAgICAgICAgICAgICAg"
        "ICAgICAgICAgIkUwIjogYVsicmVhc29uIl0gb3IgYVsiYXZhaWxhYmlsaXR5Il0sCiAgICAgICAg"
        "ICAgICAgICAgICAgICAgICJFMSI6IGJbInJlYXNvbiJdIG9yIGJbImF2YWlsYWJpbGl0eSJdLAog"
        "ICAgICAgICAgICAgICAgICAgIH0KICAgICAgICAgICAgICAgICkKICAgICAgICBzdW1tYXJ5W25h"
        "bWVdID0gewogICAgICAgICAgICAidW5pdCI6IHVuaXQsCiAgICAgICAgICAgICJjb2hvcnRfZGVu"
        "b21pbmF0b3IiOiBsZW4oYmFzZWxpbmUpLAogICAgICAgICAgICAibWVhc3VyZWRfcGFpcnMiOiBs"
        "ZW4oY291bnRzWyJtZWFzdXJlZCJdKSwKICAgICAgICAgICAgImVzdGltYXRlZF9wYWlycyI6IGxl"
        "bihjb3VudHNbImVzdGltYXRlZCJdKSwKICAgICAgICAgICAgIm1lYXN1cmVkX21lYW5fZGVsdGEi"
        "OiBtZWFuKGNvdW50c1sibWVhc3VyZWQiXSkKICAgICAgICAgICAgaWYgY291bnRzWyJtZWFzdXJl"
        "ZCJdCiAgICAgICAgICAgIGVsc2UgTm9uZSwKICAgICAgICAgICAgImVzdGltYXRlZF9tZWFuX2Rl"
        "bHRhIjogbWVhbihjb3VudHNbImVzdGltYXRlZCJdKQogICAgICAgICAgICBpZiBjb3VudHNbImVz"
        "dGltYXRlZCJdCiAgICAgICAgICAgIGVsc2UgTm9uZSwKICAgICAgICAgICAgImV4Y2x1ZGVkIjog"
        "ZXhjbHVkZWQsCiAgICAgICAgfQogICAgb3V0Y29tZXM6IGRpY3Rbc3RyLCBBbnldID0ge30KICAg"
        "IGZvciBhcm0sIHJvd3MgaW4gKCgiRTAiLCByMCksICgiRTEiLCByMSkpOgogICAgICAgIGNvdW50"
        "ID0gQ291bnRlcihyWyJzdGF0dXMiXSBmb3IgciBpbiByb3dzKQogICAgICAgIG91dGNvbWVzW2Fy"
        "bV0gPSB7czogY291bnRbc10gZm9yIHMgaW4gKCJQQVNTIiwgIkZBSUwiLCAiRVJST1IiLCAiTk9U"
        "X1JVTiIpfQogICAgICAgIG91dGNvbWVzW2FybV1bImNvaG9ydF9kZW5vbWluYXRvciJdID0gbGVu"
        "KHJvd3MpCiAgICAgICAgb3V0Y29tZXNbYXJtXVsicmVzb2x2ZWQiXSA9IHN1bShyWyJyZXNvbHZl"
        "ZCJdIGlzIFRydWUgZm9yIHIgaW4gcm93cykKICAgICAgICBvdXRjb21lc1thcm1dWyJyZXNvbHV0"
        "aW9uX3Vua25vd24iXSA9IHN1bShyWyJyZXNvbHZlZCJdIGlzIE5vbmUgZm9yIHIgaW4gcm93cykK"
        "ICAgICAgICBvdXRjb21lc1thcm1dWyJwYXRjaF9wcmVzZW50Il0gPSBzdW0oclsicGF0Y2giXVsi"
        "cHJlc2VudCJdIGZvciByIGluIHJvd3MpCiAgICAgICAgb3V0Y29tZXNbYXJtXVsicGF0Y2hfdmFs"
        "aWQiXSA9IHN1bShyWyJwYXRjaCJdWyJ2YWxpZCJdIGlzIFRydWUgZm9yIHIgaW4gcm93cykKICAg"
        "ICAgICBvdXRjb21lc1thcm1dWyJwYXRjaF92YWxpZGl0eV91bmtub3duIl0gPSBzdW0oCiAgICAg"
        "ICAgICAgIHJbInBhdGNoIl1bInZhbGlkIl0gaXMgTm9uZSBmb3IgciBpbiByb3dzCiAgICAgICAg"
        "KQogICAgICAgIG91dGNvbWVzW2FybV1bInRlc3RzX3Bhc3NlZCJdID0gc3VtKAogICAgICAgICAg"
        "ICByWyJ0ZXN0cyJdWyJvdXRjb21lIl0gPT0gIlBBU1MiIGZvciByIGluIHJvd3MKICAgICAgICAp"
        "CiAgICAgICAgb3V0Y29tZXNbYXJtXVsidGVzdHNfbm90X3J1biJdID0gc3VtKAogICAgICAgICAg"
        "ICByWyJ0ZXN0cyJdWyJvdXRjb21lIl0gPT0gIk5PVF9SVU4iIGZvciByIGluIHJvd3MKICAgICAg"
        "ICApCiAgICB0cmFuc2l0aW9ucyA9IFsKICAgICAgICB7CiAgICAgICAgICAgICJpbnN0YW5jZV9p"
        "ZCI6IGtleSwKICAgICAgICAgICAgIkUwIjogYmFzZWxpbmVba2V5XVsic3RhdHVzIl0sCiAgICAg"
        "ICAgICAgICJFMSI6IGh5YnJpZFtrZXldWyJzdGF0dXMiXSwKICAgICAgICAgICAgInJlc29sdmVk"
        "X2RlbHRhIjogaW50KGh5YnJpZFtrZXldWyJyZXNvbHZlZCJdKQogICAgICAgICAgICAtIGludChi"
        "YXNlbGluZVtrZXldWyJyZXNvbHZlZCJdKQogICAgICAgICAgICBpZiBiYXNlbGluZVtrZXldWyJy"
        "ZXNvbHZlZCJdIGlzIG5vdCBOb25lCiAgICAgICAgICAgIGFuZCBoeWJyaWRba2V5XVsicmVzb2x2"
        "ZWQiXSBpcyBub3QgTm9uZQogICAgICAgICAgICBlbHNlIE5vbmUsCiAgICAgICAgfQogICAgICAg"
        "IGZvciBrZXkgaW4gc29ydGVkKGJhc2VsaW5lKQogICAgXQogICAgcmV0dXJuIHsKICAgICAgICAi"
        "c2NoZW1hX3ZlcnNpb24iOiAxLAogICAgICAgICJleHBlcmltZW50X2lkIjogbTBbImV4cGVyaW1l"
        "bnRfaWQiXSwKICAgICAgICAicnVuX2tpbmQiOiBtMFsicnVuX2tpbmQiXSwKICAgICAgICAiY29o"
        "b3J0X2Rlbm9taW5hdG9yIjogbGVuKGJhc2VsaW5lKSwKICAgICAgICAiZGlyZWN0aW9uIjogIkUx"
        "IG1pbnVzIEUwIiwKICAgICAgICAibWV0cmljcyI6IHN1bW1hcnksCiAgICAgICAgInBhaXJzIjog"
        "cGFpcmVkLAogICAgICAgICJvdXRjb21lcyI6IG91dGNvbWVzLAogICAgICAgICJvdXRjb21lX3Bh"
        "aXJzIjogdHJhbnNpdGlvbnMsCiAgICAgICAgInByb3ZlbmFuY2UiOiB7CiAgICAgICAgICAgICJF"
        "MCI6IHsKICAgICAgICAgICAgICAgICJydW5faWQiOiBtMFsicnVuX2lkIl0sCiAgICAgICAgICAg"
        "ICAgICAic291cmNlX3J1bl9pZCI6IG0wWyJzb3VyY2VfcnVuX2lkIl0sCiAgICAgICAgICAgICAg"
        "ICAibWFuaWZlc3Rfc2hhMjU2Ijogc2hhMjU2KGNhbm9uaWNhbF9qc29uKG0wKSksCiAgICAgICAg"
        "ICAgICAgICAicmVjb3Jkc19zaGEyNTYiOiBzaGEyNTYoCiAgICAgICAgICAgICAgICAgICAgY2Fu"
        "b25pY2FsX2pzb24oc29ydGVkKHIwLCBrZXk9bGFtYmRhIHI6IHJbImluc3RhbmNlX2lkIl0pKQog"
        "ICAgICAgICAgICAgICAgKSwKICAgICAgICAgICAgfSwKICAgICAgICAgICAgIkUxIjogewogICAg"
        "ICAgICAgICAgICAgInJ1bl9pZCI6IG0xWyJydW5faWQiXSwKICAgICAgICAgICAgICAgICJzb3Vy"
        "Y2VfcnVuX2lkIjogbTFbInNvdXJjZV9ydW5faWQiXSwKICAgICAgICAgICAgICAgICJtYW5pZmVz"
        "dF9zaGEyNTYiOiBzaGEyNTYoY2Fub25pY2FsX2pzb24obTEpKSwKICAgICAgICAgICAgICAgICJy"
        "ZWNvcmRzX3NoYTI1NiI6IHNoYTI1NigKICAgICAgICAgICAgICAgICAgICBjYW5vbmljYWxfanNv"
        "bihzb3J0ZWQocjEsIGtleT1sYW1iZGEgcjogclsiaW5zdGFuY2VfaWQiXSkpCiAgICAgICAgICAg"
        "ICAgICApLAogICAgICAgICAgICB9LAogICAgICAgIH0sCiAgICAgICAgImxpbWl0YXRpb25zIjog"
        "c29ydGVkKAogICAgICAgICAgICBzZXQoCiAgICAgICAgICAgICAgICBtMFsibGltaXRhdGlvbnMi"
        "XQogICAgICAgICAgICAgICAgKyBtMVsibGltaXRhdGlvbnMiXQogICAgICAgICAgICAgICAgKyBb"
        "CiAgICAgICAgICAgICAgICAgICAgIlNtYWxsIGV4cGxvcmF0b3J5IGNvaG9ydDsgZGVzY3JpcHRp"
        "dmUgZGlmZmVyZW5jZXMgIgogICAgICAgICAgICAgICAgICAgICJkbyBub3QgZXN0YWJsaXNoIHNp"
        "Z25pZmljYW5jZSBvciBzdXBlcmlvcml0eS4iLAogICAgICAgICAgICAgICAgICAgICJNaXNzaW5n"
        "IG1lYXN1cmVtZW50cyBhcmUgZXhjbHVkZWQgcGVyIG1ldHJpYywgIgogICAgICAgICAgICAgICAg"
        "ICAgICJuZXZlciBpbXB1dGVkIGFzIHplcm8uIiwKICAgICAgICAgICAgICAgIF0KICAgICAgICAg"
        "ICAgKQogICAgICAgICksCiAgICB9Cg=="
    ),
    "src/mostaza/research/report.py": (
        "IiIiUmVjb21wdXRlIGRldGVybWluaXN0aWMgSlNPTi9NYXJrZG93biByZXBvcnRzIGZyb20gaW1t"
        "dXRhYmxlIHJhdyBydW4gcmVjb3Jkcy4iIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3Rh"
        "dGlvbnMKCmltcG9ydCBhcmdwYXJzZQpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKZnJvbSB0eXBp"
        "bmcgaW1wb3J0IEFueQoKZnJvbSBtb3N0YXphLmV2YWx1YXRpb24uY29udHJhY3RzIGltcG9ydCBj"
        "YW5vbmljYWxfanNvbgpmcm9tIG1vc3RhemEuZXZhbHVhdGlvbi5pbXBvcnRfcmVzdWx0cyBpbXBv"
        "cnQgbG9hZF9ydW4KZnJvbSBtb3N0YXphLnJlc2VhcmNoLm1ldHJpY3MgaW1wb3J0IGNvbXBhcmUK"
        "CgpkZWYgbWFya2Rvd24ocmVwb3J0OiBkaWN0W3N0ciwgQW55XSkgLT4gc3RyOgogICAgZGVmIGNl"
        "bGwodmFsdWU6IEFueSkgLT4gc3RyOgogICAgICAgIHJldHVybiBzdHIodmFsdWUpLnJlcGxhY2Uo"
        "InwiLCAiXFx8IikucmVwbGFjZSgiXG4iLCAiICIpCgogICAgbGluZXMgPSBbCiAgICAgICAgZiIj"
        "IFBhaXJlZCBleHBlcmltZW50OiB7Y2VsbChyZXBvcnRbJ2V4cGVyaW1lbnRfaWQnXSl9IiwKICAg"
        "ICAgICAiIiwKICAgICAgICBmIlJ1biBraW5kOiAqKntjZWxsKHJlcG9ydFsncnVuX2tpbmQnXSl9"
        "KiouICIKICAgICAgICBmIkNvaG9ydDoge3JlcG9ydFsnY29ob3J0X2Rlbm9taW5hdG9yJ119LiIs"
        "CiAgICAgICAgIkRpZmZlcmVuY2VzIGFyZSBFMSBtaW51cyBFMDsgIgogICAgICAgICJ0aGUgZGly"
        "ZWN0aW9uIG9mIGltcHJvdmVtZW50IGRlcGVuZHMgb24gdGhlIG1ldHJpYy4iLAogICAgICAgICIi"
        "LAogICAgICAgICJ8IE1ldHJpYyB8IFVuaXQgfCBNZWFzdXJlZCBwYWlycyB8IE1lYXN1cmVkIG1l"
        "YW4gZGVsdGEgfCAiCiAgICAgICAgIkVzdGltYXRlZCBwYWlycyB8IEVzdGltYXRlZCBtZWFuIGRl"
        "bHRhIHwiLAogICAgICAgICJ8IC0tLSB8IC0tLSB8IC0tLTogfCAtLS06IHwgLS0tOiB8IC0tLTog"
        "fCIsCiAgICBdCiAgICBmb3IgbmFtZSwgbWV0cmljIGluIHJlcG9ydFsibWV0cmljcyJdLml0ZW1z"
        "KCk6CiAgICAgICAgYSA9IG1ldHJpY1sibWVhc3VyZWRfbWVhbl9kZWx0YSJdCiAgICAgICAgYiA9"
        "IG1ldHJpY1siZXN0aW1hdGVkX21lYW5fZGVsdGEiXQogICAgICAgIGxpbmVzLmFwcGVuZCgKICAg"
        "ICAgICAgICAgZiJ8IHtuYW1lfSB8IHttZXRyaWNbJ3VuaXQnXX0gfCB7bWV0cmljWydtZWFzdXJl"
        "ZF9wYWlycyddfSB8ICIKICAgICAgICAgICAgZiJ7J3VuYXZhaWxhYmxlJyBpZiBhIGlzIE5vbmUg"
        "ZWxzZSBhfSB8IHttZXRyaWNbJ2VzdGltYXRlZF9wYWlycyddfSB8ICIKICAgICAgICAgICAgZiJ7"
        "J3VuYXZhaWxhYmxlJyBpZiBiIGlzIE5vbmUgZWxzZSBifSB8IgogICAgICAgICkKICAgIGxpbmVz"
        "ICs9IFsKICAgICAgICAiIiwKICAgICAgICAifCBBcm0gfCBQQVNTIHwgRkFJTCB8IEVSUk9SIHwg"
        "Tk9UX1JVTiB8IENvaG9ydCBkZW5vbWluYXRvciB8IiwKICAgICAgICAifCAtLS0gfCAtLS06IHwg"
        "LS0tOiB8IC0tLTogfCAtLS06IHwgLS0tOiB8IiwKICAgIF0KICAgIGZvciBhcm0sIGNvdW50cyBp"
        "biByZXBvcnRbIm91dGNvbWVzIl0uaXRlbXMoKToKICAgICAgICBsaW5lcy5hcHBlbmQoCiAgICAg"
        "ICAgICAgIGYifCB7YXJtfSB8IHtjb3VudHNbJ1BBU1MnXX0gfCB7Y291bnRzWydGQUlMJ119IHwg"
        "e2NvdW50c1snRVJST1InXX0gfCAiCiAgICAgICAgICAgIGYie2NvdW50c1snTk9UX1JVTiddfSB8"
        "IHtjb3VudHNbJ2NvaG9ydF9kZW5vbWluYXRvciddfSB8IgogICAgICAgICkKICAgIGxpbmVzICs9"
        "IFsiIiwgIiMjIFByb3ZlbmFuY2UiLCAiIl0KICAgIGZvciBhcm0sIHBpbiBpbiByZXBvcnRbInBy"
        "b3ZlbmFuY2UiXS5pdGVtcygpOgogICAgICAgIGxpbmVzLmFwcGVuZCgKICAgICAgICAgICAgZiIt"
        "IHthcm19OiB7Y2VsbChwaW5bJ3J1bl9pZCddKX07IHNvdXJjZSB7Y2VsbChwaW5bJ3NvdXJjZV9y"
        "dW5faWQnXSl9OyAiCiAgICAgICAgICAgIGYibWFuaWZlc3QgU0hBLTI1NiBge3BpblsnbWFuaWZl"
        "c3Rfc2hhMjU2J119YDsgIgogICAgICAgICAgICBmInJlY29yZHMgU0hBLTI1NiBge3BpblsncmVj"
        "b3Jkc19zaGEyNTYnXX1gLiIKICAgICAgICApCiAgICBsaW5lcyArPSBbIiIsICIjIyBVbmF2YWls"
        "YWJsZSBvciBub25jb21wYXJhYmxlIG1lYXN1cmVtZW50cyIsICIiXQogICAgZm9yIG5hbWUsIG1l"
        "dHJpYyBpbiByZXBvcnRbIm1ldHJpY3MiXS5pdGVtcygpOgogICAgICAgIGZvciBtaXNzaW5nIGlu"
        "IG1ldHJpY1siZXhjbHVkZWQiXToKICAgICAgICAgICAgbGluZXMuYXBwZW5kKAogICAgICAgICAg"
        "ICAgICAgZiItIHtuYW1lfSwge2NlbGwobWlzc2luZ1snaW5zdGFuY2VfaWQnXSl9OiAiCiAgICAg"
        "ICAgICAgICAgICBmIkUwOiB7Y2VsbChtaXNzaW5nWydFMCddKX07IEUxOiB7Y2VsbChtaXNzaW5n"
        "WydFMSddKX0uIgogICAgICAgICAgICApCiAgICBsaW5lcyArPSBbIiIsICIjIyBMaW1pdGF0aW9u"
        "cyIsICIiXQogICAgbGluZXMuZXh0ZW5kKGYiLSB7Y2VsbCh2YWx1ZSl9IiBmb3IgdmFsdWUgaW4g"
        "cmVwb3J0WyJsaW1pdGF0aW9ucyJdKQogICAgcmV0dXJuICJcbiIuam9pbihsaW5lcykgKyAiXG4i"
        "CgoKZGVmIHdyaXRlX3JlcG9ydChlMDogUGF0aCwgZTE6IFBhdGgsIGpzb25fcGF0aDogUGF0aCwg"
        "bWFya2Rvd25fcGF0aDogUGF0aCkgLT4gTm9uZToKICAgIHJlcG9ydCA9IGNvbXBhcmUobG9hZF9y"
        "dW4oZTApLCBsb2FkX3J1bihlMSkpCiAgICBqc29uX3BhdGgud3JpdGVfYnl0ZXMoY2Fub25pY2Fs"
        "X2pzb24ocmVwb3J0KSkKICAgIG1hcmtkb3duX3BhdGgud3JpdGVfdGV4dChtYXJrZG93bihyZXBv"
        "cnQpLCBlbmNvZGluZz0idXRmLTgiKQoKCmRlZiBtYWluKCkgLT4gTm9uZToKICAgIHBhcnNlciA9"
        "IGFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKGRlc2NyaXB0aW9uPV9fZG9jX18pCiAgICBwYXJzZXIu"
        "YWRkX2FyZ3VtZW50KCJlMCIsIHR5cGU9UGF0aCkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoImUx"
        "IiwgdHlwZT1QYXRoKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1qc29uIiwgdHlwZT1QYXRo"
        "LCByZXF1aXJlZD1UcnVlKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1tYXJrZG93biIsIHR5"
        "cGU9UGF0aCwgcmVxdWlyZWQ9VHJ1ZSkKICAgIGFyZ3MgPSBwYXJzZXIucGFyc2VfYXJncygpCiAg"
        "ICB3cml0ZV9yZXBvcnQoYXJncy5lMCwgYXJncy5lMSwgYXJncy5qc29uLCBhcmdzLm1hcmtkb3du"
        "KQoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICBtYWluKCkK"
    ),
    "src/mostaza/resources.py": (
        "IiIiQm91bmRlZCBvZmZpY2lhbCBhc3NldCByZXRyaWV2YWwgYW5kIGZhaWwtY2xvc2VkIHByb3Zl"
        "bmFuY2UgY2hlY2tzLiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1w"
        "b3J0IGhhc2hsaWIKaW1wb3J0IGpzb24KaW1wb3J0IG9zCmltcG9ydCB0ZW1wZmlsZQpmcm9tIHBh"
        "dGhsaWIgaW1wb3J0IFBhdGgKZnJvbSB0eXBpbmcgaW1wb3J0IEFueQpmcm9tIHVybGxpYi5wYXJz"
        "ZSBpbXBvcnQgdXJsc3BsaXQKZnJvbSB1cmxsaWIucmVxdWVzdCBpbXBvcnQgUmVxdWVzdCwgdXJs"
        "b3BlbgoKTUFYX1JFU09VUkNFX0JZVEVTID0gMTYgKiAxMDI0ICogMTAyNAoKCmNsYXNzIFJlc291"
        "cmNlRXJyb3IoVmFsdWVFcnJvcik6CiAgICAiIiJBbiBhdXRoZW50aWMgcmVxdWlyZWQgcmVzb3Vy"
        "Y2UgaXMgdW5hdmFpbGFibGUgb3IgY2hhbmdlZC4iIiIKCgpkZWYgc2hhMjU2KHBhdGg6IFBhdGgp"
        "IC0+IHN0cjoKICAgIGRpZ2VzdCA9IGhhc2hsaWIuc2hhMjU2KCkKICAgIHdpdGggcGF0aC5vcGVu"
        "KCJyYiIpIGFzIHN0cmVhbToKICAgICAgICBmb3IgYmxvY2sgaW4gaXRlcihsYW1iZGE6IHN0cmVh"
        "bS5yZWFkKDY1NTM2KSwgYiIiKToKICAgICAgICAgICAgZGlnZXN0LnVwZGF0ZShibG9jaykKICAg"
        "IHJldHVybiBkaWdlc3QuaGV4ZGlnZXN0KCkKCgpkZWYgcmVzb3VyY2VfcGF0aChyb290OiBQYXRo"
        "LCByZWxhdGl2ZTogc3RyKSAtPiBQYXRoOgogICAgcGFydHMgPSByZWxhdGl2ZS5zcGxpdCgiLyIp"
        "CiAgICBpZiAoCiAgICAgICAgbm90IHJlbGF0aXZlLnN0YXJ0c3dpdGgoInJlc291cmNlcy9vZmZp"
        "Y2lhbC8iKQogICAgICAgIG9yIGFueShwYXJ0IGluIHsiIiwgIi4iLCAiLi4ifSBmb3IgcGFydCBp"
        "biBwYXJ0cykKICAgICAgICBvciAiXFwiIGluIHJlbGF0aXZlCiAgICAgICAgb3IgIjoiIGluIHJl"
        "bGF0aXZlCiAgICApOgogICAgICAgIHJhaXNlIFJlc291cmNlRXJyb3IoZiJJbnZhbGlkIHJlc291"
        "cmNlIHBhdGg6IHtyZWxhdGl2ZX0iKQogICAgcGF0aCA9IHJvb3QgLyByZWxhdGl2ZQogICAgZm9y"
        "IHBhcmVudCBpbiBbcGF0aCwgKnBhdGgucGFyZW50c106CiAgICAgICAgaWYgcGFyZW50ID09IHJv"
        "b3QucGFyZW50OgogICAgICAgICAgICBicmVhawogICAgICAgIGlmIHBhcmVudC5pc19zeW1saW5r"
        "KCk6CiAgICAgICAgICAgIHJhaXNlIFJlc291cmNlRXJyb3IoZiJTeW1saW5rIHJlc291cmNlIHBh"
        "dGg6IHtyZWxhdGl2ZX0iKQogICAgaWYgbm90IHBhdGgucmVzb2x2ZSgpLmlzX3JlbGF0aXZlX3Rv"
        "KHJvb3QucmVzb2x2ZSgpIC8gInJlc291cmNlcyIgLyAib2ZmaWNpYWwiKToKICAgICAgICByYWlz"
        "ZSBSZXNvdXJjZUVycm9yKGYiUmVzb3VyY2UgZXNjYXBlcyBpbnZlbnRvcnk6IHtyZWxhdGl2ZX0i"
        "KQogICAgcmV0dXJuIHBhdGgKCgpkZWYgcmVhZF9tYW5pZmVzdChyb290OiBQYXRoKSAtPiBkaWN0"
        "W3N0ciwgQW55XToKICAgIHZhbHVlID0ganNvbi5sb2Fkcygocm9vdCAvICJyZXNvdXJjZXMvbWFu"
        "aWZlc3QuanNvbiIpLnJlYWRfdGV4dCgpKQogICAgaWYgbm90IGlzaW5zdGFuY2UodmFsdWUsIGRp"
        "Y3QpIG9yIHZhbHVlLmdldCgic2NoZW1hX3ZlcnNpb24iKSAhPSAxOgogICAgICAgIHJhaXNlIFJl"
        "c291cmNlRXJyb3IoIlVuc3VwcG9ydGVkIHJlc291cmNlIG1hbmlmZXN0IikKICAgIGVudHJpZXMg"
        "PSB2YWx1ZS5nZXQoInJlc291cmNlcyIpCiAgICBpZiBub3QgaXNpbnN0YW5jZShlbnRyaWVzLCBs"
        "aXN0KSBvciBhbnkobm90IGlzaW5zdGFuY2UoZSwgZGljdCkgZm9yIGUgaW4gZW50cmllcyk6CiAg"
        "ICAgICAgcmFpc2UgUmVzb3VyY2VFcnJvcigiSW52YWxpZCByZXNvdXJjZSBpbnZlbnRvcnkiKQog"
        "ICAgaWRzID0gW2VudHJ5LmdldCgiaWQiKSBmb3IgZW50cnkgaW4gZW50cmllc10KICAgIGlmIGFu"
        "eShub3QgaXNpbnN0YW5jZShpLCBzdHIpIGZvciBpIGluIGlkcykgb3IgbGVuKHNldChpZHMpKSAh"
        "PSBsZW4oaWRzKToKICAgICAgICByYWlzZSBSZXNvdXJjZUVycm9yKCJEdXBsaWNhdGUgb3IgaW52"
        "YWxpZCByZXNvdXJjZSBJRHMiKQogICAgcmV0dXJuIHZhbHVlCgoKZGVmIHJlcXVpcmVfcmVzb3Vy"
        "Y2VzKHJvb3Q6IFBhdGgsIGlkczogbGlzdFtzdHJdKSAtPiBkaWN0W3N0ciwgUGF0aF06CiAgICBl"
        "bnRyaWVzID0ge2VudHJ5WyJpZCJdOiBlbnRyeSBmb3IgZW50cnkgaW4gcmVhZF9tYW5pZmVzdChy"
        "b290KVsicmVzb3VyY2VzIl19CiAgICByZXN1bHQ6IGRpY3Rbc3RyLCBQYXRoXSA9IHt9CiAgICBm"
        "b3IgcmVzb3VyY2VfaWQgaW4gaWRzOgogICAgICAgIGVudHJ5ID0gZW50cmllcy5nZXQocmVzb3Vy"
        "Y2VfaWQpCiAgICAgICAgaWYgZW50cnkgaXMgTm9uZSBvciBlbnRyeS5nZXQoInN0YXR1cyIpICE9"
        "ICJ2ZXJpZmllZCI6CiAgICAgICAgICAgIHJhaXNlIFJlc291cmNlRXJyb3IoZiJPZmZpY2lhbCBw"
        "cmVyZXF1aXNpdGUgcGVuZGluZzoge3Jlc291cmNlX2lkfSIpCiAgICAgICAgcGF0aCA9IHJlc291"
        "cmNlX3BhdGgocm9vdCwgZW50cnlbInBhdGgiXSkKICAgICAgICBpZiBub3QgcGF0aC5pc19maWxl"
        "KCkgb3Igc2hhMjU2KHBhdGgpICE9IGVudHJ5LmdldCgic2hhMjU2Iik6CiAgICAgICAgICAgIHJh"
        "aXNlIFJlc291cmNlRXJyb3IoZiJSZXNvdXJjZSBtaXNzaW5nIG9yIGhhc2ggbWlzbWF0Y2g6IHty"
        "ZXNvdXJjZV9pZH0iKQogICAgICAgIHJlc3VsdFtyZXNvdXJjZV9pZF0gPSBwYXRoCiAgICByZXR1"
        "cm4gcmVzdWx0CgoKZGVmIGZldGNoX3Jlc291cmNlKHJvb3Q6IFBhdGgsIHJlc291cmNlX2lkOiBz"
        "dHIpIC0+IFBhdGg6CiAgICAiIiJGZXRjaCBvbmUgcGlubmVkIHB1YmxpYyBmaWxlLCBuZXZlciBk"
        "YXRhc2V0cywgYXJjaGl2ZXMgb3IgbW9kZWwgd2VpZ2h0cy4iIiIKICAgIGVudHJpZXMgPSB7ZW50"
        "cnlbImlkIl06IGVudHJ5IGZvciBlbnRyeSBpbiByZWFkX21hbmlmZXN0KHJvb3QpWyJyZXNvdXJj"
        "ZXMiXX0KICAgIGVudHJ5ID0gZW50cmllcy5nZXQocmVzb3VyY2VfaWQpCiAgICBpZiBlbnRyeSBp"
        "cyBOb25lIG9yIG5vdCBlbnRyeS5nZXQoImRvd25sb2FkX3VybCIpIG9yIG5vdCBlbnRyeS5nZXQo"
        "InNoYTI1NiIpOgogICAgICAgIHJhaXNlIFJlc291cmNlRXJyb3IoIlJlc291cmNlIG5lZWRzIGFu"
        "IGF1dGhlbnRpY2F0ZWQgb2ZmaWNpYWwgZG93bmxvYWQvcmV2aWV3IikKICAgIHVybCA9IGVudHJ5"
        "WyJkb3dubG9hZF91cmwiXQogICAgcGFyc2VkID0gdXJsc3BsaXQodXJsKQogICAgaWYgcGFyc2Vk"
        "LnNjaGVtZSAhPSAiaHR0cHMiIG9yIHBhcnNlZC5ob3N0bmFtZSAhPSAid3d3LmthZ2dsZS5jb20i"
        "OgogICAgICAgIHJhaXNlIFJlc291cmNlRXJyb3IoIkRvd25sb2FkIG11c3Qgb3JpZ2luYXRlIGF0"
        "IHRoZSBvZmZpY2lhbCBLYWdnbGUgZW5kcG9pbnQiKQogICAgdGFyZ2V0ID0gcmVzb3VyY2VfcGF0"
        "aChyb290LCBlbnRyeVsicGF0aCJdKQogICAgdGFyZ2V0LnBhcmVudC5ta2RpcihwYXJlbnRzPVRy"
        "dWUsIGV4aXN0X29rPVRydWUpCiAgICBpZiB0YXJnZXQuZXhpc3RzKCk6CiAgICAgICAgaWYgc2hh"
        "MjU2KHRhcmdldCkgIT0gZW50cnlbInNoYTI1NiJdOgogICAgICAgICAgICByYWlzZSBSZXNvdXJj"
        "ZUVycm9yKCJFeGlzdGluZyByZXNvdXJjZSBkaWZmZXJzOyByZWZ1c2luZyB0byBvdmVyd3JpdGUi"
        "KQogICAgICAgIHJldHVybiB0YXJnZXQKICAgIHRlbXBvcmFyeTogUGF0aCB8IE5vbmUgPSBOb25l"
        "CiAgICB0cnk6CiAgICAgICAgd2l0aCB1cmxvcGVuKAogICAgICAgICAgICBSZXF1ZXN0KHVybCwg"
        "aGVhZGVycz17IlVzZXItQWdlbnQiOiAiTW9zdGF6YS8wLjEifSksIHRpbWVvdXQ9NjAKICAgICAg"
        "ICApIGFzIHJlc3BvbnNlOgogICAgICAgICAgICBmaW5hbF9ob3N0ID0gdXJsc3BsaXQocmVzcG9u"
        "c2UudXJsKS5ob3N0bmFtZSBvciAiIgogICAgICAgICAgICBpZiBub3QgKAogICAgICAgICAgICAg"
        "ICAgZmluYWxfaG9zdCA9PSAid3d3LmthZ2dsZS5jb20iCiAgICAgICAgICAgICAgICBvciBmaW5h"
        "bF9ob3N0ID09ICJzdG9yYWdlLmdvb2dsZWFwaXMuY29tIgogICAgICAgICAgICAgICAgb3IgZmlu"
        "YWxfaG9zdC5lbmRzd2l0aCgiLmdvb2dsZXVzZXJjb250ZW50LmNvbSIpCiAgICAgICAgICAgICk6"
        "CiAgICAgICAgICAgICAgICByYWlzZSBSZXNvdXJjZUVycm9yKCJVbmV4cGVjdGVkIG9mZmljaWFs"
        "IGRvd25sb2FkIHJlZGlyZWN0IikKICAgICAgICAgICAgd2l0aCB0ZW1wZmlsZS5OYW1lZFRlbXBv"
        "cmFyeUZpbGUoZGlyPXRhcmdldC5wYXJlbnQsIGRlbGV0ZT1GYWxzZSkgYXMgb3V0cHV0OgogICAg"
        "ICAgICAgICAgICAgdGVtcG9yYXJ5ID0gUGF0aChvdXRwdXQubmFtZSkKICAgICAgICAgICAgICAg"
        "IHRvdGFsID0gMAogICAgICAgICAgICAgICAgd2hpbGUgYmxvY2sgOj0gcmVzcG9uc2UucmVhZCg2"
        "NTUzNik6CiAgICAgICAgICAgICAgICAgICAgdG90YWwgKz0gbGVuKGJsb2NrKQogICAgICAgICAg"
        "ICAgICAgICAgIGlmIHRvdGFsID4gbWluKAogICAgICAgICAgICAgICAgICAgICAgICBlbnRyeS5n"
        "ZXQoIm1heF9ieXRlcyIsIE1BWF9SRVNPVVJDRV9CWVRFUyksIE1BWF9SRVNPVVJDRV9CWVRFUwog"
        "ICAgICAgICAgICAgICAgICAgICk6CiAgICAgICAgICAgICAgICAgICAgICAgIHJhaXNlIFJlc291"
        "cmNlRXJyb3IoIlJlc291cmNlIGV4Y2VlZHMgYm91bmRlZCBkb3dubG9hZCBzaXplIikKICAgICAg"
        "ICAgICAgICAgICAgICBvdXRwdXQud3JpdGUoYmxvY2spCiAgICAgICAgaWYgc2hhMjU2KHRlbXBv"
        "cmFyeSkgIT0gZW50cnlbInNoYTI1NiJdOgogICAgICAgICAgICByYWlzZSBSZXNvdXJjZUVycm9y"
        "KCJEb3dubG9hZGVkIGJ5dGVzIGRpZmZlciBmcm9tIHRoZSBhcHByb3ZlZCBwaW4iKQogICAgICAg"
        "ICMgSGFyZCBsaW5rIHB1Ymxpc2hlcyBhdG9taWNhbGx5IHdpdGhvdXQgcmVwbGFjaW5nIGNvbmN1"
        "cnJlbnQvdXNlciBmaWxlcy4KICAgICAgICBvcy5saW5rKHRlbXBvcmFyeSwgdGFyZ2V0KQogICAg"
        "ICAgIHJldHVybiB0YXJnZXQKICAgIGZpbmFsbHk6CiAgICAgICAgaWYgdGVtcG9yYXJ5IGlzIG5v"
        "dCBOb25lOgogICAgICAgICAgICB0ZW1wb3JhcnkudW5saW5rKG1pc3Npbmdfb2s9VHJ1ZSkK"
    ),
}
EMBEDDED_HASHES = {
    "archives/E0.zip": (
        "aa40c075b2bf77c62cca80b9a9b1ec8fd5f28131295c9bf03926fce241464fe4"
    ),
    "archives/E1.zip": (
        "be8ebd8e660c75b6a7e467b6590e0ff4f093077055e7aeee0db6fa6a0ddd375e"
    ),
    "cohort.json": ("1ee1eef565392ee98efb2a4aa6398c79611a6e95d61def165410f2aaee918a09"),
    "execution-metadata.json": (
        "03e821e0d1261edaa92c9e35c26686cb7fd809c15564b2f6d859038e9e897897"
    ),
    "resource-review.json": (
        "5a921ab8bc6b615f6405cb9f1847732cc60d7ff6aa39569f4b682d14feb3f5a1"
    ),
    "src/mostaza/__init__.py": (
        "e6e7452e239f7a03d6645be265792ed103fcc93b27da4e1f2b982aa8af018d3c"
    ),
    "src/mostaza/evaluation/__init__.py": (
        "2106119b057d082ec60e8a8e21aef6fcc3f7d63a3ff4d0e177ddbac3db9efcac"
    ),
    "src/mostaza/evaluation/contracts.py": (
        "b8500c062a3e81b7bc0fb6051e72cc4af07689fc1cf2f8dc60f8c58a8c87e7d3"
    ),
    "src/mostaza/evaluation/import_results.py": (
        "57c94f88cf97fdc0ed9bbf168ffe5334076a11b6a71c30ecd6e0ea3e77564859"
    ),
    "src/mostaza/evaluation/runner.py": (
        "5f56dfd22e339ac72ae6c93b0b43c4229f92959b5fc28cda1f531d952b229770"
    ),
    "src/mostaza/evaluation/tasks.py": (
        "eab3240c3ddc78114b849b81b8944a74daadd6da7e1859e7bfa99455e6a8dc91"
    ),
    "src/mostaza/packaging.py": (
        "0d391b6c2aecff94f00586d0bc62cbe95b4e8d6a7cd7ddb3b7e8987f9481f725"
    ),
    "src/mostaza/research/__init__.py": (
        "f9bbf774dea93e1ddfc7a00710fbe737cfb2e1404fbb7b83857cbb24e0051059"
    ),
    "src/mostaza/research/metrics.py": (
        "4d8f359e849947fb5392d487151c61dd4b91c1bc01b2dbf4f7a81e4ce843c153"
    ),
    "src/mostaza/research/report.py": (
        "7401c01610e7d2d3fa7e5173b8ee696239fc3b552461ea4579ee86899fb3f3ef"
    ),
    "src/mostaza/resources.py": (
        "b3b628f0b40e5303d18b412664fc2f542088983838ed003960cb0886ef41530c"
    ),
}

EMBEDDED_ROOT = Path("/kaggle/working") / (
    "mostaza-reviewed-input-" + str(uuid.uuid4())
)
EMBEDDED_ROOT.mkdir(parents=True, exist_ok=False)
for name, encoded in EMBEDDED_PAYLOAD.items():
    data = base64.b64decode(encoded, validate=True)
    if hashlib.sha256(data).hexdigest() != EMBEDDED_HASHES[name]:
        raise RuntimeError("Embedded source/config bytes do not match reviewed pins")
    target = EMBEDDED_ROOT / name
    target.parent.mkdir(parents=True, exist_ok=True)
    with target.open("xb") as stream:
        stream.write(data)
print("Reviewed original host source and E0/E1 archives materialized; no inference yet")

In [ ]:
import ast
import hashlib
import os
import re
import sys
from datetime import UTC, datetime
from pathlib import Path

# Fill real Kaggle mount paths after the host prerequisites succeed.
SOURCE_ROOT = EMBEDDED_ROOT  # Reviewed original source emitted by bootstrap.
COHORT_PATH = EMBEDDED_ROOT / "cohort.json"
RESOURCE_REVIEW_PATH = EMBEDDED_ROOT / "resource-review.json"
ARCHIVES = {arm: EMBEDDED_ROOT / "archives" / (arm + ".zip") for arm in ("E0", "E1")}
EXECUTION_METADATA_PATH = EMBEDDED_ROOT / "execution-metadata.json"
SOURCE_RUN_URL = (
    "https://www.kaggle.com/code/diegofescobar/"
    "mostaza-e0-e1-phase-1-paired-benchmark"
)  # Actual existing private notebook; preserve the saved execution version.

DATA_DIR = Path("/kaggle/input/competitions/gemma-4-developer-agent")
WHEELHOUSE_DIR = Path(
    "/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse"
)
MODEL_NAME = "gemma-4-31b-it-qat-w4a16-ct"
MODEL_PATH = Path(f"/kaggle/input/models/google/gemma-4/other/{MODEL_NAME}/2")


def need(condition, reason):
    if not condition:
        raise RuntimeError(reason)


def file_sha(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


need(
    Path("/kaggle/input").is_dir(),
    "Run only in Kaggle; never download the model on the Mac",
)
need(sys.version_info[:2] == (3, 12), "Review a changed official kernel before running")
need(
    all(
        value is not None
        for value in (
            SOURCE_ROOT,
            COHORT_PATH,
            RESOURCE_REVIEW_PATH,
            EXECUTION_METADATA_PATH,
            *ARCHIVES.values(),
        )
    ),
    "Fill all real input paths; no defaults are inferred",
)
need(
    isinstance(SOURCE_RUN_URL, str)
    and SOURCE_RUN_URL.startswith("https://www.kaggle.com/code/"),
    "Provide the actual durable Kaggle run URL/version",
)
SOURCE_ROOT = Path(SOURCE_ROOT).resolve(strict=True)
need(
    (SOURCE_ROOT / "src/mostaza/__init__.py").is_file(),
    "Missing reviewed Mostaza source",
)
need(
    SOURCE_ROOT == EMBEDDED_ROOT.resolve(strict=True),
    "Use the reviewed embedded original source input",
)
for path in (
    DATA_DIR,
    WHEELHOUSE_DIR,
    MODEL_PATH,
    DATA_DIR / "snapshots",
    DATA_DIR / "graphs",
    DATA_DIR / "embeddings",
    DATA_DIR / "wheels",
):
    need(path.is_dir(), f"Missing official resource directory: {path}")
for path in (
    DATA_DIR / "HARNESS_README.md",
    DATA_DIR / "sample_submission/agent.yaml",
    DATA_DIR / "tasks.jsonl",
):
    need(path.is_file(), f"Missing authentic official resource: {path}")
cohort = json.loads(Path(COHORT_PATH).read_text())
need(
    set(cohort) == {"schema_version", "frozen_at", "selection_basis", "tasks"},
    "Pending/unknown cohort schema is invalid for execution",
)
need(
    cohort["schema_version"] == 1
    and cohort["selection_basis"] == "pre-outcome"
    and len(cohort["tasks"]) == 3,
    "Freeze the real three-task cohort first",
)
frozen_at = datetime.fromisoformat(cohort["frozen_at"].replace("Z", "+00:00"))
need(
    frozen_at.utcoffset().total_seconds() == 0 and frozen_at < datetime.now(UTC),
    "Cohort needs a past UTC freeze time",
)
for selected in cohort["tasks"]:
    need(
        isinstance(selected, dict)
        and set(selected)
        == {"instance_id", "repo", "base_commit", "snapshot_sha256", "rationale"},
        "Invalid frozen task schema",
    )
    need(
        all(isinstance(value, str) and value.strip() for value in selected.values()),
        "Frozen task identity and pre-outcome rationale are required",
    )
    need(
        re.fullmatch(r"[0-9a-f]{40}", selected["base_commit"])
        and re.fullmatch(r"[0-9a-f]{64}", selected["snapshot_sha256"]),
        "Frozen commit and snapshot digests are invalid",
    )
repos = {task["repo"].casefold() for task in cohort["tasks"]}
need(
    "psf/requests" in repos
    and "fastapi/fastapi" in repos
    and bool(repos & {"textualize/rich", "encode/httpx"}),
    "Required repository cohort missing",
)
review = json.loads(Path(RESOURCE_REVIEW_PATH).read_text())
need(
    review.get("status") == "VERIFIED"
    and review.get("official_resources_reviewed") is True,
    "Finish actual README/sample/resource review first",
)
reviewed = {entry["id"]: entry for entry in review["resources"]}
for resource_id, path in {
    "harness_readme": DATA_DIR / "HARNESS_README.md",
    "starter_agent": DATA_DIR / "sample_submission/agent.yaml",
    "tasks": DATA_DIR / "tasks.jsonl",
}.items():
    pin = reviewed[resource_id]
    need(
        pin["status"] == "verified" and file_sha(path) == pin["sha256"],
        f"Official bytes do not match reviewed {resource_id}",
    )
for name in ("swegemma", "adk_submission", "adk_eval_core"):
    pin = reviewed[name]
    wheel = WHEELHOUSE_DIR / Path(pin["path"]).name
    need(
        pin["status"] == "verified" and file_sha(wheel) == pin["sha256"],
        f"Pinned official wheel changed: {name}",
    )
# Source integration is separate from installing the Python 3.13 host distribution.
for source in (SOURCE_ROOT / "src/mostaza").rglob("*.py"):
    ast.parse(source.read_text(), filename=str(source), feature_version=(3, 12))
sys.path.insert(0, str(SOURCE_ROOT / "src"))

In [ ]:
from mostaza.evaluation.tasks import load_agent_tasks

visible_tasks = {
    task.instance_id: task for task in load_agent_tasks(DATA_DIR / "tasks.jsonl")
}
for selected in cohort["tasks"]:
    task = visible_tasks.get(selected["instance_id"])
    need(
        task is not None
        and task.repo == selected["repo"]
        and task.base_commit == selected["base_commit"],
        "Frozen cohort differs from real task metadata",
    )
need(
    len({task["instance_id"] for task in cohort["tasks"]}) == 3, "Duplicate cohort task"
)
print("Authentic resources and frozen cohort preflight passed; no inference yet")

## Offline official runtime

This setup follows starter version 3's wheel-name restoration and offline
installation. Kernel setup happens only after authentic input preflight. Run
only inside Kaggle. The local Mac compiler environment is not a GPU runtime.


In [ ]:
import glob
import importlib
import logging
import shutil
import subprocess
import sys
from pathlib import Path

# Configure environment variables for offline vLLM serving and LiteLLM routing
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TRANSFORMERS_NO_FLAX"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["VLLM_NO_USAGE_STATS"] = "1"
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"
os.environ["VLLM_ENGINE_READY_TIMEOUT_S"] = "1200"
os.environ["VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS"] = "1"
os.environ["NCCL_DEBUG"] = "WARN"
os.environ["LITELLM_LOCAL_MODEL_COST_MAP"] = "True"
os.environ["LITELLM_TELEMETRY"] = "False"
os.environ["LITELLM_LOG"] = "WARNING"
os.environ["OPENAI_API_KEY"] = os.environ.get("OPENAI_API_KEY", "EMPTY")
os.environ["LOCAL_API_KEY"] = os.environ.get("LOCAL_API_KEY", "EMPTY")
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["OTEL_SDK_DISABLED"] = "true"
os.environ["OTEL_PYTHON_DISABLED"] = "true"


def configure_logging() -> None:
    """Silence verbose third-party loggers."""
    for name in (
        "LiteLLM",
        "LiteLLM Router",
        "LiteLLM Proxy",
        "litellm",
        "httpx",
        "httpcore",
        "openai",
        "urllib3",
        "opentelemetry",
        "opentelemetry.context",
    ):
        lg = logging.getLogger(name)
        lg.setLevel(logging.WARNING)
        if "litellm" in name.lower():
            lg.propagate = False
            lg.handlers.clear()


configure_logging()

WHEELHOUSE_DIR = Path(
    "/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse"
)

# Remove broken cutlass .pth hooks if present
for pth_pattern in (
    "/usr/local/lib/python*/dist-packages/*cutlass*.pth",
    "/usr/local/lib/python*/site-packages/*cutlass*.pth",
):
    for pth in glob.glob(pth_pattern):
        try:
            os.unlink(pth)
        except OSError:
            pass

# Restore PEP 440 '+cu128' wheel filenames stripped by Kaggle dataset uploads
tmp_whl = Path("/tmp/wheelhouse")
tmp_whl.mkdir(parents=True, exist_ok=True)
for w in WHEELHOUSE_DIR.glob("*.whl"):
    if "cutlass" in w.name.lower():
        continue
    target_name = (
        w.name.replace("cu128", "+cu128")
        if ("cu128" in w.name and "+" not in w.name)
        else w.name
    )
    target = tmp_whl / target_name
    if not target.exists():
        try:
            os.symlink(w, target)
        except OSError:
            shutil.copyfile(w, target)

wheels = sorted(str(w) for w in tmp_whl.glob("*.whl"))
print(f"Installing {len(wheels)} wheels from {WHEELHOUSE_DIR}...")
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--no-deps",
        "--force-reinstall",
        *wheels,
    ],
    check=True,
)
importlib.invalidate_caches()
configure_logging()
print("Wheelhouse installation complete.")

In [ ]:
from importlib.metadata import version

import torch
from swegemma.deduplication import resolve_task_snapshot_paths

from mostaza.evaluation.contracts import canonical_json, sha256

need(
    version("swegemma") == "0.2.11"
    and version("adk-submission") == "0.2.13"
    and version("adk-eval-core") == "0.1.0",
    "Pinned organizer package versions required",
)
need(
    torch.cuda.is_available() and torch.cuda.device_count() == 4,
    "Actual L4 x4 runtime required",
)
gpu_names = [torch.cuda.get_device_name(index) for index in range(4)]
need(all(name == "NVIDIA L4" for name in gpu_names), "Unexpected GPU model")


def snapshot_pin(task):
    # Hash ALL reconstruction inputs, not only a deduplicated shared base archive.
    effective, base, delta = resolve_task_snapshot_paths(
        DATA_DIR / "snapshots", task["instance_id"], task["repo"]
    )
    components = []
    for role, path in (("effective", effective), ("base", base), ("delta", delta)):
        if path is not None:
            need(path.is_file(), "Missing frozen snapshot component")
            components.append(
                {
                    "role": role,
                    "path": str(path.relative_to(DATA_DIR)),
                    "sha256": file_sha(path),
                }
            )
    return sha256(canonical_json(components))


for selected in cohort["tasks"]:
    need(
        snapshot_pin(selected) == selected["snapshot_sha256"],
        "Snapshot reconstruction inputs changed after cohort freeze",
    )
need(
    review.get("selected_resources_complete") is True,
    "Selected asset review is incomplete",
)
for pin in review["selected_resources"]:
    path = DATA_DIR / pin["official_file_path"]
    need(
        path.is_file() and file_sha(path) == pin["sha256"],
        "Selected official resource bytes changed",
    )
print(
    "Pinned runtime and selected asset preflight passed; no reference answers displayed"
)

## Verified arm archives and execution metadata

Host commands below create the two archives after actual E0/E1 integration. The
metadata input contains **only** `experiment_id`, `code_commit`,
`working_tree_sha256`, : record the actual repository base HEAD and the reviewed runtime source-overlay inventory
digest, never example values. The digest scope is explicitly recorded in the
manifest environment; exact notebook source is retained with the actual Kaggle
run version.
The competition UI did not expose a numeric dataset version; its immutable
content version is explicitly identified as `sha256:<actual tasks digest>`. Archive
hashes, settings, measured runtime, task pins and UTC execution times are derived
below. `SOURCE_RUN_URL` must identify the saved Kaggle notebook version, not a
made-up run. Each actual attempt receives a fresh UUID run ID and a new directory.

The baseline and hybrid configs must differ only in their `instruction` value
and included prompt contents. Their native tool set, model, generation fields
and evaluation config bytes are checked before the model starts. No adapters or
subagents are accepted by this Phase 1 route.


In [ ]:
import re
import uuid
import zipfile

import yaml
from adk_submission import (
    ModelRegistry,
    compile_submission,
    discover_adapters,
    validate_directory,
)
from adk_submission.discovery import discover_declared_models
from adk_submission.schema import SandboxedAgentConfig
from adk_submission.yaml_loader import load_yaml
from google.adk.agents.context_cache_config import ContextCacheConfig
from google.adk.apps._configs import EventsCompactionConfig
from swegemma.config import EvalConfig, build_submission_limits

from mostaza.evaluation.contracts import MODEL, validate_manifest
from mostaza.evaluation.runner import describe_runtime, evaluate_official
from mostaza.packaging import MAX_BYTES, MAX_FILES, collect_submission

metadata = json.loads(Path(EXECUTION_METADATA_PATH).read_text())
need(
    set(metadata) == {"experiment_id", "code_commit", "working_tree_sha256"},
    "Explicit execution metadata fields required",
)
need(
    all(isinstance(value, str) and value.strip() for value in metadata.values()),
    "Execution metadata is incomplete",
)
need(
    re.fullmatch(r"[0-9a-f]{40}", metadata["code_commit"])
    and re.fullmatch(r"[0-9a-f]{64}", metadata["working_tree_sha256"]),
    "Actual code commit/tree digest required",
)
RUN_ROOT = Path("/kaggle/working/private-run-evidence") / str(uuid.uuid4())
RUN_ROOT.mkdir(parents=True, exist_ok=False)
limits, constraints = build_submission_limits()
agent_dirs, raw_agents, compiler_receipts, archive_hashes = {}, {}, {}, {}
for arm in ("E0", "E1"):
    archive_path = Path(ARCHIVES[arm])
    archive_hashes[arm] = file_sha(archive_path)
    target = RUN_ROOT / arm / "submission"
    target.mkdir(parents=True, exist_ok=False)
    with zipfile.ZipFile(archive_path) as archive:
        entries = archive.infolist()
        need(
            len(entries) <= MAX_FILES
            and sum(entry.file_size for entry in entries) <= MAX_BYTES,
            "Archive exceeds declarative MVP bounds",
        )
        seen = set()
        for entry in entries:
            name = entry.filename
            need(
                name not in seen
                and not entry.is_dir()
                and not Path(name).is_absolute()
                and ".." not in Path(name).parts
                and "\\" not in name,
                "Unsafe or duplicate archive member",
            )
            need(
                (entry.external_attr >> 16) & 0o170000 != 0o120000,
                "Archive symlinks are forbidden",
            )
            seen.add(name)
            path = target / name
            path.parent.mkdir(parents=True, exist_ok=True)
            with path.open("xb") as stream:
                stream.write(archive.read(entry))
    # Packaging validates transitive includes and rejects extra Python/references.
    files = collect_submission(target)
    need(set(files) == seen, "Archive has unused or forbidden content")
    validate_directory(target, limits=limits)
    raw = load_yaml(target / "agent.yaml", root_dir=target, limits=limits)
    SandboxedAgentConfig.model_validate(raw)
    need(
        discover_declared_models(target) == {MODEL},
        "Only the required 31B model is allowed",
    )
    adapters = discover_adapters(str(target))
    need(not (getattr(adapters, "adapters", None) or {}), "Phase 1 uses no adapters")
    # Real compiler API, inert bindings; this receipt explicitly proves no inference.
    registry = ModelRegistry()
    registry.register(MODEL, "compiler-only-inert-model")

    def inert_tool(**kwargs):
        raise RuntimeError("Inert compiler binding must never execute")

    compile_submission(
        target,
        {name: inert_tool for name in raw["tools"]},
        registry,
        limits=limits,
        generation_constraints=constraints,
    )
    compiler_receipts[arm] = {
        "status": "PASS",
        "archive_sha256": archive_hashes[arm],
        "source": SOURCE_RUN_URL,
        "validation_kind": "official_compiler_only_inert_bindings",
    }
    agent_dirs[arm], raw_agents[arm] = target, raw
need(
    {key: value for key, value in raw_agents["E0"].items() if key != "instruction"}
    == {key: value for key, value in raw_agents["E1"].items() if key != "instruction"},
    "Non-treatment agent fields differ",
)
need(
    raw_agents["E0"]["instruction"] != raw_agents["E1"]["instruction"],
    "The treatment instructions must differ",
)


def evaluation_bytes(directory):
    path = directory / "eval_config.yaml"
    return path.read_bytes() if path.exists() else None


need(
    evaluation_bytes(agent_dirs["E0"]) == evaluation_bytes(agent_dirs["E1"]),
    "Non-treatment evaluation config differs",
)
eval_data = evaluation_bytes(agent_dirs["E0"])
raw_eval = (yaml.safe_load(eval_data) or {}) if eval_data is not None else {}
section = raw_eval.get("evaluation", raw_eval)
# These fallbacks are verified starter-v3 values, used only after authentic review.
# They are not replacements for an unavailable sample config.
turns = section.get("max_turns", section.get("max_llm_calls"))
eval_kwargs = {
    "timeout_seconds": int(section.get("timeout_seconds", 300)),
    "verification_timeout_seconds": int(
        section.get("verification_timeout_seconds", 300)
    ),
    "max_tool_calls": int(section.get("max_tool_calls", 100)),
    "max_time_minutes": float(section.get("max_time_minutes", 60.0)),
    "max_turns": int(turns) if turns is not None else None,
}
# Explicit starter-v3 cache/compaction parameters, identical for both arms.
cache = ContextCacheConfig(min_tokens=2048, ttl_seconds=1800, cache_intervals=10)
compaction = EventsCompactionConfig(
    compaction_interval=15,
    overlap_size=2,
    token_threshold=14336,
    event_retention_size=5,
)
expected = review["baseline_derivation"]["shared_conditions"]
need(
    raw_agents["E0"]["generate_content_config"] == expected["generation_parameters"],
    "Sampling differs from authenticated baseline",
)
need(
    raw_agents["E0"]["tools"] == expected["tools"],
    "Native tool registry differs from authenticated baseline",
)
need(
    {key: eval_kwargs[key] for key in expected["budgets"]} == expected["budgets"],
    "Global budgets differ from authenticated baseline",
)
need(
    eval_kwargs["verification_timeout_seconds"] == 300,
    "Retain official verification timeout",
)
print(
    "Both real archives compiled with authenticated shared defaults; no inference yet"
)

## One official 31B server and matched paired execution

The server fields below come from starter v3's 31B branch. We require four L4s
instead of silently reducing tensor parallelism or changing models. The official
Evaluator performs generation and isolated verification. No private task objects
are serialized by this notebook. Actual result JSON, traces, patches and test
receipts stay in host output outside the agent submission.

This cell starts costly GPU inference. Stop here until the preflight and actual
resource/baseline review are complete. Run each arm on the same ordered cohort.
The `finally` block stops the server even if evaluation or provenance export fails.


In [ ]:
import shutil

import litellm
from adk_submission import VllmConfig, VllmServer

from mostaza.evaluation.import_results import normalize_official_result

litellm.drop_params = True
vllm_cfg = VllmConfig(
    model=str(MODEL_PATH),
    port=8000,
    host="127.0.0.1",
    tool_call_parser="gemma4",
    reasoning_parser="gemma4",
    max_model_len=32768,
    dtype="bfloat16" if torch.cuda.is_bf16_supported() else "auto",
    gpu_memory_utilization=0.90,
    enable_auto_tool_choice=True,
    enable_lora=False,
    max_loras=1,
    max_lora_rank=128,
    scheduler_reserve_full_isl=False,
    tensor_parallel_size=4,
    startup_timeout=60 * 20,
    extra_args=[],
)
server_instance = VllmServer(vllm_cfg)
# Hash every actual attached wheel for the wheelhouse resource identity.
wheel_inventory = [
    {"name": path.name, "sha256": file_sha(path)}
    for path in sorted(WHEELHOUSE_DIR.glob("*.whl"))
]
resources = {
    "harness": {
        "version": version("swegemma"),
        "sha256": reviewed["swegemma"]["sha256"],
        "source": reviewed["swegemma"]["url"],
    },
    "wheelhouse": {
        "version": str(reviewed["swegemma"]["wheelhouse_version"]),
        "sha256": sha256(canonical_json(wheel_inventory)),
        "source": reviewed["swegemma"]["url"],
    },
    "dataset": {
        "version": "sha256:" + file_sha(DATA_DIR / "tasks.jsonl"),
        "sha256": file_sha(DATA_DIR / "tasks.jsonl"),
        "source": reviewed["tasks"]["url"],
    },
}
environment = {
    "platform": "kaggle",
    "gpu_count": 4,
    "gpu_model": "NVIDIA L4",
    "python": sys.version,
    "packages": {
        name: version(name)
        for name in (
            "swegemma",
            "adk-submission",
            "adk-eval-core",
            "google-adk",
            "vllm",
            "torch",
        )
    },
}

environment["embedded_input_hashes"] = EMBEDDED_HASHES
environment["working_tree_digest_basis"] = (
    "SHA256(canonical_json(reviewed runtime source, archives, cohort, "
    "resource-review inventory before execution-metadata self-reference)); "
    "base code_commit is actual repository HEAD"
)
environment["snapshot_validation"] = {
    pin["instance_id"]: pin
    for pin in review["selected_resources"]
    if pin["kind"] == "snapshot"
}
manifests = {}
normalized_records = {}
try:
    server_instance.start()
    models = server_instance.create_model_registry(
        aliases=[MODEL],
        model_prefix="openai/",
        api_key=os.environ.get("LOCAL_API_KEY", "EMPTY"),
    )
    for arm in ("E0", "E1"):
        config = EvalConfig(
            tasks_path=DATA_DIR / "tasks.jsonl",
            snapshots_dir=DATA_DIR / "snapshots",
            results_dir=RUN_ROOT / arm / "results",
            submission_dir=agent_dirs[arm],
            models=models,
            sandbox="subprocess",
            limits=limits,
            generation_constraints=constraints,
            context_cache_config=cache,
            events_compaction_config=compaction,
            graph_dir=str(DATA_DIR / "graphs"),
            embeddings_dir=str(DATA_DIR / "embeddings"),
            wheels_dir=DATA_DIR / "wheels",
            verbose=False,
            **eval_kwargs,
        )
        runtime = describe_runtime(config, vllm_cfg)
        manifest = {
            "schema_version": 1,
            "experiment_id": metadata["experiment_id"],
            "run_id": str(uuid.uuid4()),
            "arm": arm,
            "run_kind": "real_kaggle",
            "timestamp": datetime.now(UTC).isoformat(),
            "code_commit": metadata["code_commit"],
            "working_tree_sha256": metadata["working_tree_sha256"],
            "config_sha256": runtime["config_sha256"],
            "archive_sha256": archive_hashes[arm],
            "model": MODEL,
            "resources": resources,
            "cohort": cohort["tasks"],
            "inference": runtime["inference"],
            "budgets": runtime["budgets"],
            "environment": environment,
            "source_run_id": SOURCE_RUN_URL,
            "limitations": [
                "Small exploratory cohort; no significance or superiority claim.",
                "Unmeasured telemetry must remain null with reasons "
                "during normalization.",
            ],
        }
        validate_manifest(manifest)
        manifest_path = RUN_ROOT / arm / "manifest.json"
        with manifest_path.open("xb") as stream:
            stream.write(canonical_json(manifest))
        manifests[arm] = manifest
        result_paths = await evaluate_official(
            config,
            manifest,
            server=server_instance,
            archive_path=Path(ARCHIVES[arm]),
            wheelhouse_dir=WHEELHOUSE_DIR,
        )
        normalized_records[arm] = [
            normalize_official_result(path, manifest, RUN_ROOT) for path in result_paths
        ]
finally:
    # stop() unlinks the official server log, so preserve it first.
    try:
        log_path = Path(server_instance.log_path)
        if log_path.is_file():
            shutil.copyfile(log_path, RUN_ROOT / "vllm-server.log")
    finally:
        server_instance.stop()
print(
    "Official execution returned with normalized sourced records. "
    "Inspect retained native verification/model traces before publication."
)

## Preserve sources, normalize only observed fields, then import

Keep this saved Kaggle version URL and its actual output bundle. Each run manifest,
raw result, trace, emitted patch and verification receipt needs its content hash.
Inspect actual trace evidence for 31B inference; server startup does not establish
`model_inference_observed`. The notebook deliberately does not invent PASS/FAIL,
patch validity, token counts or missing trace data from library defaults.

Normalize after generation using `mostaza.evaluation.contracts` schema 1. Take
`tool_calls` and `duration_seconds` only when actually observed; keep all absent
measurements null with a reason. Distinguish an execution error from a verified
unresolved task. Patch presence does not establish successful patch application.
Record the inspected apply/verification receipt as `patch.validity_source`, and
real test outcome/exit code/source. Real records require result/trace hashes.
Relevance/Recall@K require explicit **post-run** mappings and definitions, stored
outside the agent workspace. Never feed those references back to the agent.

The next cell creates a hash inventory of actual outputs; it does not authorize
or fabricate a normalized result. Private source traces/test receipts stay out of
Git. Retain this output bundle outside Git so receipt hashes can be checked later.


In [ ]:
from mostaza.evaluation.import_results import import_run, official_execution_receipt
from mostaza.research.metrics import compare
from mostaza.research.report import markdown

need(
    set(manifests) == {"E0", "E1"} and set(normalized_records) == {"E0", "E1"},
    "Both actual arms must return before paired export",
)
OUTPUT_EVIDENCE = RUN_ROOT / "compact-evidence"
OUTPUT_EVIDENCE.mkdir(exist_ok=False)
for arm in ("E0", "E1"):
    imported = import_run(
        manifests[arm], normalized_records[arm], RUN_ROOT / "immutable"
    )
    print(f"{arm} immutable bundle: {imported}")
paired = normalized_records["E0"] + normalized_records["E1"]
with (OUTPUT_EVIDENCE / "paired-results.jsonl").open("xb") as stream:
    stream.write(b"".join(canonical_json(record) for record in paired))
receipt = {
    "schema_version": 1,
    "manifests": manifests,
    "compiler": compiler_receipts,
    "execution": {
        arm: official_execution_receipt(manifests[arm], normalized_records[arm])
        for arm in ("E0", "E1")
    },
}
with (OUTPUT_EVIDENCE / "official-validation.json").open("xb") as stream:
    stream.write(canonical_json(receipt))
with (OUTPUT_EVIDENCE / "resource-review.json").open("xb") as stream:
    stream.write(canonical_json(review))
comparison = compare(
    (manifests["E0"], normalized_records["E0"]),
    (manifests["E1"], normalized_records["E1"]),
)
with (OUTPUT_EVIDENCE / "comparison.json").open("xb") as stream:
    stream.write(canonical_json(comparison))
with (OUTPUT_EVIDENCE / "report.md").open("x") as stream:
    stream.write(markdown(comparison))
print(f"Authentic compact evidence: {OUTPUT_EVIDENCE}")
print(
    "Keep private source outputs outside Git and verify the host gate before publishing"
)

In [ ]:
source_inventory = []
for path in sorted(RUN_ROOT.rglob("*")):
    if path.is_file() and "submission" not in path.relative_to(RUN_ROOT).parts:
        source_inventory.append(
            {"path": str(path.relative_to(RUN_ROOT)), "sha256": file_sha(path)}
        )
with (RUN_ROOT / "source-inventory.json").open("xb") as stream:
    stream.write(
        canonical_json({"source_run_id": SOURCE_RUN_URL, "files": source_inventory})
    )
print(
    f"Preserve the actual source bundle at {RUN_ROOT}; "
    "normalized paired records are still pending review"
)

## Host commands after actual assets and runs exist

From the locked Python 3.13 host environment (`scripts/setup.sh`), run:

```bash
uv run --frozen --no-sync python scripts/check_official.py --check --experiments E0 E1
scripts/package.sh --experiment E0 --output /absolute/new-output/E0.zip --compiler-python /absolute/project/.tools/official/bin/python
scripts/package.sh --experiment E1 --output /absolute/new-output/E1.zip --compiler-python /absolute/project/.tools/official/bin/python
```

The package commands are verified repository CLI contracts. They require the actual reviewed resources and ready experiment configurations. Choose new absolute output paths; archives
are create-only. Upload the archives and curated source input, then run the cells
above with your real mounts, metadata and versioned Kaggle source URL.

After inspecting actual raw results, create schema-valid normalized manifests and
records without altering source bytes. Import each arm and use the **printed**
content-addressed bundle paths, not guessed filenames:

```bash
scripts/run_experiment.sh import /absolute/actual/E0/manifest.json /absolute/actual/E0/records.jsonl --output /absolute/immutable-store
scripts/run_experiment.sh import /absolute/actual/E1/manifest.json /absolute/actual/E1/records.jsonl --output /absolute/immutable-store
scripts/run_experiment.sh report /absolute/printed/E0-bundle.json /absolute/printed/E1-bundle.json --json docs/implementation/evidence/mostaza-phase1-foundation/comparison.json --markdown docs/implementation/evidence/mostaza-phase1-foundation/report.md
uv run --frozen --no-sync python scripts/check_evidence.py --require-real-paired --cohort experiments/cohort.json --evidence-dir docs/implementation/evidence/mostaza-phase1-foundation
```

Publish only the declared compact evidence: verified resource review,
`official-validation.json` with authentic `manifests`, `compiler`, `execution`
receipts keyed E0/E1, exactly six `paired-results.jsonl` records, and the
deterministically recomputed comparison/report. The existing evidence gate
rejects every pending envelope. No local check or synthetic fixture completes
the real benchmark criteria.

Official provenance: [starter version 3](https://www.kaggle.com/code/ryanholbrook/getting-started-gemma-4-developer-agent),
[wheelhouse version 31](https://www.kaggle.com/datasets/metric/gemma-4-developer-agent-wheelhouse),
[competition data](https://www.kaggle.com/competitions/gemma-4-developer-agent/data).
